[Markdown Cell]
# Model A decoder — continue epochs 3 and 4, then extend whenever needed

1. Import this notebook into Kaggle. In **Add Input → Notebooks**, attach the
   **latest saved output/version with two completed epochs**, including
   `model_a_fast_run/code`, `train_config.json`, and `checkpoints/*.pt` + `*.pt.json`.
   The `.ipynb` alone does not contain the trained weights.
2. Also attach the **same full tokenized preprocessing export** used previously:
   `metadata.json`, `tokenizer.json`, `train.bin`, `val.bin`,
   `train_documents.jsonl`, `validation_documents.jsonl`, `validation_pairs.jsonl`.
   These files are normally a separate dataset; attaching a training notebook does
   not attach that notebook's input datasets automatically. Raw text is unnecessary.
3. Select **GPU T4 x2**, enable Internet for the small package installation, and
   **Run All**. Default paths are discovered automatically, including nested exports.

Default `ADDITIONAL_EPOCHS=2` reads the completed-epoch count from the **actual
latest checkpoint**. With two completed epochs, it trains **epochs 3 and 4**.
With four completed epochs in a later saved output, the same setting trains
**epochs 5 and 6**. There is no fixed three-epoch cap. A partial epoch resumes
at its saved batch and counts as the next epoch boundary, without replaying it.
Check the printed `completed_epochs` and `train_through_epoch` before training:
for your current request they must be **2** and **4**. Attach the latest two-epoch
output, rather than the old one-epoch output from the original screenshot.

The latest intact recovery checkpoint is chosen by saved step/generation.
`best.pt` is kept for validation/inference, but an older best checkpoint is not
preferred over more recent training state. We retain weights, Adam moments,
FP16 scaler, learning-rate schedule, batch cursor, history and both rank RNGs.
Your supplied `ModelA_retrain_epoch12` finished **epoch 2 at step 29,942**,
using the optimized `full-corpus-fast-v1` runtime on PyTorch `2.11.0+cu128`.
This notebook recognizes its exact saved code and uses its configuration schema,
including the 1,536-token training context, 512-token loss chunks, fused AdamW,
one loader worker per GPU and full corpus coverage. These settings are retained.
It also recognizes the older original decoder runtime. Every saved runtime file
must match one of those embedded references; checkpoint and data checks remain.
The actual attached checkpoint is authoritative, including outputs saved later.
The fingerprinted model code from the selected saved run stays intact. A declared new training
plan expands the total epoch limit and creates valid new checkpoint/configuration
pairs while retaining model weights, Adam/scaler state, history, global counters,
batch cursor, and RNGs. Epoch counts and optimizer step counters are never reset.
By default the **saved learning-rate schedule** continues through its old horizon,
then stays at its final learning rate for further epochs. With the original
two-epoch decay in your uploaded run, the remaining successful scheduler updates
finish that decay, then further training holds the final rate, **0.00003**.
Optional `CONTINUATION_LR`
selects a constant learning rate without resetting optimizer state or warmup.
Full training-state restoration follows the
[PyTorch checkpoint guidance](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

All new outputs are saved under `/kaggle/working/model_a_extended_run/`.
Re-running cells in the same session resumes its pinned target and does not
automatically start another epoch. To train more epochs, save this new output, start
a fresh Kaggle session, and attach **that latest saved output** and the same data.
Change only `ADDITIONAL_EPOCHS` to choose how many epochs to add in that fresh
session. Alternatively set `TARGET_TOTAL_EPOCHS` to an absolute target such as
`6`; it takes precedence and can explicitly extend an existing working session.
An interrupted extension keeps its saved target in a new session, so a paused
epoch-3/4 run finishes epoch 4 before adding a new stage. Its completed output
can then start the next pair of epochs.
Missing checkpoints and different data/model/settings stop before GPU training.
Your supplied two-epoch run already uses PyTorch `2.11.0+cu128`, so it needs
no downgrade or version migration. The older original run used `2.10.0+cu128`.
With `ALLOW_TORCH_UPGRADE=True` (default), this specific upgrade is handled
automatically: the old fingerprint is independently reproduced and verified,
then new working checkpoint/receipt pairs are written for the current runtime.
Every saved training-state value is retained; the attached original checkpoints
are never edited. Future outputs record the real PyTorch version and migration.
Numerical results can differ across framework versions, so cross-version resume
does not promise identical bits. Other version/build changes still stop early.

The model remains the original randomly initialized LLaMA-style decoder with
RoPE, RMSNorm, SwiGLU and **110,119,680** unique trainable parameters.


[Markdown Cell]
## 1. Environment and imports
Use Kaggle's CUDA-enabled PyTorch >=2.5. Only small tokenizer/progress packages are installed. T4 uses FP16 autocast with dynamic loss scaling; PyTorch chooses a supported attention kernel.


In [1]:
# [Code Cell]
%pip install -q "tokenizers==0.22.1" "tqdm==4.67.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.22.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# [Code Cell]
from pathlib import Path
from dataclasses import asdict, replace
import importlib
import json
import os
import signal
import sys
import subprocess
import torch
import numpy as np
from tokenizers import Tokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("Visible GPUs:", torch.cuda.device_count())
module_parts: dict[str, list[str]] = {}

def append_module(filename: str, source: str) -> None:
    """Keep a readable runtime section until the importable package is assembled."""
    assert filename.endswith(".py") and "/" not in filename and "\\" not in filename
    module_parts.setdefault(filename, []).append(source)


Python: 3.13.15 PyTorch: 2.11.0+cu128
Visible GPUs: 2


[Markdown Cell]
## 2. Configuration and unique parameters
With vocabulary $V$, width $D$, FFN width $F$, and $L$ layers:
\[
P=VD+L(4D^2+3DF+2D)+D=110{,}119{,}680.
\]
RoPE adds no trainable table; the output head reuses the token embedding.
The program counts unique trainable Parameters and enforces
$50{,}000{,}000<P<300{,}000{,}000$ before training and on each GPU.
Default training uses 1024 BPE positions, one window per GPU, and 8 accumulation
microbatches: at most 16,384 supervised tokens per update. Model capacity is 2048
tokens; validation reserves up to 1536 prompt and 512 target tokens. Word counts
and BPE lengths differ. Cropping is measured, and training on 1024 does not
establish generation quality over a longer context.


In [3]:
# [Code Cell]
append_module('config.py', r'''from dataclasses import asdict, dataclass, replace
from pathlib import Path
import json

RUNTIME_VERSION = "cs781-model-a-v1"
SPECIAL_TOKENS = ("<pad>", "<s>", "</s>", "<unk>", "<sep>", "<mask>")
SOURCES = ("supreme_court", "high_courts", "acts", "tribunals")


@dataclass(frozen=True)
class TrainConfig:
    """Immutable architecture, causal objective, and training/recovery choices."""

    data_dir: str = "/kaggle/input/cs781-preprocessed-data"
    run_dir: str = "/kaggle/working/model_a_run"
    resume_dir: str | None = None
    vocab_size: int = 32_768
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 12
    d_ff: int = 2_048
    rms_epsilon: float = 1e-6
    rope_base: float = 10_000.0
    max_seq_tokens: int = 2_048
    train_seq_tokens: int = 1_024
    max_prompt_tokens: int = 1_536  # Includes BOS; words and BPE tokens differ.
    max_target_tokens: int = 512   # Includes EOS if the reference fits.
    dropout: float = 0.1
    activation_checkpointing: bool = True
    loss_chunk_tokens: int = 128
    epochs: int = 3
    micro_batch_size: int = 1  # Per T4; effective global batch = 16 windows.
    accumulation_steps: int = 8
    world_size: int = 2
    num_workers: int = 0
    learning_rate: float = 3e-4
    min_lr_ratio: float = 0.1
    warmup_updates: int = 500
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    max_run_seconds: int = 37_800
    min_free_bytes: int = 1 << 30
    max_checkpoint_bytes: int = 6 << 30
    seed: int = 781
    deterministic: bool = False
    generation_examples_per_source: int = 2
    generation_token_limit: int = 512

    @property
    def block_tokens(self) -> int:
        """Window stride: reuse one context token so document transitions survive."""
        return self.train_seq_tokens

    def validate(self) -> None:
        assert self.vocab_size == 32_768
        assert self.d_model > 0 and self.n_heads > 0 and self.d_model % self.n_heads == 0
        assert (self.d_model // self.n_heads) % 2 == 0, "RoPE needs even head width"
        assert self.n_layers > 0 and self.d_ff > 0 and self.rms_epsilon > 0 and self.rope_base > 1
        assert 0 <= self.dropout < 1
        assert 1 <= self.train_seq_tokens <= self.max_seq_tokens
        assert self.max_prompt_tokens >= 2 and self.max_target_tokens >= 1
        assert self.max_prompt_tokens + self.max_target_tokens - 1 <= self.max_seq_tokens
        assert 1 <= self.loss_chunk_tokens <= self.max_seq_tokens
        assert self.epochs > 0 and self.micro_batch_size > 0 and self.accumulation_steps > 0
        assert self.world_size == 2 and self.num_workers >= 0
        assert self.checkpoint_every_steps == 500
        assert self.learning_rate > 0 and 0 < self.min_lr_ratio <= 1
        assert self.warmup_updates >= 0 and self.max_grad_norm > 0
        assert 0 <= self.weight_decay and 0 < self.beta1 < 1 and 0 < self.beta2 < 1
        assert self.adam_epsilon > 0 and self.max_consecutive_overflows > 0
        assert self.max_run_seconds > 0 and self.min_free_bytes >= 0 and self.max_checkpoint_bytes > 0
        assert self.generation_examples_per_source > 0
        assert 1 <= self.generation_token_limit <= self.max_seq_tokens - self.max_prompt_tokens + 1

    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        return cls(**json.loads(Path(path).read_text(encoding="utf-8")))

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2) + "\n", encoding="utf-8")


def unique_trainable_parameters(model) -> int:
    """Count each shared Parameter once; the tied LM head adds no new weights."""
    return sum(p.numel() for p in {id(p): p for p in model.parameters() if p.requires_grad}.values())


def assert_parameter_boundary(model) -> int:
    count = unique_trainable_parameters(model)
    assert 50_000_000 < count < 300_000_000, f"Parameter boundary violated: {count:,}"
    return count

''')


[Markdown Cell]
The preferred input is `/kaggle/input/cs781-preprocessed-data/`. Discover the unique complete export even inside `kaggle/working/downloaded_output/` ZIP folders under the mounted dataset. Those names describe the old export, not the live working path. Size receipts reject empty duplicate binaries; multiple complete exports require an explicit `data_dir`. Full SHA-256 verification follows.


In [4]:
# [Code Cell]
append_module('config.py', r'''REQUIRED_DATA_FILES = (
    "metadata.json", "tokenizer.json", "train.bin", "val.bin",
    "train_documents.jsonl", "validation_documents.jsonl", "validation_pairs.jsonl",
)


def complete_preprocessing_directory(directory: Path) -> bool:
    """Check names and receipt sizes without loading binaries or copying input data.

    ZIP exports may contain a second, incomplete downloaded_output directory.
    Empty binaries and incomplete exports must never win automatic discovery.
    Full SHA-256 verification still runs in the existing data-contract check.
    """
    try:
        if not all((directory / name).is_file() for name in REQUIRED_DATA_FILES):
            return False
        metadata_path = directory / "metadata.json"
        if not 0 < metadata_path.stat().st_size <= (4 << 20):
            return False
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if not isinstance(metadata, dict) or metadata.get("status") != "verified":
            return False
        if metadata.get("schema_version") != 1 or metadata.get("dtype") != "<u2":
            return False
        if metadata.get("vocab_size") != 32_768:
            return False
        for name, key in (("train.bin", "train_tokens"), ("val.bin", "val_tokens")):
            count = metadata[key]
            if type(count) is not int or count <= 0 or (directory / name).stat().st_size != count * 2:
                return False
        for name in REQUIRED_DATA_FILES[1:]:
            size = (directory / name).stat().st_size
            if size <= 0 or size != metadata["artifacts"][name]["bytes"]:
                return False
        return True
    except (OSError, ValueError, KeyError, TypeError):
        return False


def resolve_preprocessed_data_dir(requested: str | Path,
                                  input_root: str | Path = "/kaggle/input") -> Path:
    """Prefer a complete explicit folder, otherwise locate one unique mounted export.

    Multiple complete exports require an explicit choice; files are never copied
    from read-only Kaggle input into scarce working storage.
    """
    requested_path = Path(requested).expanduser()
    if complete_preprocessing_directory(requested_path):
        return requested_path.resolve()
    roots = [requested_path] if requested_path.is_dir() else []
    mounted_root = Path(input_root)
    if mounted_root.is_dir() and mounted_root not in roots:
        roots.append(mounted_root)
    candidates: set[Path] = set()
    inspected: set[Path] = set()
    for root in roots:
        for metadata_path in root.rglob("metadata.json"):
            directory = metadata_path.parent.resolve()
            inspected.add(directory)
            if complete_preprocessing_directory(directory):
                candidates.add(directory)
    if len(candidates) == 1:
        return next(iter(candidates))
    if len(candidates) > 1:
        options = "\n".join(f"  {path}" for path in sorted(candidates))
        raise ValueError("Multiple complete preprocessing exports found. Set cfg.data_dir "
                         f"to one exact folder:\n{options}")
    found = "\n".join(f"  {path}" for path in sorted(inspected)[:8]) or "  none"
    raise FileNotFoundError(
        f"No complete preprocessing export at {requested_path} or below {mounted_root}. "
        "Attach the tokenized dataset and select the folder containing metadata.json, "
        "tokenizer.json, both nonempty .bin files, and the three JSONL manifests. "
        f"Inspected metadata folders (incomplete or invalid):\n{found}"
    )


def resolve_training_config(cfg: TrainConfig,
                            input_root: str | Path = "/kaggle/input") -> TrainConfig:
    """Canonicalize the input path before saving config or spawning GPU workers."""
    resolved = resolve_preprocessed_data_dir(cfg.data_dir, input_root)
    print(f"Preprocessed data: {resolved}", flush=True)
    return replace(cfg, data_dir=str(resolved))

''')


In [5]:
# [Code Cell]
exec(compile("".join(module_parts["config.py"]), "model_a/config.py", "exec"), globals())

# Leave these paths as None for automatic discovery of the attached saved output/data.
PARENT_RUN_DIR: str | None = None  # Optional exact run folder, NOT an .ipynb path.
DATA_DIR: str | None = None        # Optional exact complete tokenized-export folder.
RUN_DIR = "/kaggle/working/model_a_extended_run"
ADDITIONAL_EPOCHS = 2              # Two completed epochs -> train epochs 3 and 4.
TARGET_TOTAL_EPOCHS: int | None = None  # Optional absolute target, e.g. 4, 6 or 8.
CONTINUATION_LR: float | None = None   # Keep saved schedule; optionally set a constant LR.
ALLOW_TORCH_UPGRADE = True         # Verified 2.10.0+cu128 -> 2.11.0+cu128 migration.
MAX_RUN_SECONDS = 37_800           # Save and pause at a safe update boundary after 10.5h.
RUN_DEVELOPMENT_GENERATION = False # Enable only if you also want example continuations.
OFFICIAL_TEST_JSON: str | None = None

# Do not edit cfg.epochs, checkpoint files, architecture, batches, or tokenizer yourself.
# Epoch/config/fingerprint transitions and the saved scheduler horizon are handled below.
# Configuration/checkpoint discovery runs after the complete original runtime is assembled.
print("Continuation: add", ADDITIONAL_EPOCHS, "epochs; absolute target:", TARGET_TOTAL_EPOCHS)


Continuation: add 2 epochs; absolute target: None


[Markdown Cell]
## 3. Normalization and rotary positions
\[
\operatorname{RMSNorm}(h)=g\odot h/\sqrt{D^{-1}\sum_i h_i^2+\epsilon}.
\]
Normalize in FP32 before casting back. For adjacent Q/K components and
$\theta_i=10000^{-2i/d_h}$, position $m$ applies
\[
R_m(a,b)=(a\cos(m\theta_i)-b\sin(m\theta_i),\,
a\sin(m\theta_i)+b\cos(m\theta_i)).
\]
The Q/K dot product depends on relative positions. Trigonometry uses FP32.


In [6]:
# [Code Cell]
append_module('model.py', r'''from dataclasses import dataclass
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from .config import TrainConfig, assert_parameter_boundary

KeyValue = tuple[Tensor, Tensor]


class RMSNorm(nn.Module):
    """RMSNorm: hidden[B,T,D] -> normalized[B,T,D]; reduce variance in FP32."""

    def __init__(self, width: int, epsilon: float) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.epsilon = epsilon

    def forward(self, hidden: Tensor) -> Tensor:
        normalized = hidden.float() * torch.rsqrt(hidden.float().square().mean(-1, keepdim=True) + self.epsilon)
        return (normalized * self.weight.float()).to(hidden.dtype)


class RotaryEmbedding(nn.Module):
    """Rotate adjacent Q/K pairs at absolute positions; no learned position table.

    q,k[B,H,T,Dh], positions[T] -> rotated q,k of the same shape.
    Frequencies/trigonometry stay FP32 even under FP16 autocast.
    """

    def __init__(self, head_dim: int, base: float) -> None:
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim, self.base = head_dim, base

    def forward(self, query: Tensor, key: Tensor, positions: Tensor) -> KeyValue:
        with torch.autocast(device_type=query.device.type, enabled=False):
            frequencies = self.base ** (-torch.arange(0, self.head_dim, 2, device=query.device,
                                                      dtype=torch.float32) / self.head_dim)
            angles = positions.float()[:, None] * frequencies[None]
            cosine, sine = angles.cos()[None, None], angles.sin()[None, None]

            def rotate(value: Tensor) -> Tensor:
                even, odd = value.float()[..., 0::2], value.float()[..., 1::2]
                return torch.stack((even * cosine - odd * sine, even * sine + odd * cosine),
                                   dim=-1).flatten(-2).to(value.dtype)
            return rotate(query), rotate(key)


''')


[Markdown Cell]
## 4. Causal attention and gated feed-forward layers
\[
A=\operatorname{softmax}(R(Q)R(K)^\top/\sqrt{d_h}+M)V,
\qquad\operatorname{SwiGLU}(h)=W_d(\operatorname{SiLU}(W_g h)\odot W_u h).
\]
Future and padding keys are masked. For unpadded training, SDPA's `is_causal=True`
avoids constructing a dense mask in Python. Right-padded batches combine causal
and key-validity masks; boolean True means allowed. All projections are bias-free.
The FFN uses three matrices and width 2048, rather than a four-times-width GELU FFN.
[SDPA documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention).


In [7]:
# [Code Cell]
append_module('model.py', r'''class CausalSelfAttention(nn.Module):
    """Bias-free multi-head attention: hidden[B,T,D] -> hidden[B,T,D].

    SDPA selects an available kernel on T4; no FlashAttention-only dependency.
    Boolean allowed[B,1,T,T] uses True for legal keys. Cached keys are rotated.
    """

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.heads, self.head_dim, self.dropout = cfg.n_heads, cfg.d_model // cfg.n_heads, cfg.dropout
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.rotary = RotaryEmbedding(self.head_dim, cfg.rope_base)

    def project(self, hidden: Tensor, positions: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        batch, length, _ = hidden.shape
        query, key, value = self.qkv(hidden).view(batch, length, 3, self.heads, self.head_dim).unbind(2)
        query, key, value = (tensor.transpose(1, 2) for tensor in (query, key, value))
        query, key = self.rotary(query, key, positions)
        return query, key, value

    def attend(self, query: Tensor, kv: KeyValue, allowed: Tensor | None,
               causal: bool) -> Tensor:
        output = F.scaled_dot_product_attention(
            query, kv[0], kv[1], attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0)
        return self.out(output.transpose(1, 2).contiguous().flatten(2))

    def forward(self, hidden: Tensor, allowed: Tensor | None) -> Tensor:
        positions = torch.arange(hidden.shape[1], device=hidden.device)
        query, key, value = self.project(hidden, positions)
        return self.attend(query, (key, value), allowed, causal=allowed is None)


class SwiGLU(nn.Module):
    """Gated FFN: hidden[B,T,D] -> Wdown(SiLU(Wgate h) * Wup h)."""

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.down(F.silu(self.gate(hidden)) * self.up(hidden))


''')


[Markdown Cell]
Residual updates are $h'=h+\operatorname{Attn}(\operatorname{RMSNorm}(h))$
and $h''=h'+\operatorname{SwiGLU}(\operatorname{RMSNorm}(h'))$.
Residual dropout is 0.1 by default. Cached keys already include their RoPE rotation.
Prompt prefill uses square causal attention; a new single query sees every cached
past key with `is_causal=False`. A non-square causal flag would hide valid keys.


In [8]:
# [Code Cell]
append_module('model.py', r'''class DecoderBlock(nn.Module):
    """RMSNorm -> rotary attention -> residual -> RMSNorm -> SwiGLU -> residual."""

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.attention_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)
        self.attention = CausalSelfAttention(cfg)
        self.ffn = SwiGLU(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, hidden: Tensor, allowed: Tensor | None) -> Tensor:
        hidden = hidden + self.dropout(self.attention(self.attention_norm(hidden), allowed))
        return hidden + self.dropout(self.ffn(self.ffn_norm(hidden)))

    def cached(self, hidden: Tensor, positions: Tensor, previous: KeyValue | None) -> tuple[Tensor, KeyValue]:
        """Prefill an unpadded prompt or append one token; return updated rotary K/V."""
        query, key, value = self.attention.project(self.attention_norm(hidden), positions)
        kv = (key, value) if previous is None else (
            torch.cat((previous[0], key), dim=2), torch.cat((previous[1], value), dim=2))
        # Prefill has square causal attention. A single new query may see every cached key.
        hidden = hidden + self.dropout(self.attention.attend(query, kv, None, causal=previous is None))
        return hidden + self.dropout(self.ffn(self.ffn_norm(hidden))), kv


@dataclass
class DecoderCache:
    """Inference-only K/V per layer; absolute position counts consumed input tokens."""
    key_values: list[KeyValue]
    position: int


''')


[Markdown Cell]
## 5. Complete model and vocabulary-memory control
Inputs and shifted labels have shape $[B,T]$. Without labels, return $[B,T,V]$
logits. With labels, return a summed FP32 NLL for the training reduction.
Non-reentrant layer checkpointing recomputes activations. Vocabulary loss is
projected in chunks of 128 positions and checkpointed too, avoiding retention of
the complete $B\times T\times V$ logit tensor during backward. This keeps the exact
cross-entropy objective. Tests compare both scalar loss and parameter gradients
with a dense vocabulary projection.
[Activation checkpointing](https://docs.pytorch.org/docs/stable/checkpoint).


In [9]:
# [Code Cell]
append_module('model.py', r'''class LegalCausalDecoder(nn.Module):
    """Randomly initialized LLaMA-style legal LM; shared input/output vocabulary.

    forward(input_ids[B,T], attention_mask[B,T] | None, labels[B,T] | None)
        -> logits[B,T,V] if labels is None, otherwise summed FP32 scalar NLL.
    Labels are already shifted by the dataset; -100 means no supervision.
    """

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList(DecoderBlock(cfg) for _ in range(cfg.n_layers))
        self.final_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)
        self.apply(self.initialize_weights)
        for layer in self.layers:
            nn.init.normal_(layer.attention.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(layer.ffn.down.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
        self.parameter_count = assert_parameter_boundary(self)

    @staticmethod
    def initialize_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def hidden_states(self, input_ids: Tensor, attention_mask: Tensor | None = None) -> Tensor:
        assert input_ids.ndim == 2 and 0 < input_ids.shape[1] <= self.cfg.max_seq_tokens
        allowed = None
        if attention_mask is not None:
            assert attention_mask.shape == input_ids.shape and attention_mask.dtype == torch.bool
            assert bool(attention_mask[:, 0].all()), "Right padding requires a valid first token"
            if not bool(attention_mask.all()):
                causal = torch.ones((input_ids.shape[1], input_ids.shape[1]),
                                    device=input_ids.device, dtype=torch.bool).tril()
                allowed = causal[None, None] & attention_mask[:, None, None, :]
        hidden = self.token_embedding(input_ids)
        for layer in self.layers:
            if self.training and self.cfg.activation_checkpointing:
                hidden = checkpoint(layer, hidden, allowed, use_reentrant=False)
            else:
                hidden = layer(hidden, allowed)
        return self.final_norm(hidden)

    def chunk_loss(self, hidden: Tensor, labels: Tensor) -> Tensor:
        logits = F.linear(hidden, self.token_embedding.weight)
        return F.cross_entropy(logits.float(), labels, ignore_index=-100, reduction="sum")

    def forward(self, input_ids: Tensor, attention_mask: Tensor | None = None,
                labels: Tensor | None = None) -> Tensor:
        hidden = self.hidden_states(input_ids, attention_mask)
        if labels is None:
            return F.linear(hidden, self.token_embedding.weight)
        assert labels.shape == input_ids.shape
        hidden, targets = hidden.flatten(0, 1), labels.flatten()
        losses = []
        for start in range(0, len(targets), self.cfg.loss_chunk_tokens):
            chunk, target = hidden[start:start + self.cfg.loss_chunk_tokens], targets[start:start + self.cfg.loss_chunk_tokens]
            if self.training and torch.is_grad_enabled():
                # Recompute the vocabulary projection on backward instead of retaining B*T*V.
                losses.append(checkpoint(self.chunk_loss, chunk, target, use_reentrant=False))
            else:
                losses.append(self.chunk_loss(chunk, target))
        return torch.stack(losses).sum()


''')


[Markdown Cell]
Prefill returns logits for the next token and per-layer rotary K/V. Append one generated token at its true absolute position. The cache stops at the configured context limit; cached and full logits are verified before training.


In [10]:
# [Code Cell]
append_module('model.py', r'''    @torch.inference_mode()
    def prefill(self, input_ids: Tensor) -> tuple[Tensor, DecoderCache]:
        """Unpadded prompt[B,T] -> next-token logits[B,V], cache with T consumed tokens."""
        assert not self.training and 0 < input_ids.shape[1] <= self.cfg.max_seq_tokens
        hidden = self.token_embedding(input_ids)
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        key_values = []
        for layer in self.layers:
            hidden, kv = layer.cached(hidden, positions, None)
            key_values.append(kv)
        logits = F.linear(self.final_norm(hidden[:, -1]), self.token_embedding.weight)
        return logits, DecoderCache(key_values, input_ids.shape[1])


@torch.inference_mode()
def decode_one_token(model: LegalCausalDecoder, token_ids: Tensor, cache: DecoderCache) -> Tensor:
    """Append token_ids[B,1] at its true RoPE position -> next-token logits[B,V]."""
    assert not model.training and token_ids.shape[1] == 1
    assert cache.position < model.cfg.max_seq_tokens
    hidden = model.token_embedding(token_ids)
    positions = torch.tensor([cache.position], device=token_ids.device)
    for index, layer in enumerate(model.layers):
        hidden, cache.key_values[index] = layer.cached(hidden, positions, cache.key_values[index])
    cache.position += 1
    return F.linear(model.final_norm(hidden[:, 0]), model.token_embedding.weight)
''')


[Markdown Cell]
## 6. Verify the preprocessing export
Check schema, uint16 byte order, 32,768 vocabulary, six special IDs, SHA-256 receipts, token ranges, the 35/35/15/15 training quotas, and equal validation cache quotas. Both binaries are scanned in bounded memmap slices. The tokenizer is reused exactly; no HF access is required.


In [11]:
# [Code Cell]
append_module('data.py', r'''from functools import partial
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import TrainConfig, SPECIAL_TOKENS, SOURCES


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 << 20):
            digest.update(block)
    return digest.hexdigest()


def strict_object(pairs: list[tuple]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key}")
        result[key] = value
    return result


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"), object_pairs_hook=strict_object,
                      parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))


def verify_data_contract(cfg: TrainConfig) -> dict:
    """Verify bounded-memory binaries, tokenizer identity, and published receipts."""
    root = Path(cfg.data_dir)
    metadata = read_json(root / "metadata.json")
    assert metadata["status"] == "verified" and metadata["schema_version"] == 1
    assert metadata["dtype"] == "<u2" and metadata["vocab_size"] == cfg.vocab_size
    assert metadata["train_tokens"] % 100 == 0 and metadata["val_tokens"] % 4 == 0
    assert metadata["training"]["source_tokens"] == dict(zip(
        SOURCES, [metadata["train_tokens"] * percent // 100 for percent in (35, 35, 15, 15)]))
    assert all(metadata["validation"][source]["cached_tokens"] == metadata["val_tokens"] // 4
               for source in SOURCES)
    required = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",
                "validation_documents.jsonl", "validation_pairs.jsonl")
    for name in tqdm(required, desc="Verify preprocessing receipts"):
        path, receipt = root / name, metadata["artifacts"][name]
        assert path.stat().st_size == receipt["bytes"], f"Size mismatch: {name}"
        assert sha256_file(path) == receipt["sha256"], f"Checksum mismatch: {name}"
    tokenizer = Tokenizer.from_file(str(root / "tokenizer.json"))
    ids = {token: tokenizer.token_to_id(token) for token in SPECIAL_TOKENS}
    assert tokenizer.get_vocab_size() == 32_768
    assert ids == metadata["special_token_ids"] == dict(zip(SPECIAL_TOKENS, range(6)))
    for name, count in (("train.bin", metadata["train_tokens"]), ("val.bin", metadata["val_tokens"])):
        assert count > 0 and (root / name).stat().st_size == count * 2
        cache = np.memmap(root / name, dtype="<u2", mode="r", shape=(count,))
        for start in range(0, count, 1_000_000):
            assert int(cache[start:start + 1_000_000].max()) < cfg.vocab_size
        del cache
    identity = {name: metadata["artifacts"][name]["sha256"] for name in required}
    identity["metadata.json"] = sha256_file(root / "metadata.json")
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    return {"fingerprint": fingerprint, "metadata": metadata, "special_ids": ids}


''')


[Markdown Cell]
Index document offsets and window counts. Consecutive windows share one input token so every within-document transition is supervised once. Never predict across documents. One-token documents have no target and are reported. Training and validation document hashes must be disjoint. Only compact indices are written to working storage.


In [12]:
# [Code Cell]
append_module('data.py', r'''def prepare_indices(cfg: TrainConfig, contract: dict) -> dict:
    """Index document-local next-token windows; retain every within-document transition.

    Adjacent windows share one input context token. Each target is supervised once
    per document pass, without predicting across independent document boundaries.
    """
    root, run = Path(cfg.data_dir), Path(cfg.run_dir)
    index_dir = run / "indices"
    index_dir.mkdir(parents=True, exist_ok=True)
    offsets, lengths, counts, source_indices = [], [], [], []
    train_hashes, validation_hashes = set(), set()
    source_tokens, target_tokens, dropped_tokens = Counter(), Counter(), Counter()
    cursor = 0
    with (root / "train_documents.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            row = json.loads(line, object_pairs_hook=strict_object)
            offset, length, source = row["cache_offset"], row["cache_length"], row["source"]
            assert source in SOURCES and offset == cursor and length > 0
            assert row["sha256"] not in train_hashes
            train_hashes.add(row["sha256"])
            predictions = length - 1
            count = (predictions + cfg.train_seq_tokens - 1) // cfg.train_seq_tokens
            if count:
                offsets.append(offset); lengths.append(length); counts.append(count)
                source_indices.append(SOURCES.index(source))
            source_tokens[source] += length
            target_tokens[source] += predictions
            dropped_tokens[source] += int(length == 1)
            cursor += length
    assert cursor == contract["metadata"]["train_tokens"]
    assert dict(source_tokens) == contract["metadata"]["training"]["source_tokens"]
    with (root / "validation_documents.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            digest = json.loads(line)["sha256"]
            assert digest not in validation_hashes
            validation_hashes.add(digest)
    assert train_hashes.isdisjoint(validation_hashes)
    cumulative = np.cumsum(np.asarray(counts, dtype=np.int64))
    assert len(cumulative) and int(cumulative[-1]) >= cfg.world_size * cfg.micro_batch_size
    train_path = index_dir / "train_index.npz"
    temporary = train_path.with_suffix(".partial")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, offsets=np.array(offsets, dtype=np.int64),
                            lengths=np.array(lengths, dtype=np.int64), cumulative=cumulative,
                            sources=np.array(source_indices, dtype=np.uint8))
        stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, train_path)
    validation = prepare_validation_index(cfg, validation_hashes, index_dir)
    report = {"train_examples": int(cumulative[-1]), "cached_source_tokens": dict(source_tokens),
              "eligible_target_tokens": dict(target_tokens), "one_token_documents_dropped": dict(dropped_tokens),
              "global_batch_tail_max_examples": cfg.world_size * cfg.micro_batch_size - 1,
              "validation": validation, "data_fingerprint": contract["fingerprint"]}
    (index_dir / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    return report


''')


[Markdown Cell]
Read all complete 512-word/128-word JSONL pairs, even if the last val.bin slice was cropped for its exact quota. Encode prompt and reference separately; labels ignore the prompt. Long prompts keep their recent tail; targets receive EOS only when the full reference fits. Report prompt and target truncations explicitly.


In [13]:
# [Code Cell]
append_module('data.py', r'''def tokenize_pair(tokenizer: Tokenizer, prompt: str, reference: str,
                  cfg: TrainConfig) -> tuple[list[int], list[int], bool, bool]:
    """Encode prompt and target separately; report BPE crops, preserve the prompt tail."""
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    target_ids = tokenizer.encode(" " + reference, add_special_tokens=False).ids
    prompt_cropped = len(prompt_ids) > cfg.max_prompt_tokens - 1
    target_cropped = len(target_ids) + 1 > cfg.max_target_tokens
    prefix = [1] + prompt_ids[-(cfg.max_prompt_tokens - 1):]
    target = target_ids[:cfg.max_target_tokens] if target_cropped else target_ids + [2]
    assert prefix and target and len(prefix) + len(target) - 1 <= cfg.max_seq_tokens
    return prefix, target, prompt_cropped, target_cropped


def prepare_validation_index(cfg: TrainConfig, allowed_hashes: set[str], index_dir: Path) -> dict:
    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))
    offsets, identifiers, statistics = [], set(), Counter()
    with (Path(cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:
        while True:
            offset = stream.tell()
            line = stream.readline()
            if not line:
                break
            row = json.loads(line, object_pairs_hook=strict_object)
            assert row["id"] not in identifiers and row["document_sha256"] in allowed_hashes
            assert len(row["prompt"].split()) == 512 and len(row["reference"].split()) == 128
            assert row["source"] in SOURCES
            identifiers.add(row["id"]); offsets.append(offset)
            _, _, prompt_crop, target_crop = tokenize_pair(tokenizer, row["prompt"], row["reference"], cfg)
            statistics[row["source"]] += 1
            statistics["prompt_truncations"] += int(prompt_crop)
            statistics["target_truncations"] += int(target_crop)
    assert offsets and all(statistics[source] for source in SOURCES)
    path = index_dir / "validation_offsets.npy"
    with path.with_suffix(".partial").open("wb") as stream:
        np.save(stream, np.asarray(offsets, dtype=np.int64), allow_pickle=False)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(path.with_suffix(".partial"), path)
    return dict(statistics, examples=len(offsets))


''')


[Markdown Cell]
Training uses $x=[t_0,\ldots,t_{T-1}]$ and $y=[t_1,\ldots,t_T]$.
Only genuine cached BOS/EOS tokens are used; arbitrary window edges get no
invented EOS. Validation places reference tokens after the prompt, masks prompt
labels, and uses causality to prevent future-reference leakage. Each process
opens its read-only memmap lazily; worker serialization drops live handles.


In [14]:
# [Code Cell]
append_module('data.py', r'''class ContinuationDataset(Dataset):
    """One document window -> input_ids[T], shifted labels[T]; all transitions train."""

    def __init__(self, cfg: TrainConfig) -> None:
        self.cfg, self.cache = cfg, None
        with np.load(Path(cfg.run_dir) / "indices/train_index.npz", allow_pickle=False) as index:
            self.offsets, self.lengths = index["offsets"].copy(), index["lengths"].copy()
            self.cumulative = index["cumulative"].copy()

    def __len__(self) -> int:
        return int(self.cumulative[-1])

    def __getstate__(self) -> dict:
        return dict(self.__dict__, cache=None)

    def __getitem__(self, index: int) -> dict:
        assert 0 <= index < len(self)
        if self.cache is None:
            self.cache = np.memmap(Path(self.cfg.data_dir) / "train.bin", dtype="<u2", mode="r")
        document = int(np.searchsorted(self.cumulative, index, side="right"))
        previous_count = int(self.cumulative[document - 1]) if document else 0
        within = (index - previous_count) * self.cfg.train_seq_tokens
        length = min(self.cfg.train_seq_tokens + 1, int(self.lengths[document]) - within)
        start = int(self.offsets[document]) + within
        tokens = np.asarray(self.cache[start:start + length], dtype=np.int64).tolist()
        assert 2 <= len(tokens) <= self.cfg.train_seq_tokens + 1
        return {"input_ids": tokens[:-1], "labels": tokens[1:]}


class ValidationDataset(Dataset):
    """JSONL prompt/continuation teacher forcing; only reference tokens affect NLL."""

    def __init__(self, cfg: TrainConfig) -> None:
        self.cfg, self.tokenizer = cfg, None
        self.offsets = np.load(Path(cfg.run_dir) / "indices/validation_offsets.npy", allow_pickle=False)

    def __len__(self) -> int:
        return len(self.offsets)

    def __getstate__(self) -> dict:
        return dict(self.__dict__, tokenizer=None)

    def __getitem__(self, index: int) -> dict:
        if self.tokenizer is None:
            self.tokenizer = Tokenizer.from_file(str(Path(self.cfg.data_dir) / "tokenizer.json"))
        with (Path(self.cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:
            stream.seek(int(self.offsets[index]))
            row = json.loads(stream.readline())
        prefix, target, _, _ = tokenize_pair(self.tokenizer, row["prompt"], row["reference"], self.cfg)
        return {"input_ids": (prefix + target)[:-1],
                "labels": [-100] * (len(prefix) - 1) + target}


''')


[Markdown Cell]
Right-pad input IDs with PAD=0, labels with -100, and mark valid attention keys. A seeded global permutation supplies disjoint rank batches without duplicate padding. The sampler resumes directly at `next_batch`; at most `world_size*micro_batch_size-1` examples are dropped at an epoch's final incomplete global batch.


In [15]:
# [Code Cell]
append_module('data.py', r'''def collate_continuations(rows: list[dict]) -> dict[str, torch.Tensor]:
    """Pad input IDs with PAD=0; ignored labels=-100; valid keys=True."""
    length = max(len(row["input_ids"]) for row in rows)
    input_ids = torch.zeros((len(rows), length), dtype=torch.long)
    labels = torch.full_like(input_ids, -100)
    attention_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for index, row in enumerate(rows):
        count = len(row["input_ids"])
        assert count == len(row["labels"]) and count > 0
        input_ids[index, :count] = torch.tensor(row["input_ids"])
        labels[index, :count] = torch.tensor(row["labels"])
        attention_mask[index, :count] = True
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}



class EpochBatchSampler(Sampler):
    """Seeded global permutation with disjoint ranks and a direct batch cursor.

    No prefix batches are loaded on resume. At most world_size*batch_size-1
    examples are dropped per epoch; no duplicate examples pad the final batch.
    """

    def __init__(self, size: int, batch_size: int, rank: int, world_size: int,
                 seed: int, epoch: int, start_batch: int = 0) -> None:
        self.batch_size, self.rank, self.world_size = batch_size, rank, world_size
        self.num_batches = size // (world_size * batch_size)
        assert 0 <= start_batch <= self.num_batches and 0 <= rank < world_size
        self.start_batch = start_batch
        generator = torch.Generator().manual_seed(seed + epoch)
        count = self.num_batches * world_size * batch_size
        self.batches = torch.randperm(size, generator=generator)[:count].reshape(
            self.num_batches, world_size, batch_size,
        )[:, rank]

    def __iter__(self):
        for index in range(self.start_batch, self.num_batches):
            yield self.batches[index].tolist()

    def __len__(self) -> int:
        return self.num_batches - self.start_batch
''')


[Markdown Cell]
## 7. Checkpoint integrity and reproducibility
The original decoder and the supplied optimized two-epoch runtime are embedded
as exact reference sources. The setup cell selects the matching saved runtime,
uses its configuration schema, and checks data identity,
checkpoint receipts and the current PyTorch build before GPU training.
The continuation helper records an explicit epoch-plan transition. The external
wrapper preserves the saved scheduler horizon and controls this session's target.


In [16]:
# [Code Cell]
append_module('state.py', r'''from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import hashlib
import json
import math
import os
import random
import shutil
import numpy as np
import torch
import torch.distributed as dist
from .config import TrainConfig, RUNTIME_VERSION
from .data import sha256_file, read_json


def distributed() -> bool:
    return dist.is_available() and dist.is_initialized()


def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n"); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)


def runtime_fingerprint(cfg: TrainConfig, contract: dict) -> str:
    """Paths and time/space limits may change; learning and data choices may not."""
    choices = asdict(cfg)
    for key in ("data_dir", "run_dir", "resume_dir", "max_run_seconds", "min_free_bytes",
                "max_checkpoint_bytes", "generation_examples_per_source", "generation_token_limit"):
        choices.pop(key)
    sources = {path.name: sha256_file(path) for path in sorted(Path(__file__).parent.glob("*.py"))}
    payload = {"version": RUNTIME_VERSION, "config": choices,
               "data": contract["fingerprint"], "sources": sources,
               "torch": str(torch.__version__)}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


@contextmanager
def exclusive_run_lock(run_dir: Path):
    """Linux advisory lock is released by the OS even after a killed kernel."""
    import fcntl  # Kaggle/Linux only; production does not silently fall back.
    run_dir.mkdir(parents=True, exist_ok=True)
    with (run_dir / ".training.lock").open("a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Another Model A training process owns this run directory.") from exc
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


''')


[Markdown Cell]
Save Python, NumPy, CPU Torch, and each GPU rank's CUDA RNG state. Dedicated loader and sampler generators keep prefetch and restart from perturbing dropout. Load tensors and primitives with `weights_only=True`. Saved-cursor replay is exact; CUDA bitwise identity also depends on kernels and the runtime.


In [17]:
# [Code Cell]
append_module('state.py', r'''def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {"python": random.getstate(), "torch_cpu": torch.get_rng_state(),
            "numpy": {"kind": numpy_state[0], "keys": torch.tensor(numpy_state[1].astype(np.int64)),
                      "position": numpy_state[2], "has_gauss": numpy_state[3],
                      "cached_gaussian": numpy_state[4]},
            "torch_cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state["python"]); torch.set_rng_state(state["torch_cpu"])
    ns = state["numpy"]
    np.random.set_state((ns["kind"], ns["keys"].numpy().astype(np.uint32), ns["position"],
                         ns["has_gauss"], ns["cached_gaussian"]))
    if device.type == "cuda":
        torch.cuda.set_rng_state(state["torch_cuda"], device)


def cpu_tree(value):
    """Own a CPU snapshot, retaining only tensors and weights-only-safe primitives."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return value


''')


[Markdown Cell]
Retain two full recovery slots and one full best checkpoint. Receipts verify SHA-256 and commit last; an incomplete latest write falls back to an intact older slot. Automatically load the newest compatible checkpoint in the working run or the explicitly attached `resume_dir`.


In [18]:
# [Code Cell]
append_module('state.py', r'''class CheckpointManager:
    """Two full recovery slots plus one full best checkpoint, each with SHA-256.

    The inactive slot is replaced first; its receipt commits last. A crash
    between those writes leaves the previously committed slot recoverable.
    Every full checkpoint includes optimizer, scaler, scheduler, and cursor.
    """

    def __init__(self, cfg: TrainConfig, fingerprint: str) -> None:
        self.cfg, self.fingerprint, self.generation = cfg, fingerprint, 0
        self.directory = Path(cfg.run_dir) / "checkpoints"
        self.directory.mkdir(parents=True, exist_ok=True)

    def latest(self) -> tuple[Path, dict] | None:
        roots = [self.directory]
        if self.cfg.resume_dir:
            roots.append(Path(self.cfg.resume_dir))
        candidates, seen, problems = [], False, []
        for root in roots:
            for name in ("slot_0.pt", "slot_1.pt", "best.pt"):
                path, receipt_path = root / name, root / (name + ".json")
                if not path.exists() and not receipt_path.exists():
                    continue
                seen = True
                try:
                    receipt = read_json(receipt_path)
                    assert isinstance(receipt, dict)
                    assert all(type(receipt[key]) is int for key in ("bytes", "step", "generation", "epoch"))
                    assert receipt["bytes"] > 0 and receipt["step"] >= 0
                    assert receipt["generation"] > 0 and receipt["epoch"] >= 0
                    assert path.stat().st_size == receipt["bytes"]
                    assert sha256_file(path) == receipt["sha256"]
                    candidates.append((path, receipt))
                except (OSError, ValueError, TypeError, KeyError, AssertionError) as exc:
                    problems.append(f"{name}: {type(exc).__name__}")
        if seen and not candidates:
            raise RuntimeError(f"No intact checkpoint; preserve files and inspect receipts. {problems}")
        if not candidates:
            if self.cfg.resume_dir:
                raise FileNotFoundError("resume_dir contains no committed checkpoints.")
            return None
        chosen = max(candidates, key=lambda item: (item[1]["step"], item[1]["generation"]))
        if chosen[1]["fingerprint"] != self.fingerprint:
            raise RuntimeError("Checkpoint data/configuration/runtime differs. Use a fresh run directory.")
        self.generation = chosen[1]["generation"]
        if problems and (not distributed() or dist.get_rank() == 0):
            print(f"Ignoring incomplete checkpoints: {problems}; recovered step {chosen[1]['step']}", flush=True)
        return chosen

    def ensure_space(self, estimated_bytes: int) -> None:
        owned = sum(path.stat().st_size for path in self.directory.iterdir() if path.is_file())
        free = shutil.disk_usage(self.directory).free
        if free < estimated_bytes + self.cfg.min_free_bytes:
            raise OSError("Insufficient disk space for an atomic checkpoint and free-space reserve.")
        if owned + estimated_bytes > self.cfg.max_checkpoint_bytes:
            raise OSError("Checkpoint storage cap reached. Inspect this run's checkpoint files.")

    def restore(self, model, optimizer, scaler, scheduler, device: torch.device) -> dict | None:
        chosen = self.latest()
        if chosen is None:
            return None
        path, receipt = chosen
        payload = torch.load(path, map_location="cpu", weights_only=True)
        assert payload["fingerprint"] == self.fingerprint
        assert payload["step"] == receipt["step"] and payload["epoch"] == receipt["epoch"]
        assert payload["world_size"] == self.cfg.world_size
        model.load_state_dict(payload["model"], strict=True)
        optimizer.load_state_dict(payload["optimizer"])
        scaler.load_state_dict(payload["scaler"])
        scheduler.load_state_dict(payload["scheduler"])
        assert scheduler.steps == payload["cursor"]["optimizer_steps"]
        return payload


''')


[Markdown Cell]
Rank 0 saves model, optimizer, scaler, scheduler, epoch/step integers, next batch, epoch statistics, and all rank RNG states. Save only after a complete accumulation group. Notify every rank of success or failure. Check disk space and the 6GiB owned-file cap before serialization. Best snapshots also contain full recovery state.


In [19]:
# [Code Cell]
append_module('state.py', r'''    def save(self, model, optimizer, scaler, scheduler, cursor: dict,
             device: torch.device, rank: int, best: bool = False) -> None:
        local_rng = capture_rng(device)
        rng_states = [None] * dist.get_world_size() if distributed() else [local_rng]
        if distributed():
            dist.all_gather_object(rng_states, local_rng)
        outcome = [None]
        if rank == 0:
            try:
                self.generation += 1
                path = self.directory / f"slot_{self.generation % 2}.pt"
                temporary = path.with_name(path.name + ".partial")
                # FP32 weights + Adam's two FP32 moments + serialization headroom.
                estimated = sum(p.numel() for p in model.parameters()) * 12 + (32 << 20)
                self.ensure_space(estimated)
                payload = {"version": RUNTIME_VERSION, "fingerprint": self.fingerprint,
                           "world_size": self.cfg.world_size, "config": asdict(self.cfg),
                           "epoch": int(cursor["epoch"]), "step": int(cursor["global_step"]),
                           "model": cpu_tree(model.state_dict()),
                           "optimizer": cpu_tree(optimizer.state_dict()), "scaler": scaler.state_dict(),
                           "scheduler": scheduler.state_dict(), "cursor": dict(cursor),
                           "rng_by_rank": rng_states}
                with temporary.open("wb") as stream:
                    torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())
                receipt = {"bytes": temporary.stat().st_size, "sha256": sha256_file(temporary),
                           "fingerprint": self.fingerprint, "generation": self.generation,
                           "epoch": payload["epoch"], "step": payload["step"],
                           "validation_nll": (cursor["best_validation_nll"]
                                              if math.isfinite(cursor["best_validation_nll"]) else None)}
                os.replace(temporary, path)
                atomic_json(path.with_name(path.name + ".json"), receipt)
                if best:
                    self.publish_best(path, receipt)
                outcome[0] = {"ok": True, "generation": self.generation}
            except Exception as exc:
                outcome[0] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
        if distributed():
            dist.broadcast_object_list(outcome, src=0)
        if not outcome[0]["ok"]:
            raise RuntimeError(f"Checkpoint failed: {outcome[0]['error']}")
        self.generation = outcome[0]["generation"]

    def publish_best(self, source: Path, receipt: dict) -> None:
        """Hard-link the complete immutable snapshot; copy if links are unavailable."""
        path = self.directory / "best.pt"
        temporary = self.directory / "best.pt.partial"
        if temporary.exists():
            temporary.unlink()
        try:
            os.link(source, temporary)
        except OSError:
            self.ensure_space(source.stat().st_size)
            shutil.copyfile(source, temporary)
            with temporary.open("r+b") as stream:
                os.fsync(stream.fileno())
        os.replace(temporary, path)
        atomic_json(path.with_name(path.name + ".json"), receipt)


''')


[Markdown Cell]
Warm up to the peak LR and decay toward 10% of it with cosine scheduling. Advance only after a successful optimizer update. Restore scheduler position and LR with the optimizer; an AMP overflow consumes the group without advancing either.


In [20]:
# [Code Cell]
append_module('state.py', r'''class WarmupCosineScheduler:
    """LR advances only after a successful optimizer update, including resume."""

    def __init__(self, optimizer, total_steps: int, warmup_steps: int,
                 peak_lr: float, minimum_ratio: float) -> None:
        self.optimizer, self.total_steps = optimizer, total_steps
        self.warmup_steps = min(warmup_steps, max(0, total_steps - 1))
        self.peak_lr, self.minimum_ratio, self.steps = peak_lr, minimum_ratio, 0
        self.apply_lr()

    def apply_lr(self) -> None:
        if self.warmup_steps and self.steps < self.warmup_steps:
            ratio = (self.steps + 1) / self.warmup_steps
        else:
            denominator = max(1, self.total_steps - self.warmup_steps - 1)
            progress = min(1.0, max(0.0, (self.steps - self.warmup_steps) / denominator))
            ratio = self.minimum_ratio + (1 - self.minimum_ratio) * (1 + math.cos(math.pi * progress)) / 2
        for group in self.optimizer.param_groups:
            group["lr"] = self.peak_lr * ratio

    def step(self) -> None:
        self.steps += 1
        self.apply_lr()

    def state_dict(self) -> dict:
        return {key: getattr(self, key) for key in (
            "total_steps", "warmup_steps", "peak_lr", "minimum_ratio", "steps")}

    def load_state_dict(self, state: dict) -> None:
        for key in ("total_steps", "warmup_steps", "peak_lr", "minimum_ratio"):
            assert state[key] == getattr(self, key), f"Scheduler mismatch: {key}"
        self.steps = state["steps"]
        self.apply_lr()
''')


[Markdown Cell]
## 8. Distributed causal optimization
\[
\mathcal L_{\rm train}=-N^{-1}\sum_{d}\sum_t\log p_\theta(t_{t+1}\mid t_{\le t}).
\]
Train on all document transitions. Validation NLL measures only reference tokens
conditioned on the held-out prompt, making it comparable to Model B's continuation
metric. AdamW decays matrix weights once and excludes normalization vectors.
FP16 autocast accelerates projections; FP32 cross-entropy protects the softmax.


In [21]:
# [Code Cell]
append_module('train.py', r'''import argparse
from contextlib import nullcontext
from datetime import timedelta
from functools import partial
from pathlib import Path
import json
import math
import os
import signal
import socket
import time
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from .config import TrainConfig, assert_parameter_boundary, resolve_training_config
from .data import (verify_data_contract, prepare_indices, ContinuationDataset,
                   ValidationDataset, EpochBatchSampler, collate_continuations)
from .model import LegalCausalDecoder
from .state import (CheckpointManager, WarmupCosineScheduler, atomic_json, distributed,
                    exclusive_run_lock, runtime_fingerprint, seed_everything, restore_rng)


def build_optimizer(model: nn.Module, cfg: TrainConfig):
    """Decay matrix weights once; exclude bias and normalization vectors."""
    decay, no_decay = [], []
    for parameter in model.parameters():
        if parameter.requires_grad:
            (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2), eps=cfg.adam_epsilon,
        foreach=False,  # Avoid large optimizer temporary tensor lists on T4.
    )


def reduce_sum(tensor: torch.Tensor) -> torch.Tensor:
    if distributed():
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor


def any_rank(flag: bool, device: torch.device) -> bool:
    value = torch.tensor(int(flag), device=device, dtype=torch.int32)
    if distributed():
        dist.all_reduce(value, op=dist.ReduceOp.MAX)
    return bool(value.item())


def move_batch(batch: dict, device: torch.device) -> dict:
    return {key: value.to(device, non_blocking=device.type == "cuda") for key, value in batch.items()}


def autocast_context(device: torch.device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()


def summed_loss(model, batch: dict) -> torch.Tensor:
    return model(batch["input_ids"], batch["attention_mask"], labels=batch["labels"])


''')


[Markdown Cell]
For $R$ ranks, scale each microbatch's summed loss by $R/N_{\rm global}$;
DDP's gradient average gives the global token-mean gradient, including variable
lengths and short final accumulation groups. `no_sync()` encloses forward and
backward for nonfinal microbatches. Unscale before clipping. Any rank's nonfinite
gradient makes all ranks skip the optimizer and back off their scales together.
[DDP](https://docs.pytorch.org/docs/stable/notes/ddp),
[AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples).


In [22]:
# [Code Cell]
append_module('train.py', r'''def train_one_update(model, optimizer, scaler, scheduler, batches: list[dict],
                     cfg: TrainConfig, device: torch.device) -> dict:
    """One complete accumulation group; no checkpoint can contain partial gradients.

    DDP averages rank gradients. Scaling each summed loss by R / global_tokens
    yields the exact token-mean gradient across variable lengths and microbatches.
    """
    world_size = dist.get_world_size() if distributed() else 1
    count = sum(int((batch["labels"] != -100).sum()) for batch in batches)
    global_tokens = int(reduce_sum(torch.tensor(count, dtype=torch.int64, device=device)).item())
    assert global_tokens > 0
    optimizer.zero_grad(set_to_none=True)
    local_loss = torch.zeros((), dtype=torch.float64, device=device)
    used_lr = optimizer.param_groups[0]["lr"]
    for index, cpu_batch in enumerate(batches):
        batch = move_batch(cpu_batch, device)
        synchronize = index == len(batches) - 1
        context = model.no_sync() if isinstance(model, DDP) and not synchronize else nullcontext()
        with context:
            with autocast_context(device):
                loss_sum = summed_loss(model, batch)
            if any_rank(not bool(torch.isfinite(loss_sum).item()), device):
                raise FloatingPointError("Non-finite loss; recover from the last committed checkpoint.")
            local_loss += loss_sum.detach().double()
            scaler.scale(loss_sum * (world_size / global_tokens)).backward()
    scaler.unscale_(optimizer)
    gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm,
                                            error_if_nonfinite=False, foreach=False)
    overflow = any_rank(not bool(torch.isfinite(gradient_norm).item()), device)
    if overflow:
        if not scaler.is_enabled():
            raise FloatingPointError("Non-finite FP32 gradients.")
        # Every rank skips together; explicit scale change also resets scaler state.
        scaler.update(new_scale=scaler.get_scale() * scaler.get_backoff_factor())
    else:
        scaler.step(optimizer); scaler.update(); scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    loss_value = float(reduce_sum(local_loss).item())
    return {"loss_sum": loss_value, "tokens": global_tokens, "loss": loss_value / global_tokens,
            "learning_rate": used_lr, "overflow": overflow,
            "gradient_norm": float(gradient_norm.item())}


''')


[Markdown Cell]
Each held-out example is visited exactly once globally. Evaluate the raw module so unequal rank batch counts do not trigger DDP forward collectives; then reduce summed NLL and target tokens. Best selection uses token-weighted conditional NLL. Perplexity is specific to this conditional objective; the provided evaluator measures BERTScore/ROUGE/BLEU separately.


In [23]:
# [Code Cell]
append_module('train.py', r'''@torch.inference_mode()
def evaluate(model: LegalCausalDecoder, cfg: TrainConfig, device: torch.device,
             rank: int, world_size: int) -> dict:
    """Visit every held-out pair exactly once globally, without sampler padding."""
    model.eval()
    dataset = ValidationDataset(cfg)
    subset = Subset(dataset, range(rank, len(dataset), world_size))
    loader = DataLoader(subset, batch_size=cfg.micro_batch_size, shuffle=False,
                        collate_fn=collate_continuations, num_workers=0,
                        generator=torch.Generator().manual_seed(cfg.seed + 10_000),
                        pin_memory=device.type == "cuda")
    totals = torch.zeros(2, dtype=torch.float64, device=device)
    for cpu_batch in tqdm(loader, desc="Held-out continuation NLL", disable=rank != 0, mininterval=1):
        batch = move_batch(cpu_batch, device)
        with autocast_context(device):
            loss = summed_loss(model, batch)
        totals[0] += loss.double()
        totals[1] += (batch["labels"] != -100).sum()
    reduce_sum(totals)
    assert totals[1] > 0 and torch.isfinite(totals).all(), "Invalid validation loss"
    nll = float((totals[0] / totals[1]).item())
    model.train()
    return {"conditional_nll": nll, "conditional_perplexity": math.exp(min(nll, 80)),
            "perplexity_capped": nll > 80, "target_tokens": int(totals[1].item()),
            "examples": len(dataset)}


''')


[Markdown Cell]
Record the next rank-local microbatch, completed accumulation groups, successful optimizer updates, and epoch totals. Resume jumps directly to that batch; remove replayed log tails after a corrupt-checkpoint fallback. Sampling and item contents are deterministic.


In [24]:
# [Code Cell]
append_module('train.py', r'''def new_cursor() -> dict:
    return {"epoch": 0, "next_batch": 0, "global_step": 0, "optimizer_steps": 0,
            "epoch_loss_sum": 0.0, "epoch_tokens": 0, "consecutive_overflows": 0,
            "epoch_overflows": 0, "best_validation_nll": float("inf"), "history": []}


def training_loader(dataset, cfg: TrainConfig, epoch: int, start_batch: int, rank: int):
    sampler = EpochBatchSampler(len(dataset), cfg.micro_batch_size, rank, cfg.world_size,
                                cfg.seed, epoch, start_batch)
    loader = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_continuations,
                        num_workers=cfg.num_workers, pin_memory=True, persistent_workers=False,
                        generator=torch.Generator().manual_seed(cfg.seed + epoch * 31 + rank),
                        **({"multiprocessing_context": "spawn", "prefetch_factor": 1}
                           if cfg.num_workers else {}))
    return loader, sampler


def repair_step_log(path: Path, saved_step: int) -> None:
    """Remove interrupted/replayed log tails newer than the committed cursor."""
    if not path.exists():
        return
    temporary = path.with_name(path.name + ".partial")
    with path.open(encoding="utf-8") as source, temporary.open("w", encoding="utf-8") as target:
        for line in source:
            try:
                row = json.loads(line)
            except ValueError:
                break
            if row["step"] <= saved_step:
                target.write(line)
        target.flush(); os.fsync(target.fileno())
    os.replace(temporary, path)


''')


[Markdown Cell]
One `spawn` process owns each T4, with `backend='nccl'`. Restore full training state and that rank's RNG before loading the next saved batch. Rank 0 alone shows tqdm progress with epoch, step, loss, and LR. The production code uses DDP throughout.


In [25]:
# [Code Cell]
append_module('train.py', r'''def worker(rank: int, cfg: TrainConfig, contract: dict, fingerprint: str,
           port: int, stop_event) -> None:
    """One spawned process per T4; all training collectives follow the same order."""
    os.environ["MASTER_ADDR"], os.environ["MASTER_PORT"] = "127.0.0.1", str(port)
    torch.cuda.set_device(rank)
    device = torch.device("cuda", rank)
    signal.signal(signal.SIGTERM, lambda *_: stop_event.set())
    signal.signal(signal.SIGINT, lambda *_: stop_event.set())
    dist.init_process_group(backend="nccl", rank=rank, world_size=cfg.world_size,
                            timeout=timedelta(minutes=15))
    started = time.monotonic()
    try:
        seed_everything(cfg.seed + rank, cfg.deterministic)
        model = LegalCausalDecoder(cfg).to(device)
        assert_parameter_boundary(model)
        ddp = DDP(model, device_ids=[rank], output_device=rank,
                  broadcast_buffers=False, find_unused_parameters=False,
                  gradient_as_bucket_view=True)
        optimizer = build_optimizer(model, cfg)
        dataset = ContinuationDataset(cfg)
        batches_per_epoch = len(dataset) // (cfg.world_size * cfg.micro_batch_size)
        updates_per_epoch = math.ceil(batches_per_epoch / cfg.accumulation_steps)
        scheduler = WarmupCosineScheduler(optimizer, updates_per_epoch * cfg.epochs,
                                         cfg.warmup_updates, cfg.learning_rate, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        manager = CheckpointManager(cfg, fingerprint)
        payload = manager.restore(model, optimizer, scaler, scheduler, device)
        cursor = new_cursor() if payload is None else payload["cursor"]
        assert 0 <= cursor["epoch"] <= cfg.epochs and 0 <= cursor["next_batch"] <= batches_per_epoch
        assert cursor["next_batch"] % cfg.accumulation_steps == 0 or cursor["next_batch"] == batches_per_epoch
        if payload is not None:
            assert len(payload["rng_by_rank"]) == cfg.world_size
            restore_rng(payload["rng_by_rank"][rank], device)
        log_path = Path(cfg.run_dir) / "training_steps.jsonl"
        if rank == 0:
            repair_step_log(log_path, cursor["global_step"])
            print(f"Model A: {model.parameter_count:,} unique trainable parameters", flush=True)
            print(f"Resume: epoch={cursor['epoch']} next_batch={cursor['next_batch']} "
                  f"step={cursor['global_step']} optimizer_step={cursor['optimizer_steps']}", flush=True)
        run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                            cursor, cfg, device, rank, stop_event, started, log_path)
    finally:
        dist.destroy_process_group()


''')


[Markdown Cell]
Automatically save every **500 completed accumulation groups** and **at every epoch end**. SIGINT/SIGTERM or the 10.5-hour limit requests a save at the next update boundary. An abrupt kill can lose up to 499 groups after the last checkpoint. Resume is exact from the saved cursor; working files must be preserved across sessions.


In [26]:
# [Code Cell]
append_module('train.py', r'''def run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                        cursor: dict, cfg: TrainConfig, device: torch.device, rank: int,
                        stop_event, started: float, log_path: Path) -> None:
    epoch_bar = tqdm(total=cfg.epochs, initial=cursor["epoch"], desc="Model A epochs", disable=rank != 0)
    stopped = False
    try:
        while cursor["epoch"] < cfg.epochs:
            loader, sampler = training_loader(dataset, cfg, cursor["epoch"], cursor["next_batch"], rank)
            iterator = iter(loader)
            total_updates = math.ceil(sampler.num_batches / cfg.accumulation_steps)
            step_bar = tqdm(total=total_updates,
                            initial=math.ceil(cursor["next_batch"] / cfg.accumulation_steps),
                            desc=f"Epoch {cursor['epoch'] + 1}/{cfg.epochs}", disable=rank != 0,
                            mininterval=1, leave=False)
            try:
                while cursor["next_batch"] < sampler.num_batches:
                    count = min(cfg.accumulation_steps, sampler.num_batches - cursor["next_batch"])
                    batches = [next(iterator) for _ in range(count)]
                    metrics = train_one_update(ddp, optimizer, scaler, scheduler, batches, cfg, device)
                    cursor["next_batch"] += count
                    cursor["global_step"] += 1
                    cursor["optimizer_steps"] += int(not metrics["overflow"])
                    cursor["epoch_loss_sum"] += metrics["loss_sum"]
                    cursor["epoch_tokens"] += metrics["tokens"]
                    cursor["epoch_overflows"] += int(metrics["overflow"])
                    cursor["consecutive_overflows"] = cursor["consecutive_overflows"] + 1 if metrics["overflow"] else 0
                    assert cursor["optimizer_steps"] == scheduler.steps
                    if rank == 0:
                        with log_path.open("a", encoding="utf-8") as stream:
                            row = {"epoch": cursor["epoch"], "step": cursor["global_step"],
                                   "optimizer_step": cursor["optimizer_steps"],
                                   "loss": metrics["loss"], "learning_rate": metrics["learning_rate"],
                                   "tokens": metrics["tokens"], "overflow": metrics["overflow"]}
                            stream.write(json.dumps(row, allow_nan=False) + "\n")
                        step_bar.update(1)
                        step_bar.set_postfix(step=cursor["global_step"], loss=f"{metrics['loss']:.4f}",
                                             lr=f"{metrics['learning_rate']:.2e}")
                    stopping = any_rank(stop_event.is_set() or time.monotonic() - started >= cfg.max_run_seconds, device)
                    repeated_overflow = cursor["consecutive_overflows"] >= cfg.max_consecutive_overflows
                    if cursor["global_step"] % cfg.checkpoint_every_steps == 0 or stopping or repeated_overflow:
                        manager.save(model, optimizer, scaler, scheduler, cursor, device, rank)
                    if repeated_overflow:
                        raise FloatingPointError("Repeated AMP overflows; checkpoint saved. Inspect the scale/LR before resuming.")
                    if stopping:
                        stopped = True
                        break
            finally:
                step_bar.close()
                del iterator, loader
            if stopped:
                break
            validation = evaluate(model, cfg, device, rank, cfg.world_size)
            improved = validation["conditional_nll"] < cursor["best_validation_nll"]
            if improved:
                cursor["best_validation_nll"] = validation["conditional_nll"]
            record = {"epoch": cursor["epoch"] + 1,
                      "train_nll": cursor["epoch_loss_sum"] / cursor["epoch_tokens"],
                      "training_target_tokens": cursor["epoch_tokens"],
                      "overflows": cursor["epoch_overflows"], **validation}
            cursor["history"].append(record)
            cursor.update(epoch=cursor["epoch"] + 1, next_batch=0, epoch_loss_sum=0.0,
                          epoch_tokens=0, epoch_overflows=0)
            manager.save(model, optimizer, scaler, scheduler, cursor, device, rank, best=improved)
            if rank == 0:
                epoch_bar.update(1); epoch_bar.set_postfix(val_nll=f"{validation['conditional_nll']:.4f}")
                print(json.dumps(record), flush=True)
        if rank == 0:
            atomic_json(Path(cfg.run_dir) / "training_summary.json", {
                "status": "paused_at_saved_update" if stopped else "complete",
                "epoch": cursor["epoch"], "next_batch": cursor["next_batch"],
                "step": cursor["global_step"], "optimizer_steps": cursor["optimizer_steps"],
                "history": cursor["history"], "fingerprint": manager.fingerprint,
            })
    finally:
        epoch_bar.close()


''')


[Markdown Cell]
Require exactly two T4s and an NCCL-enabled PyTorch build compatible with the
saved checkpoint. The complete saved training state is copied into this run's
working output, so the next saved notebook output is independently resumable.
The tokenized corpus remains in read-only Input storage.


In [27]:
# [Code Cell]
append_module('train.py', r'''def require_dual_t4() -> None:
    version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
    assert version >= (2, 5), "Use Kaggle's current PyTorch CUDA runtime (>=2.5)."
    assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select Kaggle accelerator T4 x2."
    assert dist.is_nccl_available(), "The PyTorch build needs NCCL support."
    for rank in range(2):
        name = torch.cuda.get_device_name(rank)
        assert "T4" in name, f"Expected T4 on device {rank}: {name}"
        print(f"GPU {rank}: {name}, {torch.cuda.get_device_properties(rank).total_memory / 2**30:.1f} GiB")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    cfg = resolve_training_config(TrainConfig.load(args.config)); cfg.validate()
    require_dual_t4()
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    with exclusive_run_lock(Path(cfg.run_dir)):
        contract = verify_data_contract(cfg)
        report = prepare_indices(cfg, contract)
        fingerprint = runtime_fingerprint(cfg, contract)
        atomic_json(Path(cfg.run_dir) / "data_report.json", report)
        with torch.device("meta"):
            count = assert_parameter_boundary(LegalCausalDecoder(cfg))
        print(f"Verified Model A parameter count: {count:,}", flush=True)
        # Uncommitted pipeline-owned temporaries cannot be restored.
        checkpoint_dir = Path(cfg.run_dir) / "checkpoints"
        if checkpoint_dir.exists():
            for name in ("slot_0.pt.partial", "slot_1.pt.partial", "best.pt.partial",
                         "slot_0.pt.json.partial", "slot_1.pt.json.partial", "best.pt.json.partial"):
                (checkpoint_dir / name).unlink(missing_ok=True)
        context = mp.get_context("spawn")
        stop_event = context.Event()
        signal.signal(signal.SIGTERM, lambda *_: stop_event.set())
        signal.signal(signal.SIGINT, lambda *_: stop_event.set())
        with socket.socket() as sock:
            sock.bind(("127.0.0.1", 0)); port = sock.getsockname()[1]
        processes = mp.spawn(worker, args=(cfg, contract, fingerprint, port, stop_event),
                             nprocs=cfg.world_size, join=False)
        while not processes.join(timeout=1):
            pass


if __name__ == "__main__":
    main()
''')


[Markdown Cell]
## 9. Continuation generation
Greedy decoding suppresses control tokens and uses rotary K/V caching. Generate an extra word so the 128th word is closed, then export exactly 128 words without the prompt. Exceeding the token/context cap fails clearly and keeps completed predictions; no filler text is invented.


In [28]:
# [Code Cell]
append_module('infer.py', r'''import argparse
from pathlib import Path
import hashlib
import json
import os
import shutil
import sqlite3
import zipfile
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import TrainConfig, SOURCES, SPECIAL_TOKENS, resolve_training_config
from .data import read_json, sha256_file, verify_data_contract
from .model import LegalCausalDecoder, decode_one_token
from .state import CheckpointManager, atomic_json, runtime_fingerprint, seed_everything, exclusive_run_lock
from .train import autocast_context


@torch.inference_mode()
def generate_continuation(model: LegalCausalDecoder, tokenizer: Tokenizer, prompt: str,
                          device: torch.device, token_limit: int, target_words: int = 128) -> str:
    """Generate until an extra word closes the requested final word; return 128.

    Token count is not word count. EOS/control tokens are suppressed; exhausting
    the context/token bound fails explicitly instead of inventing padding text.
    """
    model.eval()
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    ids = [1] + prompt_ids[-(model.cfg.max_prompt_tokens - 1):]
    token_ids = torch.tensor([ids], device=device)
    generated = []
    limit = min(token_limit, model.cfg.max_seq_tokens - len(ids) + 1)
    forbidden = [tokenizer.token_to_id(special) for special in SPECIAL_TOKENS]
    with autocast_context(device):
        logits, cache = model.prefill(token_ids)
    for step in range(limit):
        if step:
            with autocast_context(device):
                logits = decode_one_token(model, token, cache)
        logits = logits.float()
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite generation logits")
        logits[:, forbidden] = float("-inf")
        chosen = int(logits.argmax(dim=-1).item())
        generated.append(chosen)
        token = torch.tensor([[chosen]], device=device)
        if (step + 1) % 8 == 0 or step + 1 == limit:
            text = tokenizer.decode(generated, skip_special_tokens=True)
            words = text.split()
            if len(words) >= target_words + 1:
                continuation = " ".join(words[:target_words])
                assert len(continuation.split()) == target_words
                return continuation
    raise RuntimeError(f"Could not close {target_words} words within {limit} generated tokens. "
                       "Inspect training/generation quality or use a larger causal context capacity in a new experiment.")


''')


[Markdown Cell]
Use the full best held-out checkpoint when available; otherwise use the latest committed checkpoint. Verify its receipts and compatibility before loading weights. The tokenizer remains identical to training.


In [29]:
# [Code Cell]
append_module('infer.py', r'''def load_inference_model(cfg: TrainConfig, fingerprint: str, device: torch.device):
    manager = CheckpointManager(cfg, fingerprint)
    latest = manager.latest()
    if latest is None:
        raise FileNotFoundError("No trained checkpoint exists. Run distributed training first.")
    chosen = latest
    roots = [manager.directory] + ([Path(cfg.resume_dir)] if cfg.resume_dir else [])
    best_candidates = []
    for root in roots:
        path, receipt_path = root / "best.pt", root / "best.pt.json"
        if path.exists() and receipt_path.exists():
            try:
                receipt = read_json(receipt_path)
                assert receipt["fingerprint"] == fingerprint and path.stat().st_size == receipt["bytes"]
                assert sha256_file(path) == receipt["sha256"]
                score = receipt.get("validation_nll")
                assert score is None or isinstance(score, (int, float))
                best_candidates.append((path, receipt))
            except (OSError, ValueError, TypeError, KeyError, AssertionError):
                continue
    if best_candidates:
        chosen = min(best_candidates, key=lambda item: (
            item[1].get("validation_nll") if item[1].get("validation_nll") is not None else float("inf"),
            -item[1]["step"], -item[1]["generation"]))
    payload = torch.load(chosen[0], map_location="cpu", weights_only=True)
    assert payload["fingerprint"] == fingerprint
    model = LegalCausalDecoder(cfg)
    model.load_state_dict(payload["model"], strict=True)
    model.to(device).eval()
    receipt = chosen[1]
    del payload
    return model, receipt


''')


[Markdown Cell]
Generate two held-out examples per legal source by default. Optional official inference reads the attached prompt JSON and preserves exact string IDs. Output records contain only `text` with the continuation, matching the supplied evaluator.


In [30]:
# [Code Cell]
append_module('infer.py', r'''def generation_records(cfg: TrainConfig, kind: str, input_json: str | None):
    """Use exact official string IDs; development records come only from held-out data."""
    if kind == "test":
        if input_json is None:
            raise ValueError("Provide --input-json pointing to the official Phase-1 test prompts.")
        payload = read_json(Path(input_json))
        assert isinstance(payload, dict) and payload
        for identifier, row in payload.items():
            assert isinstance(identifier, str) and isinstance(row, dict)
            assert isinstance(row.get("text"), str) and len(row["text"].split()) == 512
            yield {"id": identifier, "prompt": row["text"]}
    else:
        counts = dict.fromkeys(SOURCES, 0)
        with (Path(cfg.data_dir) / "validation_pairs.jsonl").open(encoding="utf-8") as stream:
            for line in stream:
                row = json.loads(line)
                source = row["source"]
                if counts[source] < cfg.generation_examples_per_source:
                    counts[source] += 1
                    yield row
        assert all(count == cfg.generation_examples_per_source for count in counts.values()), counts


def export_predictions(path: Path, predictions: dict) -> None:
    assert predictions and all(len(row["text"].split()) == 128 for row in predictions.values())
    atomic_json(path, predictions)


''')


[Markdown Cell]
Commit each completed prediction to SQLite. Resume skips entries only when checkpoint, prompt, and decoding settings match. Official inference creates `prediction.json` and a ZIP containing it at the archive root. No external submission is performed.


In [31]:
# [Code Cell]
append_module('infer.py', r'''def run_generation(cfg: TrainConfig, kind: str, input_json: str | None = None,
                   device: torch.device | None = None) -> None:
    cfg.validate()
    if device is None:
        assert torch.cuda.is_available(), "Generation requires the Kaggle CUDA runtime."
        device = torch.device("cuda", 0)
    seed_everything(cfg.seed, cfg.deterministic)
    contract = verify_data_contract(cfg)
    fingerprint = runtime_fingerprint(cfg, contract)
    model, receipt = load_inference_model(cfg, fingerprint, device)
    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))
    directory = Path(cfg.run_dir) / "generation" / kind
    directory.mkdir(parents=True, exist_ok=True)
    records = list(generation_records(cfg, kind, input_json))
    assert len({row["id"] for row in records}) == len(records)
    prompt_truncations = sum(len(tokenizer.encode(row["prompt"], add_special_tokens=False).ids)
                             > cfg.max_prompt_tokens - 1 for row in records)
    namespace = hashlib.sha256(json.dumps({"checkpoint": receipt["sha256"], "kind": kind,
                "tokens": cfg.generation_token_limit, "words": 128}, sort_keys=True).encode()).hexdigest()
    connection = sqlite3.connect(directory / "generation_cache.sqlite")
    connection.execute("PRAGMA journal_mode=WAL")
    connection.execute("PRAGMA synchronous=FULL")
    connection.execute("CREATE TABLE IF NOT EXISTS predictions (namespace TEXT, id TEXT, "
                       "prompt_sha TEXT, text TEXT, PRIMARY KEY(namespace,id))")
    predictions, truth = {}, {}
    try:
        for row in tqdm(records, desc=f"Model A {kind}: 128-word continuations"):
            prompt_sha = hashlib.sha256(row["prompt"].encode()).hexdigest()
            cached = connection.execute("SELECT prompt_sha,text FROM predictions WHERE namespace=? AND id=?",
                                        (namespace, row["id"])).fetchone()
            if cached is not None and cached[0] == prompt_sha and len(cached[1].split()) == 128:
                text = cached[1]
            else:
                if shutil.disk_usage(directory).free < cfg.min_free_bytes + (1 << 20):
                    raise OSError("Insufficient disk space for durable generation outputs.")
                text = generate_continuation(model, tokenizer, row["prompt"], device, cfg.generation_token_limit)
                connection.execute("INSERT OR REPLACE INTO predictions VALUES (?,?,?,?)",
                                   (namespace, row["id"], prompt_sha, text))
                connection.commit()  # Each completed example survives a later failure.
            predictions[row["id"]] = {"text": text}
            if "reference" in row:
                truth[row["id"]] = {"text": row["reference"]}
        prediction_path = directory / "prediction.json"
        export_predictions(prediction_path, predictions)
        if truth:
            assert truth.keys() == predictions.keys()
            atomic_json(directory / "truth.json", truth)
        if kind == "test":
            temporary = directory / "submission_model_a.zip.partial"
            with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
                archive.write(prediction_path, arcname="prediction.json")
            with temporary.open("r+b") as stream:
                os.fsync(stream.fileno())
            os.replace(temporary, directory / "submission_model_a.zip")
        atomic_json(directory / "generation_report.json", {
            "checkpoint_sha256": receipt["sha256"], "checkpoint_epoch": receipt["epoch"],
            "checkpoint_step": receipt["step"], "examples": len(predictions),
            "words_per_continuation": 128, "decoding": "greedy with cached rotary self-attention K/V",
            "prompt_tail_limit": cfg.max_prompt_tokens - 1,
            "prompt_truncations": prompt_truncations,
            "token_limit": cfg.generation_token_limit, "data_fingerprint": contract["fingerprint"],
        })
        print(f"Saved {len(predictions)} continuations to {prediction_path}", flush=True)
    finally:
        connection.close()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--kind", choices=("validation", "test"), default="validation")
    parser.add_argument("--input-json")
    args = parser.parse_args()
    cfg = resolve_training_config(TrainConfig.load(args.config))
    with exclusive_run_lock(Path(cfg.run_dir)):
        run_generation(cfg, args.kind, args.input_json)


if __name__ == "__main__":
    main()
''')


[Markdown Cell]
## 10. Sanity checks
Exercise the actual production architecture on short sequences: unique parameter count, causal isolation, right padding, relative rotary positions, cached/full logits, dense/chunked loss, shifted labels, and gradients through attention, SwiGLU, and tied vocabulary. GPU 0 may be used briefly, then memory is released before DDP starts.


In [32]:
# [Code Cell]
append_module('sanity.py', r'''import gc
import torch
from torch.nn import functional as F
from .config import TrainConfig, assert_parameter_boundary
from .model import LegalCausalDecoder, RotaryEmbedding, decode_one_token
from .data import collate_continuations


def architecture_sanity(cfg: TrainConfig, device: torch.device | None = None) -> dict:
    """Exercise the full production architecture on short sequences before DDP."""
    device = torch.device("cpu") if device is None else device
    threads = torch.get_num_threads()
    torch.set_num_threads(min(4, threads))
    try:
        with torch.device("meta"):
            count = assert_parameter_boundary(LegalCausalDecoder(cfg))
        expected = cfg.vocab_size * cfg.d_model + cfg.n_layers * (
            4 * cfg.d_model ** 2 + 3 * cfg.d_model * cfg.d_ff + 2 * cfg.d_model) + cfg.d_model
        assert count == expected
        with torch.device(device):
            model = LegalCausalDecoder(cfg).eval()
        ids = torch.randint(6, cfg.vocab_size, (2, 12), device=device)
        mask = torch.ones_like(ids, dtype=torch.bool)
        mask[1, -3:] = False
        with torch.inference_mode():
            logits = model(ids, mask)
            assert logits.shape == (2, 12, cfg.vocab_size) and torch.isfinite(logits).all()
            changed = ids.clone(); changed[:, 6:] = (changed[:, 6:] + 11) % cfg.vocab_size
            torch.testing.assert_close(logits[:, :6], model(changed, mask)[:, :6], rtol=1e-4, atol=3e-6)
            padded = ids.clone(); padded[1, -3:] = 9
            torch.testing.assert_close(logits[1, :9], model(padded, mask)[1, :9], rtol=1e-4, atol=3e-6)
            unpadded = ids[:1]
            full = model(unpadded)
            first, cache = model.prefill(unpadded[:, :4])
            cached = [first]
            for position in range(4, unpadded.shape[1]):
                cached.append(decode_one_token(model, unpadded[:, position:position + 1], cache))
            incremental = torch.stack(cached, dim=1)
            torch.testing.assert_close(full[:, 3:], incremental, rtol=1e-4, atol=1e-5)
            cache_error = float((full[:, 3:] - incremental).abs().max())
            labels = torch.randint(6, cfg.vocab_size, ids.shape, device=device)
            labels[1, -3:] = -100
            labels[:, :2] = -100
            dense_loss = F.cross_entropy(logits.flatten(0, 1).float(), labels.flatten(),
                                         ignore_index=-100, reduction="sum")
            chunk_loss = model(ids, mask, labels)
            torch.testing.assert_close(dense_loss, chunk_loss, rtol=1e-5, atol=1e-4)
            rotary = RotaryEmbedding(8, cfg.rope_base)
            q, k = torch.randn(1, 2, 4, 8, device=device), torch.randn(1, 2, 4, 8, device=device)
            positions = torch.arange(4, device=device)
            q1, k1 = rotary(q, k, positions)
            q2, k2 = rotary(q, k, positions + 17)
            torch.testing.assert_close(q1 @ k1.transpose(-1, -2), q2 @ k2.transpose(-1, -2),
                                       rtol=1e-4, atol=4e-6)
        batch = collate_continuations([
            {"input_ids": [1, 6, 7], "labels": [6, 7, 2]},
            {"input_ids": [1, 8], "labels": [-100, 9]},
        ])
        assert batch["labels"][1].tolist() == [-100, 9, -100]
        assert batch["attention_mask"][1].tolist() == [True, True, False]
        model.train()
        batch = {name: value.to(device) for name, value in batch.items()}
        loss = model(batch["input_ids"], batch["attention_mask"], batch["labels"])
        loss.backward()
        for parameter in (model.token_embedding.weight, model.layers[0].attention.qkv.weight,
                          model.layers[-1].ffn.gate.weight):
            assert parameter.grad is not None and torch.isfinite(parameter.grad).all()
            assert parameter.grad.abs().sum() > 0
        result = {"parameters": count, "logit_shape": list(logits.shape), "device": str(device),
                  "causal_isolation": "passed", "right_padding": "passed", "rotary_relative_positions": "passed",
                  "cached_logits_max_abs_error": cache_error, "dense_vs_chunk_loss": "passed",
                  "activation_checkpointed_gradient_flow": "passed"}
        del model, loss, logits, full, incremental, cache, chunk_loss, dense_loss
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return result
    finally:
        torch.set_num_threads(threads)
''')


[Markdown Cell]
## 11. Discover the saved output and prepare continuation
The next cell chooses the latest valid checkpoint, restores its original
configuration, locates the same tokenized data, and preserves the original code.
It prints the saved batch/step and target epoch before any GPU training.
Only the latest recovery state and the previous best snapshot are copied;
two recovery slots continue rotating as training proceeds.


In [33]:
# [Code Cell]
import types
CONTINUATION_SUPPORT_SOURCE = r'''"""Prepare full-state resume and verified runtime migration with unchanged model code."""
from dataclasses import asdict, replace
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import shutil
import sys
import types
import torch

SESSION_VERSION = "model-a-continuation-session-v1"
CHECKPOINT_NAMES = ("slot_0.pt", "slot_1.pt", "best.pt")
FINGERPRINT_EXCLUSIONS = (
    "data_dir", "run_dir", "resume_dir", "initial_checkpoint", "max_run_seconds", "min_free_bytes",
    "max_checkpoint_bytes", "generation_examples_per_source", "generation_token_limit",
)
# This is the specific stable CUDA build upgrade observed in the user's Kaggle run.
# Unrelated major, minor, nightly, CUDA-build and downgrade changes still fail.
SUPPORTED_TORCH_UPGRADES = {("2.10.0+cu128", "2.11.0+cu128")}


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while block := stream.read(8 << 20):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8", newline="\n") as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def parent_root(selected):
    path = Path(selected).expanduser().resolve()
    if path.name in CHECKPOINT_NAMES and path.parent.name == "checkpoints":
        path = path.parent.parent
    elif path.name in ("checkpoints", "train_config.json"):
        path = path.parent
    if not (path / "train_config.json").is_file():
        raise FileNotFoundError(
            f"No saved Model A output at {path}. Attach the previous notebook's OUTPUT "
            "with train_config.json, code/model_a and checkpoints/*.pt + *.pt.json. "
            "Uploading an .ipynb alone does not provide trained weights."
        )
    return path


def discover_parent(input_root, selected=None):
    if selected is not None:
        return parent_root(selected)
    matches = sorted({p.parent.resolve() for p in Path(input_root).rglob("train_config.json")
                      if (p.parent / "code/model_a/model.py").is_file()
                      and (p.parent / "checkpoints").is_dir()})
    if len(matches) != 1:
        if not matches:
            raise FileNotFoundError(
                "No saved Model A output found. In Kaggle, Add Input -> Notebooks and "
                "attach the saved Model A decoder version shown in your screenshot. "
                "An .ipynb alone contains no model weights; this notebook never starts from scratch."
            )
        raise ValueError(
            "Several Model A outputs are attached. Attach only the latest one, or set "
            f"PARENT_RUN_DIR to its model_a_run/model_a_resume_run folder: {matches}"
        )
    return matches[0]


def intact_checkpoints(run):
    choices, problems = [], []
    for name in CHECKPOINT_NAMES:
        path = Path(run) / "checkpoints" / name
        receipt_path = path.with_name(name + ".json")
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt = read_json(receipt_path)
            if not all(type(receipt.get(key)) is int for key in
                       ("bytes", "step", "generation", "epoch")):
                raise ValueError("invalid receipt cursor")
            if (receipt["bytes"] <= 0 or receipt["step"] < 0 or
                    receipt["generation"] <= 0 or receipt["epoch"] < 0):
                raise ValueError("invalid receipt values")
            if path.stat().st_size != receipt["bytes"] or sha256(path) != receipt["sha256"]:
                raise ValueError("size/SHA-256 mismatch")
            choices.append((path, receipt))
        except (OSError, ValueError, TypeError, KeyError) as error:
            problems.append(f"{name}: {error}")
    if not choices:
        raise RuntimeError(f"No intact committed checkpoint in {run}. {problems}")
    if problems:
        print("Ignoring incomplete checkpoint files:", problems, flush=True)
    return choices


def latest_checkpoint(run):
    choices = intact_checkpoints(run)
    selected = max(choices, key=lambda pair: (pair[1]["step"], pair[1]["generation"]))
    print(f"Using latest committed checkpoint: {selected[0]} (step {selected[1]['step']})",
          flush=True)
    return selected, choices


def learning_choices(values):
    return {key: value for key, value in values.items() if key not in FINGERPRINT_EXCLUSIONS}


def verified_saved_runtime(run, config_type, resolve_data, template_sources, verified_runtime_sources=None):
    """Choose only an exact known runtime, and use that runtime's configuration schema."""
    sources = {p.name: p.read_bytes() for p in (Path(run) / "code/model_a").glob("*.py")}
    actual_hashes = {name: hashlib.sha256(body).hexdigest() for name, body in sources.items()}
    references = {"original_decoder": template_sources}
    if verified_runtime_sources:
        references.update(verified_runtime_sources)
    for name, reference in references.items():
        expected = {filename: hashlib.sha256(body.encode("utf-8")).hexdigest()
                    for filename, body in reference.items()}
        if actual_hashes != expected:
            continue
        if name != "original_decoder":
            # Execute the audited, embedded reference after verifying every saved file.
            # Register its module because dataclasses resolves annotations through sys.modules.
            module_name = "_verified_model_a_config_" + expected["config.py"]
            module = types.ModuleType(module_name)
            module.__file__ = str(Path(run) / "code/model_a/config.py")
            sys.modules[module_name] = module
            exec(compile(reference["config.py"], module.__file__, "exec"), module.__dict__)
            config_type = module.TrainConfig
            resolve_data = module.resolve_preprocessed_data_dir
        print(f"Verified saved runtime: {name}; retaining its code and training settings", flush=True)
        return sources, actual_hashes, config_type, resolve_data, name
    changed = {name: sorted(filename for filename in set(actual_hashes) | set(reference)
                           if actual_hashes.get(filename) != hashlib.sha256(reference.get(filename, "").encode()).hexdigest())
               for name, reference in references.items()}
    raise ValueError(f"Attached Model A runtime matches neither the original nor the supplied two-epoch "
                     f"runtime. Differing files: {changed}. Attach the output from ModelA_retrain_epoch12 "
                     "or a later output of this fixed continuation notebook.")


def data_fingerprint(data_dir):
    root = Path(data_dir)
    metadata = read_json(root / "metadata.json")
    required = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",
                "validation_documents.jsonl", "validation_pairs.jsonl")
    identity = {name: metadata["artifacts"][name]["sha256"] for name in required}
    identity["metadata.json"] = sha256(root / "metadata.json")
    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()


def expected_fingerprint(values, data_identity, source_hashes, torch_version):
    payload = {"version": "cs781-model-a-v1", "config": learning_choices(values),
               "data": data_identity, "sources": source_hashes, "torch": torch_version}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def read_checkpoint_info(path, receipt):
    # mmap keeps 1.32GB checkpoints out of the notebook's resident memory.
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    try:
        if (payload["fingerprint"] != receipt["fingerprint"] or
                payload["step"] != receipt["step"] or payload["epoch"] != receipt["epoch"]):
            raise ValueError("Checkpoint payload and receipt disagree")
        config, cursor = payload["config"], payload["cursor"]
        if (not all(type(cursor.get(key)) is int and cursor[key] >= 0 for key in
                    ("epoch", "next_batch", "global_step", "optimizer_steps")) or
                type(config.get("epochs")) is not int or config["epochs"] < 1 or
                not 0 <= cursor["epoch"] <= config["epochs"] or
                cursor["epoch"] != payload["epoch"] or cursor["global_step"] != payload["step"] or
                cursor["optimizer_steps"] > cursor["global_step"] or
                payload["scheduler"]["steps"] != cursor["optimizer_steps"] or
                payload["world_size"] != config["world_size"] or
                len(payload["rng_by_rank"]) != config["world_size"] or
                (cursor["epoch"] == config["epochs"] and cursor["next_batch"] != 0)):
            raise ValueError("Saved training cursor/scheduler/RNG state is inconsistent")
        for key in ("model", "optimizer", "scaler", "scheduler", "rng_by_rank"):
            if key not in payload:
                raise ValueError(f"Checkpoint is missing full recovery state: {key}")
        return {"config": dict(config), "cursor": dict(cursor), "scheduler": dict(payload["scheduler"]),
                "runtime_migration_history": list(payload.get("runtime_migration_history", []))}
    finally:
        del payload
        gc.collect()


def copy_committed_checkpoint(source, receipt, destination):
    target = Path(destination) / source.name
    temporary = target.with_name(target.name + ".partial")
    print(f"Preserving {source.name} in the new output (tokenized data stays in Input)", flush=True)
    shutil.copyfile(source, temporary)
    if temporary.stat().st_size != receipt["bytes"] or sha256(temporary) != receipt["sha256"]:
        raise OSError(f"Checkpoint changed while copying: {source}")
    os.replace(temporary, target)
    write_json(target.with_name(target.name + ".json"), receipt)


def resolve_runtime_transition(values, identity, source_hashes, receipt_fingerprint,
                               recorded_torch, original_torch, allow_torch_upgrade):
    current = str(torch.__version__)
    target_fingerprint = expected_fingerprint(values, identity, source_hashes, current)
    if receipt_fingerprint == target_fingerprint:
        return target_fingerprint, None
    candidates = dict.fromkeys((recorded_torch, original_torch,
                               *(before for before, after in SUPPORTED_TORCH_UPGRADES)))
    matched = next((version for version in candidates if isinstance(version, str) and
                    expected_fingerprint(values, identity, source_hashes, version) == receipt_fingerprint), None)
    if matched is None:
        raise ValueError(
            "Checkpoint compatibility check failed before GPU training. Its fingerprint cannot "
            "be reproduced from the unchanged saved code, settings, data and recorded PyTorch build. "
            "A version upgrade cannot bypass a data/model/configuration mismatch."
        )
    if not allow_torch_upgrade or (matched, current) not in SUPPORTED_TORCH_UPGRADES:
        raise ValueError(
            "Checkpoint compatibility check failed before GPU training. "
            f"Saved PyTorch is {matched}, current is {current}. "
            "The supported migration is 2.10.0+cu128 -> 2.11.0+cu128 with "
            "ALLOW_TORCH_UPGRADE=True; other runtime changes require a matching environment."
        )
    migration = {"from_torch": matched, "to_torch": current,
                 "source_fingerprint": receipt_fingerprint, "target_fingerprint": target_fingerprint,
                 "model_data_training_settings": "unchanged_and_verified",
                 "training_state": "weights_optimizer_scaler_scheduler_cursor_history_rank_rng_retained",
                 "bitwise_continuation_guaranteed": False}
    print(f"Migrating full saved training state: PyTorch {matched} -> {current}. "
          "Only new working checkpoints are written; original Input files stay intact. "
          "Numerical results may differ across PyTorch versions.", flush=True)
    return target_fingerprint, migration


def migrate_committed_checkpoint(source, receipt, destination, migration, expected_config=None,
                                 updated_config=None, scheduler_plan=None, override_learning_rate=None):
    """Write a new valid checkpoint/receipt pair; retain every training-state value."""
    target = Path(destination) / source.name
    temporary = target.with_name(target.name + ".partial")
    # Validate full recovery metadata before rewriting even an older best/recovery slot.
    info = read_checkpoint_info(source, receipt)
    if expected_config is not None and learning_choices(info["config"]) != learning_choices(expected_config):
        raise ValueError("Recovery/best checkpoint training settings differ from the verified latest state")
    payload = torch.load(source, map_location="cpu", weights_only=True, mmap=True)
    try:
        if payload["fingerprint"] != migration["source_fingerprint"]:
            raise ValueError("Checkpoint is not from the verified source runtime")
        payload["fingerprint"] = migration["target_fingerprint"]
        if updated_config is not None:
            payload["config"] = dict(updated_config)
        if scheduler_plan is not None:
            payload["scheduler"].update(scheduler_plan)  # Keep the global scheduler step counter.
        if override_learning_rate is not None:
            for group in payload["optimizer"]["param_groups"]:
                group["lr"] = override_learning_rate
        history = list(payload.get("runtime_migration_history", []))
        history.append({**migration, "source_sha256": receipt["sha256"]})
        payload["runtime_migration_history"] = history
        print(f"Writing migrated {source.name}; saved step {receipt['step']} is retained", flush=True)
        with temporary.open("wb") as stream:
            torch.save(payload, stream)
            stream.flush()
            os.fsync(stream.fileno())
        migrated_receipt = {**receipt, "bytes": temporary.stat().st_size,
                            "sha256": sha256(temporary), "fingerprint": payload["fingerprint"]}
        # Detect an input changing during migration, including an owned in-place upgrade.
        if sha256(source) != receipt["sha256"]:
            raise OSError(f"Source checkpoint changed during migration: {source}")
        os.replace(temporary, target)
        write_json(target.with_name(target.name + ".json"), migrated_receipt)
        return migrated_receipt
    finally:
        del payload
        gc.collect()


def prepare_continuation(config_type, resolve_data, template_sources,
                         run_dir="/kaggle/working/model_a_resume_run", input_root="/kaggle/input",
                         parent_run=None, data_dir=None, epochs_this_run=1,
                         max_run_seconds=37_800, original_torch="2.10.0+cu128",
                         allow_torch_upgrade=True):
    if type(epochs_this_run) is not int or epochs_this_run < 1:
        raise ValueError("EPOCHS_THIS_RUN must be a positive integer")
    if type(max_run_seconds) is not int or max_run_seconds < 1:
        raise ValueError("MAX_RUN_SECONDS must be a positive integer")
    if type(allow_torch_upgrade) is not bool:
        raise ValueError("ALLOW_TORCH_UPGRADE must be True or False")
    working = Path(run_dir).expanduser().resolve()
    mounted = Path(input_root).expanduser().resolve()
    if working == mounted or mounted in working.parents:
        raise ValueError("RUN_DIR must be writable working storage, outside the read-only Input tree")
    manifest_path = working / "continuation_session.json"
    existing = any((working / "checkpoints" / name).exists() for name in CHECKPOINT_NAMES)
    if existing and not manifest_path.is_file():
        raise ValueError("Working checkpoints lack their continuation_session.json; use a fresh RUN_DIR")
    run = parent_root(working) if existing else discover_parent(input_root, parent_run)
    if run == working and not existing:
        raise ValueError("The parent must be an attached saved output, not an empty working directory")

    saved_sources = {p.name: p.read_bytes() for p in (run / "code/model_a").glob("*.py")}
    source_hashes = {name: hashlib.sha256(body.encode("utf-8")).hexdigest()
                     for name, body in template_sources.items()}
    saved_hashes = {name: hashlib.sha256(body).hexdigest() for name, body in saved_sources.items()}
    if saved_hashes != source_hashes:
        raise ValueError(
            "The attached output uses different model/runtime code. Attach the original Model A "
            "decoder output or an output from this continuation notebook; its fingerprinted code "
            "must be preserved exactly."
        )
    (path, receipt), choices = latest_checkpoint(run)
    info = read_checkpoint_info(path, receipt)
    values, cursor = info["config"], info["cursor"]
    if learning_choices(read_json(run / "train_config.json")) != learning_choices(values):
        raise ValueError("train_config.json disagrees with the committed checkpoint configuration")
    saved_cfg = config_type(**values)
    saved_cfg.validate()
    data_path = resolve_data(data_dir or saved_cfg.data_dir, input_root)
    identity = data_fingerprint(data_path)
    report_path = run / "data_report.json"
    if report_path.is_file() and read_json(report_path).get("data_fingerprint") != identity:
        raise ValueError("Attach the SAME tokenized dataset/tokenizer used by the saved model")
    runtime_identity = run / "continuation_runtime_identity.json"
    saved_torch = (read_json(runtime_identity)["torch"] if runtime_identity.is_file() else original_torch)
    fingerprint, migration = resolve_runtime_transition(
        values, identity, source_hashes, receipt["fingerprint"], saved_torch,
        original_torch, allow_torch_upgrade)

    if existing:
        session = read_json(manifest_path)
        allowed_session_fingerprints = {fingerprint, receipt["fingerprint"]}
        if allow_torch_upgrade and str(torch.__version__) == "2.11.0+cu128":
            # Also recover an interrupted preparation where some pairs were migrated.
            allowed_session_fingerprints.add(expected_fingerprint(values, identity, source_hashes, "2.10.0+cu128"))
        if (session.get("version") != SESSION_VERSION or session.get("fingerprint") not in allowed_session_fingerprints or
                type(session.get("target_epoch")) is not int or
                not cursor["epoch"] <= session["target_epoch"] <= saved_cfg.epochs):
            raise ValueError("Continuation session manifest disagrees with the working checkpoint")
    else:
        if cursor["epoch"] >= saved_cfg.epochs:
            raise ValueError(
                f"The original {saved_cfg.epochs}-epoch plan is complete. This notebook resumes that "
                "plan without resetting or extending its learning-rate schedule."
            )
        session = {"version": SESSION_VERSION, "parent_run": str(run),
                   "parent_checkpoint": str(path), "parent_sha256": receipt["sha256"],
                   "fingerprint": fingerprint, "start_epoch": cursor["epoch"],
                   "start_next_batch": cursor["next_batch"], "start_step": cursor["global_step"],
                   "epochs_this_run": epochs_this_run,
                   "target_epoch": min(saved_cfg.epochs, cursor["epoch"] + epochs_this_run),
                   "original_plan_epochs": saved_cfg.epochs,
                   "meaning": "Finish the saved partial epoch; stop after the requested epoch boundaries"}
    session["fingerprint"] = fingerprint
    if migration is not None:
        session["runtime_migration"] = migration
    elif info["runtime_migration_history"]:
        session.setdefault("runtime_migration", info["runtime_migration_history"][-1])

    # Only non-fingerprinted paths and the original time limit are replaced.
    cfg = replace(saved_cfg, data_dir=str(data_path), run_dir=str(working),
                  resume_dir=None, max_run_seconds=max_run_seconds)
    cfg.validate()
    code_dir, checkpoint_dir = working / "code", working / "checkpoints"
    package = code_dir / "model_a"
    package.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(exist_ok=True)
    for name, body in saved_sources.items():
        (package / name).write_bytes(body)
    cfg.save(working / "train_config.json")
    write_json(manifest_path, session)
    write_json(working / "continuation_runtime_identity.json",
               {"torch": str(torch.__version__), "sources": source_hashes, "fingerprint": fingerprint,
                "runtime_migration": session.get("runtime_migration")})
    if not existing:
        selected = {path.name: (path, receipt)}
        for candidate, candidate_receipt in choices:
            if candidate.name == "best.pt" and candidate_receipt["fingerprint"] == receipt["fingerprint"]:
                selected[candidate.name] = (candidate, candidate_receipt)
        required_bytes = sum(r["bytes"] for _, r in selected.values())
        if shutil.disk_usage(working).free < required_bytes + cfg.min_free_bytes:
            raise OSError("Insufficient working disk space to preserve checkpoints")
        for candidate, candidate_receipt in selected.values():
            if migration is None:
                copy_committed_checkpoint(candidate, candidate_receipt, checkpoint_dir)
            else:
                migrate_committed_checkpoint(candidate, candidate_receipt, checkpoint_dir, migration, values)
        for name in ("training_steps.jsonl", "data_report.json"):
            if (run / name).is_file():
                shutil.copyfile(run / name, working / name)
    else:
        # Upgrade every older committed working slot so corruption fallback and best
        # inference also work after migration. Current-version pairs remain untouched.
        for candidate, candidate_receipt in choices:
            if candidate_receipt["fingerprint"] == fingerprint:
                continue
            _, candidate_migration = resolve_runtime_transition(
                values, identity, source_hashes, candidate_receipt["fingerprint"], saved_torch,
                original_torch, allow_torch_upgrade)
            if shutil.disk_usage(working).free < candidate_receipt["bytes"] + cfg.min_free_bytes:
                raise OSError("Insufficient disk space to atomically migrate a working checkpoint")
            migrate_committed_checkpoint(candidate, candidate_receipt, checkpoint_dir, candidate_migration, values)
    print(json.dumps({**session, "latest_saved_epoch": cursor["epoch"],
                      "latest_saved_next_batch": cursor["next_batch"],
                      "latest_saved_step": cursor["global_step"], "data_dir": str(data_path),
                      "output": str(working), "optimizer_scheduler_scaler_rng": "retained",
                      "already_at_session_target": cursor["epoch"] >= session["target_epoch"]}, indent=2),
          flush=True)
    return cfg, session


def prepare_extended_continuation(config_type, resolve_data, template_sources,
                                  run_dir="/kaggle/working/model_a_extended_run", input_root="/kaggle/input",
                                  parent_run=None, data_dir=None, additional_epochs=2, target_total_epochs=None,
                                  continuation_lr=None, max_run_seconds=37_800,
                                  original_torch="2.10.0+cu128", allow_torch_upgrade=True,
                                  verified_runtime_sources=None):
    """Continue any completed/partial run, retaining global counters and its schedule.

    Default schedule: retain the saved warmup/cosine curve, then keep its final LR
    after the old horizon. A declared constant LR override keeps Adam/scaler/RNGs
    and global scheduler steps; it never restarts warmup or optimizer moments.
    """
    if type(additional_epochs) is not int or additional_epochs < 1:
        raise ValueError("ADDITIONAL_EPOCHS must be a positive integer")
    if target_total_epochs is not None and (type(target_total_epochs) is not int or target_total_epochs < 1):
        raise ValueError("TARGET_TOTAL_EPOCHS must be None or a positive integer")
    if continuation_lr is not None and (type(continuation_lr) not in (int, float) or
                                        not math.isfinite(continuation_lr) or continuation_lr <= 0):
        raise ValueError("CONTINUATION_LR must be None or a finite positive learning rate")
    if type(max_run_seconds) is not int or max_run_seconds < 1 or type(allow_torch_upgrade) is not bool:
        raise ValueError("MAX_RUN_SECONDS must be positive; ALLOW_TORCH_UPGRADE must be boolean")
    working, mounted = Path(run_dir).expanduser().resolve(), Path(input_root).expanduser().resolve()
    if working == mounted or mounted in working.parents:
        raise ValueError("RUN_DIR must be writable working storage, outside the read-only Input tree")
    manifest_path = working / "continuation_session.json"
    existing = any((working / "checkpoints" / name).exists() for name in CHECKPOINT_NAMES)
    prior_session = read_json(manifest_path) if manifest_path.is_file() else None
    if existing and prior_session is None:
        raise ValueError("Working checkpoints lack their continuation manifest; use a fresh RUN_DIR")
    run = parent_root(working) if existing else discover_parent(input_root, parent_run)
    sources, source_hashes, config_type, resolve_data, runtime_name = verified_saved_runtime(
        run, config_type, resolve_data, template_sources, verified_runtime_sources)
    (path, receipt), choices = latest_checkpoint(run)
    info = read_checkpoint_info(path, receipt)
    values, cursor = info["config"], info["cursor"]
    file_values = read_json(run / "train_config.json")
    allowed_configurations = [learning_choices(values)]
    if existing and prior_session.get("preparation_state") == "revising_checkpoints":
        allowed_configurations = prior_session.get("preparation_learning_choices", [
            prior_session["source_learning_choices"], prior_session["target_learning_choices"]])
    if learning_choices(values) not in allowed_configurations or learning_choices(file_values) not in allowed_configurations:
        raise ValueError("Saved configuration differs from the committed checkpoint/declared preparation")
    saved_cfg = config_type(**values)
    saved_cfg.validate()
    data_path = resolve_data(data_dir or saved_cfg.data_dir, input_root)
    identity = data_fingerprint(data_path)
    if (run / "data_report.json").is_file() and read_json(run / "data_report.json").get("data_fingerprint") != identity:
        raise ValueError("Attach the SAME tokenized dataset/tokenizer used by the saved model")
    runtime_record = read_json(run / "continuation_runtime_identity.json") if (run / "continuation_runtime_identity.json").is_file() else {}
    saved_torch = runtime_record.get("torch", original_torch)
    source_current_fingerprint, runtime_migration = resolve_runtime_transition(
        values, identity, source_hashes, receipt["fingerprint"], saved_torch, original_torch, allow_torch_upgrade)

    if existing:
        session = dict(prior_session)
        if session.get("version") != SESSION_VERSION or type(session.get("target_epoch")) is not int:
            raise ValueError("Invalid working continuation session")
        target = session["target_epoch"] if target_total_epochs is None else target_total_epochs
        if target < cursor["epoch"] or (target == cursor["epoch"] and cursor["next_batch"]):
            raise ValueError("TARGET_TOTAL_EPOCHS is behind the saved training cursor")
    else:
        target = cursor["epoch"] + additional_epochs if target_total_epochs is None else target_total_epochs
        parent_session_path = run / "continuation_session.json"
        if target_total_epochs is None and parent_session_path.is_file():
            parent_session = read_json(parent_session_path)
            if (parent_session.get("extensible_training") is True and
                    type(parent_session.get("target_epoch")) is int and
                    parent_session["target_epoch"] > cursor["epoch"]):
                target = parent_session["target_epoch"]  # Finish an interrupted extension first.
        if target <= cursor["epoch"]:
            raise ValueError("TARGET_TOTAL_EPOCHS must exceed the completed epochs in a new session")
        session = {"version": SESSION_VERSION, "parent_run": str(run), "parent_checkpoint": str(path),
                   "parent_sha256": receipt["sha256"], "start_epoch": cursor["epoch"],
                   "start_next_batch": cursor["next_batch"], "start_step": cursor["global_step"],
                   "original_plan_epochs": saved_cfg.epochs}
    plan = {key: info["scheduler"][key] for key in ("total_steps", "warmup_steps", "peak_lr", "minimum_ratio")}
    if (type(plan["total_steps"]) is not int or plan["total_steps"] < 1 or
            type(plan["warmup_steps"]) is not int or not 0 <= plan["warmup_steps"] < plan["total_steps"] or
            not math.isfinite(plan["peak_lr"]) or plan["peak_lr"] <= 0 or not 0 < plan["minimum_ratio"] <= 1):
        raise ValueError("Saved learning-rate schedule is invalid")
    cfg = replace(saved_cfg, data_dir=str(data_path), run_dir=str(working), resume_dir=None,
                  epochs=max(saved_cfg.epochs, target), max_run_seconds=max_run_seconds)
    if continuation_lr is not None:
        cfg = replace(cfg, learning_rate=float(continuation_lr), warmup_updates=0, min_lr_ratio=1.0)
        plan.update(warmup_steps=0, peak_lr=float(continuation_lr), minimum_ratio=1.0)
    cfg.validate()
    fingerprint = expected_fingerprint(asdict(cfg), identity, source_hashes, str(torch.__version__))
    session.update(target_epoch=target, additional_epochs=additional_epochs, fingerprint=fingerprint,
                   verified_runtime=runtime_name,
                   latest_saved_epoch=cursor["epoch"], latest_saved_step=cursor["global_step"],
                   active_plan_epochs=cfg.epochs, scheduler_plan=plan,
                   learning_rate_policy="saved_schedule_then_final_rate" if continuation_lr is None else "constant_override",
                   continuation_lr=continuation_lr, extensible_training=True,
                   meaning="Train additional epoch boundaries from the saved cursor; no fixed total-epoch cap",
                   preparation_state="revising_checkpoints",
                   source_learning_choices=learning_choices(values), target_learning_choices=learning_choices(asdict(cfg)))
    session["preparation_learning_choices"] = allowed_configurations + [learning_choices(asdict(cfg))]
    if runtime_migration is not None:
        session["runtime_migration"] = runtime_migration
    elif runtime_record.get("runtime_migration"):
        session.setdefault("runtime_migration", runtime_record["runtime_migration"])
    code_dir, checkpoint_dir = working / "code", working / "checkpoints"
    package = code_dir / "model_a"
    package.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(exist_ok=True)
    for name, body in sources.items():
        (package / name).write_bytes(body)
    write_json(manifest_path, session)  # Record allowed transition before any config/checkpoint replacement.
    cfg.save(working / "train_config.json")
    selected = choices if existing else [pair for pair in choices if pair[0] == path or pair[0].name == "best.pt"]
    recovery_parent = None
    if existing and prior_session.get("preparation_state") == "revising_checkpoints":
        candidate_parent = Path(prior_session["parent_run"])
        if candidate_parent.resolve() != working and candidate_parent.is_dir():
            recovery_parent = candidate_parent
            if not any(p.name == "best.pt" for p, _ in selected):
                for parent_path, parent_receipt in intact_checkpoints(candidate_parent):
                    if parent_path.name == "best.pt" and learning_choices(read_checkpoint_info(
                            parent_path, parent_receipt)["config"]) in allowed_configurations:
                        selected = selected + [(parent_path, parent_receipt)]
    needed = max(r["bytes"] for _, r in selected) if existing else sum(r["bytes"] for _, r in selected)
    if shutil.disk_usage(working).free < needed + cfg.min_free_bytes:
        raise OSError("Insufficient working disk space for continued checkpoint preparation")
    for candidate, candidate_receipt in selected:
        candidate_info = read_checkpoint_info(candidate, candidate_receipt)
        if learning_choices(candidate_info["config"]) not in allowed_configurations + [learning_choices(asdict(cfg))]:
            raise ValueError("Recovery/best checkpoint settings differ from the verified continuation")
        _, candidate_upgrade = resolve_runtime_transition(
            candidate_info["config"], identity, source_hashes, candidate_receipt["fingerprint"],
            saved_torch, original_torch, allow_torch_upgrade)
        if candidate_receipt["fingerprint"] == fingerprint:
            if not existing:
                copy_committed_checkpoint(candidate, candidate_receipt, checkpoint_dir)
            continue
        revision = {"kind": "training_plan_extension", "source_fingerprint": candidate_receipt["fingerprint"],
                    "target_fingerprint": fingerprint, "source_plan_epochs": candidate_info["config"]["epochs"],
                    "target_plan_epochs": cfg.epochs, "scheduler_policy": session["learning_rate_policy"],
                    "global_cursor_optimizer_scaler_rank_rng": "retained", "runtime_migration": candidate_upgrade}
        migrate_committed_checkpoint(candidate, candidate_receipt, checkpoint_dir, revision,
                                     candidate_info["config"], asdict(cfg),
                                     plan, continuation_lr)
    if not existing or recovery_parent is not None:
        reports_parent = recovery_parent or run
        for name in ("training_steps.jsonl", "data_report.json"):
            if (reports_parent / name).is_file() and not (working / name).exists():
                shutil.copyfile(reports_parent / name, working / name)
    session["preparation_state"] = "ready"
    write_json(manifest_path, session)
    write_json(working / "continuation_runtime_identity.json",
               {"torch": str(torch.__version__), "sources": source_hashes, "fingerprint": fingerprint,
                "verified_runtime": runtime_name,
                "runtime_migration": session.get("runtime_migration")})
    print(json.dumps({"verified_runtime": runtime_name, "completed_epochs": cursor["epoch"], "saved_next_batch": cursor["next_batch"],
                      "saved_step": cursor["global_step"], "train_through_epoch": target,
                      "remaining_epoch_boundaries": target - cursor["epoch"], "active_plan_epochs": cfg.epochs,
                      "scheduler_global_steps_retained": info["scheduler"]["steps"],
                      "learning_rate_policy": session["learning_rate_policy"],
                      "final_learning_rate": plan["peak_lr"] * plan["minimum_ratio"], "output": str(working)}, indent=2), flush=True)
    return cfg, session
'''
RESUME_SESSION_SOURCE = r'''"""Limit one Kaggle session while leaving the original checkpoint/schedule intact."""
from dataclasses import replace
from pathlib import Path
import argparse
import json
import sys

# This wrapper lives OUTSIDE model_a/. Original source fingerprints stay unchanged.
from model_a import train
from model_a.state import atomic_json

ORIGINAL_WORKER = train.worker
ORIGINAL_EPOCH_LOOP = train.run_training_epochs
ORIGINAL_SCHEDULER = train.WarmupCosineScheduler
WORKER_SCHEDULER_PLAN = None


def continuation_scheduler(optimizer, total_steps, warmup_steps, peak_lr, minimum_ratio):
    """Preserve the saved LR horizon while allowing unlimited cumulative epochs."""
    plan = WORKER_SCHEDULER_PLAN
    if plan is None:
        return ORIGINAL_SCHEDULER(optimizer, total_steps, warmup_steps, peak_lr, minimum_ratio)
    return ORIGINAL_SCHEDULER(optimizer, plan["total_steps"], plan["warmup_steps"],
                              plan["peak_lr"], plan["minimum_ratio"])


def limited_epoch_loop(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                       cursor, cfg, device, rank, stop_event, started, log_path):
    session = json.loads((Path(cfg.run_dir) / "continuation_session.json").read_text(encoding="utf-8"))
    target = session["target_epoch"]
    if not cursor["epoch"] <= target <= cfg.epochs:
        raise ValueError("Continuation target is inconsistent with the restored cursor")
    # The worker restores the declared saved scheduler horizon and the active plan.
    # A loop-only configuration changes this session's stopping boundary.
    loop_cfg = replace(cfg, epochs=target)
    ORIGINAL_EPOCH_LOOP(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                        cursor, loop_cfg, device, rank, stop_event, started, log_path)
    if rank == 0:
        summary_path = Path(cfg.run_dir) / "training_summary.json"
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        summary.update(session_target_epoch=target, original_plan_epochs=cfg.epochs,
                       original_plan_complete=cursor["epoch"] == cfg.epochs,
                       session_complete=cursor["epoch"] == target and cursor["next_batch"] == 0)
        summary.update(completed_epochs=cursor["epoch"], training_plan_epochs=cfg.epochs,
                       learning_rate_policy=session.get("learning_rate_policy", "saved_schedule"))
        if summary["session_complete"] and not summary["original_plan_complete"]:
            summary["status"] = "session_complete_original_plan_remaining"
        atomic_json(summary_path, summary)
        print(f"Continuation saved: completed epochs={cursor['epoch']}/{cfg.epochs}; "
              f"next_batch={cursor['next_batch']}; output={cfg.run_dir}", flush=True)


def continuation_worker(rank, cfg, contract, fingerprint, port, stop_event):
    global WORKER_SCHEDULER_PLAN
    # Each spawned process installs its own hook; child imports never depend on Jupyter.
    train.run_training_epochs = limited_epoch_loop
    session = json.loads((Path(cfg.run_dir) / "continuation_session.json").read_text(encoding="utf-8"))
    WORKER_SCHEDULER_PLAN = session.get("scheduler_plan")
    train.WarmupCosineScheduler = continuation_scheduler
    ORIGINAL_WORKER(rank, cfg, contract, fingerprint, port, stop_event)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    cfg = train.TrainConfig.load(args.config)
    if not (Path(cfg.run_dir) / "continuation_session.json").is_file():
        raise FileNotFoundError("Run the continuation setup cell before training")
    checkpoint_dir = Path(cfg.run_dir) / "checkpoints"
    if not any((checkpoint_dir / name).is_file() and
               (checkpoint_dir / (name + ".json")).is_file()
               for name in ("slot_0.pt", "slot_1.pt", "best.pt")):
        raise FileNotFoundError("Continuation requires a committed checkpoint; run setup again")
    train.worker = continuation_worker
    # Reuse original input validation, lock, fingerprint, NCCL spawning and signals.
    sys.argv = [sys.argv[0], "--config", args.config]
    train.main()


if __name__ == "__main__":
    main()
'''
helper = types.ModuleType("continuation_support")
sys.modules[helper.__name__] = helper
exec(compile(CONTINUATION_SUPPORT_SOURCE, "continuation_support.py", "exec"), helper.__dict__)
original_runtime_sources = {name: "".join(parts) for name, parts in module_parts.items()}
original_runtime_sources["__init__.py"] = '"""From-scratch causal legal LM and resumable DDP runtime."""\n'
VERIFIED_RUNTIME_SOURCES = {'epoch12_fast': {'config.py': 'from dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\nimport json\n\nRUNTIME_VERSION = "cs781-model-a-v1"\nSPECIAL_TOKENS = ("<pad>", "<s>", "</s>", "<unk>", "<sep>", "<mask>")\nSOURCES = ("supreme_court", "high_courts", "acts", "tribunals")\n\n\n@dataclass(frozen=True)\nclass TrainConfig:\n    """Immutable architecture, causal objective, and training/recovery choices."""\n\n    data_dir: str = "/kaggle/input/cs781-preprocessed-data"\n    run_dir: str = "/kaggle/working/model_a_run"\n    resume_dir: str | None = None\n    vocab_size: int = 32_768\n    d_model: int = 768\n    n_heads: int = 12\n    n_layers: int = 12\n    d_ff: int = 2_048\n    rms_epsilon: float = 1e-6\n    rope_base: float = 10_000.0\n    max_seq_tokens: int = 2_048\n    train_seq_tokens: int = 1_024\n    max_prompt_tokens: int = 1_536  # Includes BOS; words and BPE tokens differ.\n    max_target_tokens: int = 512   # Includes EOS if the reference fits.\n    dropout: float = 0.1\n    activation_checkpointing: bool = True\n    loss_chunk_tokens: int = 128\n    epochs: int = 2\n    micro_batch_size: int = 1  # Per T4; effective global batch = 16 windows.\n    accumulation_steps: int = 8\n    world_size: int = 2\n    num_workers: int = 0\n    learning_rate: float = 3e-4\n    min_lr_ratio: float = 0.1\n    warmup_updates: int = 500\n    weight_decay: float = 0.1\n    beta1: float = 0.9\n    beta2: float = 0.95\n    adam_epsilon: float = 1e-8\n    max_grad_norm: float = 1.0\n    checkpoint_every_steps: int = 500\n    max_consecutive_overflows: int = 8\n    max_run_seconds: int = 37_800\n    min_free_bytes: int = 1 << 30\n    max_checkpoint_bytes: int = 6 << 30\n    seed: int = 781\n    deterministic: bool = False\n    generation_examples_per_source: int = 2\n    generation_token_limit: int = 512\n    experiment_kind: str = "cs781-fast-stage-v1"\n    initial_checkpoint: str | None = None\n    initial_checkpoint_sha256: str | None = None\n    parent_data_fingerprint: str | None = None\n    optimization_version: str = "full-corpus-fast-v1"\n    fused_adamw: bool = True\n    loader_prefetch_factor: int = 2\n\n    @property\n    def block_tokens(self) -> int:\n        """Window stride: reuse one context token so document transitions survive."""\n        return self.train_seq_tokens\n\n    def validate(self) -> None:\n        assert self.vocab_size == 32_768\n        assert self.d_model > 0 and self.n_heads > 0 and self.d_model % self.n_heads == 0\n        assert (self.d_model // self.n_heads) % 2 == 0, "RoPE needs even head width"\n        assert self.n_layers > 0 and self.d_ff > 0 and self.rms_epsilon > 0 and self.rope_base > 1\n        assert 0 <= self.dropout < 1\n        assert 1 <= self.train_seq_tokens <= self.max_seq_tokens\n        assert self.max_prompt_tokens >= 2 and self.max_target_tokens >= 1\n        assert self.max_prompt_tokens + self.max_target_tokens - 1 <= self.max_seq_tokens\n        assert 1 <= self.loss_chunk_tokens <= self.max_seq_tokens\n        assert self.epochs > 0 and self.micro_batch_size > 0 and self.accumulation_steps > 0\n        assert self.loader_prefetch_factor > 0 and self.optimization_version == "full-corpus-fast-v1"\n        assert isinstance(self.fused_adamw, bool)\n        assert self.world_size == 2 and self.num_workers >= 0\n        assert self.checkpoint_every_steps == 500\n        assert self.learning_rate > 0 and 0 < self.min_lr_ratio <= 1\n        assert self.warmup_updates >= 0 and self.max_grad_norm > 0\n        assert 0 <= self.weight_decay and 0 < self.beta1 < 1 and 0 < self.beta2 < 1\n        assert self.adam_epsilon > 0 and self.max_consecutive_overflows > 0\n        assert self.max_run_seconds > 0 and self.min_free_bytes >= 0 and self.max_checkpoint_bytes > 0\n        assert self.generation_examples_per_source > 0\n        assert 1 <= self.generation_token_limit <= self.max_seq_tokens - self.max_prompt_tokens + 1\n\n    @classmethod\n    def load(cls, path: str | Path) -> "TrainConfig":\n        return cls(**json.loads(Path(path).read_text(encoding="utf-8")))\n\n    def save(self, path: str | Path) -> None:\n        Path(path).write_text(json.dumps(asdict(self), indent=2) + "\\n", encoding="utf-8")\n\n\ndef unique_trainable_parameters(model) -> int:\n    """Count each shared Parameter once; the tied LM head adds no new weights."""\n    return sum(p.numel() for p in {id(p): p for p in model.parameters() if p.requires_grad}.values())\n\n\ndef assert_parameter_boundary(model) -> int:\n    count = unique_trainable_parameters(model)\n    assert 50_000_000 < count < 300_000_000, f"Parameter boundary violated: {count:,}"\n    return count\n\nREQUIRED_DATA_FILES = (\n    "metadata.json", "tokenizer.json", "train.bin", "val.bin",\n    "train_documents.jsonl", "validation_documents.jsonl", "validation_pairs.jsonl",\n)\n\n\ndef complete_preprocessing_directory(directory: Path) -> bool:\n    """Check names and receipt sizes without loading binaries or copying input data.\n\n    ZIP exports may contain a second, incomplete downloaded_output directory.\n    Empty binaries and incomplete exports must never win automatic discovery.\n    Full SHA-256 verification still runs in the existing data-contract check.\n    """\n    try:\n        if not all((directory / name).is_file() for name in REQUIRED_DATA_FILES):\n            return False\n        metadata_path = directory / "metadata.json"\n        if not 0 < metadata_path.stat().st_size <= (4 << 20):\n            return False\n        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if not isinstance(metadata, dict) or metadata.get("status") != "verified":\n            return False\n        if metadata.get("schema_version") != 1 or metadata.get("dtype") != "<u2":\n            return False\n        if metadata.get("vocab_size") != 32_768:\n            return False\n        for name, key in (("train.bin", "train_tokens"), ("val.bin", "val_tokens")):\n            count = metadata[key]\n            if type(count) is not int or count <= 0 or (directory / name).stat().st_size != count * 2:\n                return False\n        for name in REQUIRED_DATA_FILES[1:]:\n            size = (directory / name).stat().st_size\n            if size <= 0 or size != metadata["artifacts"][name]["bytes"]:\n                return False\n        return True\n    except (OSError, ValueError, KeyError, TypeError):\n        return False\n\n\ndef resolve_preprocessed_data_dir(requested: str | Path,\n                                  input_root: str | Path = "/kaggle/input") -> Path:\n    """Prefer a complete explicit folder, otherwise locate one unique mounted export.\n\n    Multiple complete exports require an explicit choice; files are never copied\n    from read-only Kaggle input into scarce working storage.\n    """\n    requested_path = Path(requested).expanduser()\n    if complete_preprocessing_directory(requested_path):\n        return requested_path.resolve()\n    roots = [requested_path] if requested_path.is_dir() else []\n    mounted_root = Path(input_root)\n    if mounted_root.is_dir() and mounted_root not in roots:\n        roots.append(mounted_root)\n    candidates: set[Path] = set()\n    inspected: set[Path] = set()\n    for root in roots:\n        for metadata_path in root.rglob("metadata.json"):\n            directory = metadata_path.parent.resolve()\n            inspected.add(directory)\n            if complete_preprocessing_directory(directory):\n                candidates.add(directory)\n    if len(candidates) == 1:\n        return next(iter(candidates))\n    if len(candidates) > 1:\n        options = "\\n".join(f"  {path}" for path in sorted(candidates))\n        raise ValueError("Multiple complete preprocessing exports found. Set cfg.data_dir "\n                         f"to one exact folder:\\n{options}")\n    found = "\\n".join(f"  {path}" for path in sorted(inspected)[:8]) or "  none"\n    raise FileNotFoundError(\n        f"No complete preprocessing export at {requested_path} or below {mounted_root}. "\n        "Attach the tokenized dataset and select the folder containing metadata.json, "\n        "tokenizer.json, both nonempty .bin files, and the three JSONL manifests. "\n        f"Inspected metadata folders (incomplete or invalid):\\n{found}"\n    )\n\n\ndef resolve_training_config(cfg: TrainConfig,\n                            input_root: str | Path = "/kaggle/input") -> TrainConfig:\n    """Canonicalize the input path before saving config or spawning GPU workers."""\n    resolved = resolve_preprocessed_data_dir(cfg.data_dir, input_root)\n    print(f"Preprocessed data: {resolved}", flush=True)\n    return replace(cfg, data_dir=str(resolved))\n\nimport ast\nimport hashlib\n\nAUTO_STAGE = "cs781-fast-stage-v1"\n\n\ndef auto_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        while block := stream.read(8 << 20):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef auto_saved_runs(input_root: Path, family: str) -> list[Path]:\n    """Discover output folders; notebook execution logs never count as weights."""\n    if not input_root.is_dir():\n        return []\n    return sorted({path.parent.resolve() for path in input_root.rglob("train_config.json")\n                   if (path.parent / "code" / f"model_{family}" / "model.py").is_file()\n                   and any((path.parent / "checkpoints" / name).is_file()\n                           for name in ("best.pt", "slot_0.pt", "slot_1.pt"))})\n\n\ndef auto_parent_checkpoint(run: Path) -> tuple[Path, dict]:\n    """Prefer best validation weights; require the selected checkpoint\'s receipt."""\n    choices = []\n    for name in ("best.pt", "slot_0.pt", "slot_1.pt"):\n        path = run / "checkpoints" / name\n        if not path.is_file():\n            continue\n        receipt_path = path.with_name(path.name + ".json")\n        if not receipt_path.is_file():\n            raise FileNotFoundError(f"Checkpoint receipt missing: {receipt_path}")\n        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n        if type(receipt.get("step")) is not int or receipt["step"] <= 0:\n            raise ValueError(f"No successful saved progress in {path}")\n        choices.append((path, receipt))\n    if not choices:\n        raise FileNotFoundError(f"No committed checkpoint in {run}/checkpoints")\n    selected = next((item for item in choices if item[0].name == "best.pt"), None)\n    path, receipt = selected or max(choices, key=lambda item: (item[1]["step"], item[1]["generation"]))\n    if path.stat().st_size != receipt["bytes"] or auto_sha256(path) != receipt["sha256"]:\n        raise ValueError(f"Selected parent checkpoint failed SHA-256/size verification: {path}")\n    return path, receipt\n\n\ndef auto_data_fingerprint(data_dir: Path) -> str:\n    """Derive the original identity; full artifact SHA verification follows in train.py."""\n    metadata = json.loads((data_dir / "metadata.json").read_text(encoding="utf-8"))\n    names = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",\n             "validation_documents.jsonl", "validation_pairs.jsonl")\n    identity = {name: metadata["artifacts"][name]["sha256"] for name in names}\n    identity["metadata.json"] = auto_sha256(data_dir / "metadata.json")\n    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n\n\ndef auto_configuration(family: str, model_ast_sha256: str, data_dir: str,\n                       run_dir: str, parent_run: str | None = None,\n                       input_root: str = "/kaggle/input") -> tuple[TrainConfig, dict]:\n    """Warm-start old weights once; resume this stage without changing its schedule.\n\n    A new continuation resets Adam, scaler, scheduler, epoch/step, and rank RNGs.\n    An interrupted auto stage instead restores all of them through CheckpointManager.\n    """\n    data_path = resolve_preprocessed_data_dir(data_dir, input_root)\n    working = Path(run_dir)\n    local_config = working / "train_config.json"\n    if local_config.is_file() and any((working / "checkpoints" / name).is_file()\n                                      for name in ("best.pt", "slot_0.pt", "slot_1.pt")):\n        saved = json.loads(local_config.read_text(encoding="utf-8"))\n        if saved.get("experiment_kind") != AUTO_STAGE:\n            raise ValueError("The working folder belongs to another experiment; choose a fresh output folder")\n        cfg = replace(TrainConfig(**saved), data_dir=str(data_path), run_dir=str(working), resume_dir=None)\n        return cfg, {"mode": "resume_working_stage", "saved_config": str(local_config)}\n    runs = [Path(parent_run).resolve()] if parent_run else auto_saved_runs(Path(input_root), family)\n    if parent_run and not (runs[0] / "train_config.json").is_file():\n        raise FileNotFoundError(f"Select the run root, containing train_config.json: {runs[0]}")\n    stage_runs = [run for run in runs if\n                  json.loads((run / "train_config.json").read_text(encoding="utf-8")).get("experiment_kind") == AUTO_STAGE]\n    old_auto_runs = [run for run in runs if json.loads((run / "train_config.json").read_text(encoding="utf-8")).get("experiment_kind") == "cs781-auto-stage-v1"]\n    if not stage_runs and len(old_auto_runs) == 1:\n        runs = old_auto_runs\n    if len(stage_runs) > 1 or (not stage_runs and len(runs) > 1):\n        raise ValueError(f"Several Model {family.upper()} runs match; set PARENT_RUN_DIR to one exact run root: {runs}")\n    if stage_runs:\n        run = stage_runs[0]\n        saved = json.loads((run / "train_config.json").read_text(encoding="utf-8"))\n        cfg = replace(TrainConfig(**saved), data_dir=str(data_path), run_dir=str(working),\n                      resume_dir=str(run / "checkpoints"))\n        return cfg, {"mode": "resume_attached_stage", "saved_run": str(run)}\n    base = TrainConfig()\n    # Prefer the latest auto stage over an original run if both are attached.\n    auto_runs = [run for run in runs if json.loads((run / "train_config.json").read_text(encoding="utf-8")).get("experiment_kind") == "cs781-auto-stage-v1"]\n    if not stage_runs and len(auto_runs) == 1:\n        runs = auto_runs\n    parent_path, parent_receipt, parent_identity = None, None, None\n    if runs:\n        run = runs[0]\n        previous = json.loads((run / "train_config.json").read_text(encoding="utf-8"))\n        base = TrainConfig(**previous)\n        parent_source = (run / "code" / f"model_{family}" / "model.py").read_text(encoding="utf-8")\n        parent_model_digest = hashlib.sha256(ast.dump(ast.parse(parent_source)).encode()).hexdigest()\n        if parent_model_digest not in ({model_ast_sha256} if isinstance(model_ast_sha256, str) else set(model_ast_sha256)):\n            raise ValueError("Parent architecture code differs; this adapter supports your original prepared A/B/C/D implementations")\n        data_report = run / "data_report.json"\n        if not data_report.is_file():\n            data_report = run / "indices" / "report.json"\n        if not data_report.is_file():\n            raise FileNotFoundError("Attach the full saved run, including data_report.json; tokenizer/data compatibility must be proved")\n        parent_identity = json.loads(data_report.read_text(encoding="utf-8")).get("data_fingerprint")\n        if not parent_identity or parent_identity != auto_data_fingerprint(data_path):\n            raise ValueError("Parent tokenizer/data export differs from the attached preprocessing dataset")\n        parent_path, parent_receipt = auto_parent_checkpoint(run)\n    settings = dict(data_dir=str(data_path), run_dir=str(working), resume_dir=None,\n                    experiment_kind=AUTO_STAGE, initial_checkpoint=str(parent_path) if parent_path else None,\n                    initial_checkpoint_sha256=parent_receipt["sha256"] if parent_receipt else None,\n                    parent_data_fingerprint=parent_identity,\n                    epochs=2, learning_rate=1e-4 if runs else 3e-4,\n                    warmup_updates=250 if runs else 500, micro_batch_size=1,\n                    accumulation_steps=16 if family == "b" else 8,\n                    activation_checkpointing=True, num_workers=1,\n                    optimization_version="full-corpus-fast-v1", fused_adamw=True, loader_prefetch_factor=2,\n                    loss_chunk_tokens=min(512, base.max_decoder_tokens if family == "b" else base.max_seq_tokens),\n                    generation_token_limit=min(512, base.max_decoder_tokens) if family == "b" else\n                        min(512, base.max_seq_tokens - base.max_prompt_tokens + 1))\n    if family == "b":\n        settings.update(train_prefix_tokens=min(1536, base.max_source_tokens - 1),\n                        train_target_tokens=min(512, base.max_decoder_tokens))\n    else:\n        settings.update(train_seq_tokens=min(1536, base.max_seq_tokens))\n    cfg = replace(base, **settings)\n    cfg.validate()\n    report = {"mode": "continue_parent_weights" if runs else "fresh_random_initialization",\n              "parent_run": str(runs[0]) if runs else None,\n              "parent_checkpoint": str(parent_path) if parent_path else None,\n              "parent_sha256": parent_receipt["sha256"] if parent_receipt else None,\n              "parent_epoch": parent_receipt["epoch"] if parent_receipt else None,\n              "parent_step": parent_receipt["step"] if parent_receipt else None,\n              "stage_epochs": cfg.epochs, "stage_learning_rate": cfg.learning_rate,\n              "optimizer_schedule": "New Adam/scaler/cosine schedule; exact full-state resume after first stage checkpoint"}\n    return cfg, report\n', 'model.py': 'from dataclasses import dataclass\nimport math\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom .config import TrainConfig, assert_parameter_boundary\n\nKeyValue = tuple[Tensor, Tensor]\n\n\nclass RMSNorm(nn.Module):\n    """RMSNorm: hidden[B,T,D] -> normalized[B,T,D]; reduce variance in FP32."""\n\n    def __init__(self, width: int, epsilon: float) -> None:\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(width))\n        self.epsilon = epsilon\n\n    def forward(self, hidden: Tensor) -> Tensor:\n        normalized = hidden.float() * torch.rsqrt(hidden.float().square().mean(-1, keepdim=True) + self.epsilon)\n        return (normalized * self.weight.float()).to(hidden.dtype)\n\n\nclass RotaryEmbedding(nn.Module):\n    """Rotate adjacent Q/K pairs at absolute positions; no learned position table.\n\n    q,k[B,H,T,Dh], positions[T] -> rotated q,k of the same shape.\n    Frequencies/trigonometry stay FP32 even under FP16 autocast.\n    """\n\n    def __init__(self, head_dim: int, base: float, max_positions: int = 2048) -> None:\n        super().__init__()\n        assert head_dim % 2 == 0\n        self.head_dim, self.base, self.max_positions = head_dim, base, max_positions\n        frequencies = base ** (-torch.arange(0, head_dim, 2, dtype=torch.float32, device="cpu") / head_dim)\n        angles = torch.arange(max_positions, dtype=torch.float32, device="cpu")[:, None] * frequencies[None]\n        self.register_buffer("cosine_cache", angles.cos(), persistent=False)\n        self.register_buffer("sine_cache", angles.sin(), persistent=False)\n\n    def forward(self, query: Tensor, key: Tensor, positions: Tensor) -> KeyValue:\n        """Rotate [B,H,T,Dh] using cached FP32 trigonometry at true positions[T]."""\n        with torch.autocast(device_type=query.device.type, enabled=False):\n            cos_cache = self.cosine_cache.to(positions.device)\n            sin_cache = self.sine_cache.to(positions.device)\n            cosine = cos_cache.index_select(0, positions)[None, None]\n            sine = sin_cache.index_select(0, positions)[None, None]\n            def rotate(value: Tensor) -> Tensor:\n                even, odd = value.float()[..., 0::2], value.float()[..., 1::2]\n                return torch.stack((even * cosine - odd * sine, even * sine + odd * cosine),\n                                   dim=-1).flatten(-2).to(value.dtype)\n            return rotate(query), rotate(key)\n\n\nclass CausalSelfAttention(nn.Module):\n    """Bias-free multi-head attention: hidden[B,T,D] -> hidden[B,T,D].\n\n    SDPA selects an available kernel on T4; no FlashAttention-only dependency.\n    Boolean allowed[B,1,T,T] uses True for legal keys. Cached keys are rotated.\n    """\n\n    def __init__(self, cfg: TrainConfig) -> None:\n        super().__init__()\n        self.heads, self.head_dim, self.dropout = cfg.n_heads, cfg.d_model // cfg.n_heads, cfg.dropout\n        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)\n        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)\n        self.rotary = RotaryEmbedding(self.head_dim, cfg.rope_base, cfg.max_seq_tokens)\n\n    def project(self, hidden: Tensor, positions: Tensor) -> tuple[Tensor, Tensor, Tensor]:\n        batch, length, _ = hidden.shape\n        query, key, value = self.qkv(hidden).view(batch, length, 3, self.heads, self.head_dim).unbind(2)\n        query, key, value = (tensor.transpose(1, 2) for tensor in (query, key, value))\n        query, key = self.rotary(query, key, positions)\n        return query, key, value\n\n    def attend(self, query: Tensor, kv: KeyValue, allowed: Tensor | None,\n               causal: bool) -> Tensor:\n        output = F.scaled_dot_product_attention(\n            query, kv[0], kv[1], attn_mask=allowed, is_causal=causal,\n            dropout_p=self.dropout if self.training else 0.0)\n        return self.out(output.transpose(1, 2).contiguous().flatten(2))\n\n    def forward(self, hidden: Tensor, allowed: Tensor | None) -> Tensor:\n        positions = torch.arange(hidden.shape[1], device=hidden.device)\n        query, key, value = self.project(hidden, positions)\n        return self.attend(query, (key, value), allowed, causal=allowed is None)\n\n\nclass SwiGLU(nn.Module):\n    """Gated FFN: hidden[B,T,D] -> Wdown(SiLU(Wgate h) * Wup h)."""\n\n    def __init__(self, cfg: TrainConfig) -> None:\n        super().__init__()\n        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)\n        self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)\n        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)\n\n    def forward(self, hidden: Tensor) -> Tensor:\n        return self.down(F.silu(self.gate(hidden)) * self.up(hidden))\n\n\nclass DecoderBlock(nn.Module):\n    """RMSNorm -> rotary attention -> residual -> RMSNorm -> SwiGLU -> residual."""\n\n    def __init__(self, cfg: TrainConfig) -> None:\n        super().__init__()\n        self.attention_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)\n        self.ffn_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)\n        self.attention = CausalSelfAttention(cfg)\n        self.ffn = SwiGLU(cfg)\n        self.dropout = nn.Dropout(cfg.dropout)\n\n    def forward(self, hidden: Tensor, allowed: Tensor | None) -> Tensor:\n        hidden = hidden + self.dropout(self.attention(self.attention_norm(hidden), allowed))\n        return hidden + self.dropout(self.ffn(self.ffn_norm(hidden)))\n\n    def cached(self, hidden: Tensor, positions: Tensor, previous: KeyValue | None) -> tuple[Tensor, KeyValue]:\n        """Prefill an unpadded prompt or append one token; return updated rotary K/V."""\n        query, key, value = self.attention.project(self.attention_norm(hidden), positions)\n        kv = (key, value) if previous is None else (\n            torch.cat((previous[0], key), dim=2), torch.cat((previous[1], value), dim=2))\n        # Prefill has square causal attention. A single new query may see every cached key.\n        hidden = hidden + self.dropout(self.attention.attend(query, kv, None, causal=previous is None))\n        return hidden + self.dropout(self.ffn(self.ffn_norm(hidden))), kv\n\n\n@dataclass\nclass DecoderCache:\n    """Inference-only K/V per layer; absolute position counts consumed input tokens."""\n    key_values: list[KeyValue]\n    position: int\n\n\nclass LegalCausalDecoder(nn.Module):\n    """Randomly initialized LLaMA-style legal LM; shared input/output vocabulary.\n\n    forward(input_ids[B,T], attention_mask[B,T] | None, labels[B,T] | None)\n        -> logits[B,T,V] if labels is None, otherwise summed FP32 scalar NLL.\n    Labels are already shifted by the dataset; -100 means no supervision.\n    """\n\n    def __init__(self, cfg: TrainConfig) -> None:\n        super().__init__()\n        self.cfg = cfg\n        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)\n        self.layers = nn.ModuleList(DecoderBlock(cfg) for _ in range(cfg.n_layers))\n        self.final_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)\n        self.apply(self.initialize_weights)\n        for layer in self.layers:\n            nn.init.normal_(layer.attention.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))\n            nn.init.normal_(layer.ffn.down.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))\n        self.parameter_count = assert_parameter_boundary(self)\n\n    @staticmethod\n    def initialize_weights(module: nn.Module) -> None:\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n\n    def hidden_states(self, input_ids: Tensor, attention_mask: Tensor | None = None) -> Tensor:\n        assert input_ids.ndim == 2 and 0 < input_ids.shape[1] <= self.cfg.max_seq_tokens\n        allowed = None\n        if attention_mask is not None:\n            assert attention_mask.shape == input_ids.shape and attention_mask.dtype == torch.bool\n            assert bool(attention_mask[:, 0].all()), "Right padding requires a valid first token"\n            if not bool(attention_mask.all()):\n                causal = torch.ones((input_ids.shape[1], input_ids.shape[1]),\n                                    device=input_ids.device, dtype=torch.bool).tril()\n                allowed = causal[None, None] & attention_mask[:, None, None, :]\n        hidden = self.token_embedding(input_ids)\n        for layer in self.layers:\n            if self.training and self.cfg.activation_checkpointing:\n                hidden = checkpoint(layer, hidden, allowed, use_reentrant=False)\n            else:\n                hidden = layer(hidden, allowed)\n        return self.final_norm(hidden)\n\n    def chunk_loss(self, hidden: Tensor, labels: Tensor) -> Tensor:\n        logits = F.linear(hidden, self.token_embedding.weight)\n        return F.cross_entropy(logits.float(), labels, ignore_index=-100, reduction="sum")\n\n    def forward(self, input_ids: Tensor, attention_mask: Tensor | None = None,\n                labels: Tensor | None = None, label_indices: Tensor | None = None) -> Tensor:\n        """IDs[B,T], right-padding mask, labels -> logits or exact summed target NLL."""\n        hidden = self.hidden_states(input_ids, attention_mask)\n        if labels is None:\n            return F.linear(hidden, self.token_embedding.weight)\n        assert labels.shape == input_ids.shape\n        return exact_vocabulary_loss(hidden, labels, self.token_embedding.weight,\n                                     self.cfg.loss_chunk_tokens, self.training, label_indices)\n\n\n    @torch.inference_mode()\n    def prefill(self, input_ids: Tensor) -> tuple[Tensor, DecoderCache]:\n        """Unpadded prompt[B,T] -> next-token logits[B,V], cache with T consumed tokens."""\n        assert not self.training and 0 < input_ids.shape[1] <= self.cfg.max_seq_tokens\n        hidden = self.token_embedding(input_ids)\n        positions = torch.arange(input_ids.shape[1], device=input_ids.device)\n        key_values = []\n        for layer in self.layers:\n            hidden, kv = layer.cached(hidden, positions, None)\n            key_values.append(kv)\n        logits = F.linear(self.final_norm(hidden[:, -1]), self.token_embedding.weight)\n        return logits, DecoderCache(key_values, input_ids.shape[1])\n\n\n@torch.inference_mode()\ndef decode_one_token(model: LegalCausalDecoder, token_ids: Tensor, cache: DecoderCache) -> Tensor:\n    """Append token_ids[B,1] at its true RoPE position -> next-token logits[B,V]."""\n    assert not model.training and token_ids.shape[1] == 1\n    assert cache.position < model.cfg.max_seq_tokens\n    hidden = model.token_embedding(token_ids)\n    positions = torch.tensor([cache.position], device=token_ids.device)\n    for index, layer in enumerate(model.layers):\n        hidden, cache.key_values[index] = layer.cached(hidden, positions, cache.key_values[index])\n    cache.position += 1\n    return F.linear(model.final_norm(hidden[:, 0]), model.token_embedding.weight)\n\n# CELL: Exact supervised-token vocabulary projection in bounded chunks\ndef exact_vocabulary_loss(hidden: Tensor, labels: Tensor, weight: Tensor,\n                          chunk_tokens: int, training: bool,\n                          label_indices: Tensor | None = None) -> Tensor:\n    """[B,T,D], shifted labels[B,T] -> summed FP32 NLL with the same objective.\n\n    Collation computes supervised flat indices on CPU, avoiding CUDA nonzero\n    synchronization. Only PAD/prompt positions with label=-100 are omitted.\n    Every real target remains supervised. Recompute bounded logits in backward.\n    """\n    hidden, targets = hidden.flatten(0, 1), labels.flatten()\n    if label_indices is not None:\n        hidden = hidden.index_select(0, label_indices)\n        targets = targets.index_select(0, label_indices)\n    if targets.numel() == 0:\n        # A final distributed sentinel contributes zero but visits every layer.\n        return hidden.sum() * 0.0\n\n    def project_loss(values: Tensor, target: Tensor, embedding: Tensor) -> Tensor:\n        logits = F.linear(values, embedding)\n        return F.cross_entropy(logits.float(), target, ignore_index=-100, reduction="sum")\n\n    losses = []\n    for start in range(0, targets.numel(), chunk_tokens):\n        values, target = hidden[start:start + chunk_tokens], targets[start:start + chunk_tokens]\n        if training and torch.is_grad_enabled():\n            losses.append(checkpoint(project_loss, values, target, weight, use_reentrant=False))\n        else:\n            losses.append(project_loss(values, target, weight))\n    return torch.stack(losses).sum()\n', 'data.py': 'from functools import partial\nfrom pathlib import Path\nfrom collections import Counter\nimport hashlib\nimport json\nimport os\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, Sampler\nfrom tokenizers import Tokenizer\nfrom tqdm.auto import tqdm\nfrom .config import TrainConfig, SPECIAL_TOKENS, SOURCES\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        while block := stream.read(8 << 20):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef strict_object(pairs: list[tuple]) -> dict:\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise ValueError(f"Duplicate JSON key: {key}")\n        result[key] = value\n    return result\n\n\ndef read_json(path: Path) -> dict:\n    return json.loads(path.read_text(encoding="utf-8"), object_pairs_hook=strict_object,\n                      parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))\n\n\ndef verify_data_contract(cfg: TrainConfig) -> dict:\n    """Verify bounded-memory binaries, tokenizer identity, and published receipts."""\n    root = Path(cfg.data_dir)\n    metadata = read_json(root / "metadata.json")\n    assert metadata["status"] == "verified" and metadata["schema_version"] == 1\n    assert metadata["dtype"] == "<u2" and metadata["vocab_size"] == cfg.vocab_size\n    assert metadata["train_tokens"] % 100 == 0 and metadata["val_tokens"] % 4 == 0\n    assert metadata["training"]["source_tokens"] == dict(zip(\n        SOURCES, [metadata["train_tokens"] * percent // 100 for percent in (35, 35, 15, 15)]))\n    assert all(metadata["validation"][source]["cached_tokens"] == metadata["val_tokens"] // 4\n               for source in SOURCES)\n    required = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",\n                "validation_documents.jsonl", "validation_pairs.jsonl")\n    for name in tqdm(required, desc="Verify preprocessing receipts"):\n        path, receipt = root / name, metadata["artifacts"][name]\n        assert path.stat().st_size == receipt["bytes"], f"Size mismatch: {name}"\n        assert sha256_file(path) == receipt["sha256"], f"Checksum mismatch: {name}"\n    tokenizer = Tokenizer.from_file(str(root / "tokenizer.json"))\n    ids = {token: tokenizer.token_to_id(token) for token in SPECIAL_TOKENS}\n    assert tokenizer.get_vocab_size() == 32_768\n    assert ids == metadata["special_token_ids"] == dict(zip(SPECIAL_TOKENS, range(6)))\n    for name, count in (("train.bin", metadata["train_tokens"]), ("val.bin", metadata["val_tokens"])):\n        assert count > 0 and (root / name).stat().st_size == count * 2\n        cache = np.memmap(root / name, dtype="<u2", mode="r", shape=(count,))\n        for start in range(0, count, 1_000_000):\n            assert int(cache[start:start + 1_000_000].max()) < cfg.vocab_size\n        del cache\n    identity = {name: metadata["artifacts"][name]["sha256"] for name in required}\n    identity["metadata.json"] = sha256_file(root / "metadata.json")\n    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    return {"fingerprint": fingerprint, "metadata": metadata, "special_ids": ids}\n\n\ndef prepare_indices(cfg: TrainConfig, contract: dict) -> dict:\n    """Index document-local next-token windows; retain every within-document transition.\n\n    Adjacent windows share one input context token. Each target is supervised once\n    per document pass, without predicting across independent document boundaries.\n    """\n    root, run = Path(cfg.data_dir), Path(cfg.run_dir)\n    index_dir = run / "indices"\n    index_dir.mkdir(parents=True, exist_ok=True)\n    offsets, lengths, counts, source_indices = [], [], [], []\n    train_hashes, validation_hashes = set(), set()\n    source_tokens, target_tokens, dropped_tokens = Counter(), Counter(), Counter()\n    cursor = 0\n    with (root / "train_documents.jsonl").open(encoding="utf-8") as stream:\n        for line in stream:\n            row = json.loads(line, object_pairs_hook=strict_object)\n            offset, length, source = row["cache_offset"], row["cache_length"], row["source"]\n            assert source in SOURCES and offset == cursor and length > 0\n            assert row["sha256"] not in train_hashes\n            train_hashes.add(row["sha256"])\n            predictions = length - 1\n            count = (predictions + cfg.train_seq_tokens - 1) // cfg.train_seq_tokens\n            if count:\n                offsets.append(offset); lengths.append(length); counts.append(count)\n                source_indices.append(SOURCES.index(source))\n            source_tokens[source] += length\n            target_tokens[source] += predictions\n            dropped_tokens[source] += int(length == 1)\n            cursor += length\n    assert cursor == contract["metadata"]["train_tokens"]\n    assert dict(source_tokens) == contract["metadata"]["training"]["source_tokens"]\n    with (root / "validation_documents.jsonl").open(encoding="utf-8") as stream:\n        for line in stream:\n            digest = json.loads(line)["sha256"]\n            assert digest not in validation_hashes\n            validation_hashes.add(digest)\n    assert train_hashes.isdisjoint(validation_hashes)\n    cumulative = np.cumsum(np.asarray(counts, dtype=np.int64))\n    assert len(cumulative) and int(cumulative[-1]) >= cfg.world_size * cfg.micro_batch_size\n    train_path = index_dir / "train_index.npz"\n    temporary = train_path.with_suffix(".partial")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, offsets=np.array(offsets, dtype=np.int64),\n                            lengths=np.array(lengths, dtype=np.int64), cumulative=cumulative,\n                            sources=np.array(source_indices, dtype=np.uint8))\n        stream.flush(); os.fsync(stream.fileno())\n    os.replace(temporary, train_path)\n    validation = prepare_validation_index(cfg, validation_hashes, index_dir)\n    report = {"train_examples": int(cumulative[-1]), "cached_source_tokens": dict(source_tokens),\n              "eligible_target_tokens": dict(target_tokens), "one_token_documents_dropped": dict(dropped_tokens),\n              "global_batch_tail_max_examples": 0, "full_window_coverage": True,\n              "validation": validation, "data_fingerprint": contract["fingerprint"]}\n    (index_dir / "report.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n    return report\n\n\ndef tokenize_pair(tokenizer: Tokenizer, prompt: str, reference: str,\n                  cfg: TrainConfig) -> tuple[list[int], list[int], bool, bool]:\n    """Encode prompt and target separately; report BPE crops, preserve the prompt tail."""\n    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids\n    target_ids = tokenizer.encode(" " + reference, add_special_tokens=False).ids\n    prompt_cropped = len(prompt_ids) > cfg.max_prompt_tokens - 1\n    target_cropped = len(target_ids) + 1 > cfg.max_target_tokens\n    prefix = [1] + prompt_ids[-(cfg.max_prompt_tokens - 1):]\n    target = target_ids[:cfg.max_target_tokens] if target_cropped else target_ids + [2]\n    assert prefix and target and len(prefix) + len(target) - 1 <= cfg.max_seq_tokens\n    return prefix, target, prompt_cropped, target_cropped\n\n\ndef prepare_validation_index(cfg: TrainConfig, allowed_hashes: set[str], index_dir: Path) -> dict:\n    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))\n    offsets, identifiers, statistics = [], set(), Counter()\n    with (Path(cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:\n        while True:\n            offset = stream.tell()\n            line = stream.readline()\n            if not line:\n                break\n            row = json.loads(line, object_pairs_hook=strict_object)\n            assert row["id"] not in identifiers and row["document_sha256"] in allowed_hashes\n            assert len(row["prompt"].split()) == 512 and len(row["reference"].split()) == 128\n            assert row["source"] in SOURCES\n            identifiers.add(row["id"]); offsets.append(offset)\n            _, _, prompt_crop, target_crop = tokenize_pair(tokenizer, row["prompt"], row["reference"], cfg)\n            statistics[row["source"]] += 1\n            statistics["prompt_truncations"] += int(prompt_crop)\n            statistics["target_truncations"] += int(target_crop)\n    assert offsets and all(statistics[source] for source in SOURCES)\n    path = index_dir / "validation_offsets.npy"\n    with path.with_suffix(".partial").open("wb") as stream:\n        np.save(stream, np.asarray(offsets, dtype=np.int64), allow_pickle=False)\n        stream.flush(); os.fsync(stream.fileno())\n    os.replace(path.with_suffix(".partial"), path)\n    return dict(statistics, examples=len(offsets))\n\n\nclass ContinuationDataset(Dataset):\n    """One document window -> input_ids[T], shifted labels[T]; all transitions train."""\n\n    def __init__(self, cfg: TrainConfig) -> None:\n        self.cfg, self.cache = cfg, None\n        with np.load(Path(cfg.run_dir) / "indices/train_index.npz", allow_pickle=False) as index:\n            self.offsets, self.lengths = index["offsets"].copy(), index["lengths"].copy()\n            self.cumulative = index["cumulative"].copy()\n\n    def __len__(self) -> int:\n        return int(self.cumulative[-1])\n\n    def __getstate__(self) -> dict:\n        return dict(self.__dict__, cache=None)\n\n    def __getitem__(self, index: int) -> dict:\n        if index == -1:\n            return {"input_ids": [1], "labels": [-100]}\n        assert 0 <= index < len(self)\n        if self.cache is None:\n            self.cache = np.memmap(Path(self.cfg.data_dir) / "train.bin", dtype="<u2", mode="r")\n        document = int(np.searchsorted(self.cumulative, index, side="right"))\n        previous_count = int(self.cumulative[document - 1]) if document else 0\n        within = (index - previous_count) * self.cfg.train_seq_tokens\n        length = min(self.cfg.train_seq_tokens + 1, int(self.lengths[document]) - within)\n        start = int(self.offsets[document]) + within\n        tokens = np.asarray(self.cache[start:start + length], dtype=np.int64)\n        assert 2 <= len(tokens) <= self.cfg.train_seq_tokens + 1\n        return {"input_ids": tokens[:-1], "labels": tokens[1:]}\n\n\nclass ValidationDataset(Dataset):\n    """JSONL prompt/continuation teacher forcing; only reference tokens affect NLL."""\n\n    def __init__(self, cfg: TrainConfig) -> None:\n        self.cfg, self.tokenizer = cfg, None\n        self.offsets = np.load(Path(cfg.run_dir) / "indices/validation_offsets.npy", allow_pickle=False)\n\n    def __len__(self) -> int:\n        return len(self.offsets)\n\n    def __getstate__(self) -> dict:\n        return dict(self.__dict__, tokenizer=None)\n\n    def __getitem__(self, index: int) -> dict:\n        if self.tokenizer is None:\n            self.tokenizer = Tokenizer.from_file(str(Path(self.cfg.data_dir) / "tokenizer.json"))\n        with (Path(self.cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:\n            stream.seek(int(self.offsets[index]))\n            row = json.loads(stream.readline())\n        prefix, target, _, _ = tokenize_pair(self.tokenizer, row["prompt"], row["reference"], self.cfg)\n        return {"input_ids": (prefix + target)[:-1],\n                "labels": [-100] * (len(prefix) - 1) + target}\n\n\ndef collate_continuations(rows: list[dict]) -> dict[str, torch.Tensor]:\n    """Pad input IDs with PAD=0; ignored labels=-100; valid keys=True."""\n    length = max(len(row["input_ids"]) for row in rows)\n    input_ids = torch.zeros((len(rows), length), dtype=torch.long)\n    labels = torch.full_like(input_ids, -100)\n    attention_mask = torch.zeros_like(input_ids, dtype=torch.bool)\n    for index, row in enumerate(rows):\n        count = len(row["input_ids"])\n        assert count == len(row["labels"]) and count > 0\n        input_ids[index, :count] = torch.as_tensor(row["input_ids"])\n        labels[index, :count] = torch.as_tensor(row["labels"])\n        attention_mask[index, :count] = True\n    label_indices = torch.nonzero(labels.flatten() != -100, as_tuple=False).flatten()\n    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "label_indices": label_indices}\n\n\n\nclass EpochBatchSampler(Sampler):\n    """Seeded global permutation; every real window appears once on exactly one rank.\n\n    Pad only the final global batch with -1 sentinels, never repeated documents.\n    Sentinel labels are all ignored, so global token weighting stays exact.\n    Resume jumps directly to the saved batch without consuming earlier data.\n    """\n    def __init__(self, size: int, batch_size: int, rank: int, world_size: int,\n                 seed: int, epoch: int, start_batch: int = 0) -> None:\n        self.batch_size, self.rank, self.world_size = batch_size, rank, world_size\n        global_batch = world_size * batch_size\n        self.num_batches = (size + global_batch - 1) // global_batch\n        assert size > 0 and 0 <= start_batch <= self.num_batches and 0 <= rank < world_size\n        self.start_batch = start_batch\n        generator = torch.Generator().manual_seed(seed + epoch)\n        order = torch.randperm(size, generator=generator)\n        padding = self.num_batches * global_batch - size\n        if padding:\n            order = torch.cat((order, torch.full((padding,), -1, dtype=torch.long)))\n        self.batches = order.reshape(self.num_batches, world_size, batch_size)[:, rank]\n\n    def __iter__(self):\n        for index in range(self.start_batch, self.num_batches):\n            yield self.batches[index].tolist()\n\n    def __len__(self) -> int:\n        return self.num_batches - self.start_batch\n', 'state.py': 'from contextlib import contextmanager\nfrom dataclasses import asdict\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom .config import TrainConfig, RUNTIME_VERSION\nfrom .data import sha256_file, read_json\n\n\ndef distributed() -> bool:\n    return dist.is_available() and dist.is_initialized()\n\n\ndef atomic_json(path: Path, payload: dict) -> None:\n    temporary = path.with_name(path.name + ".partial")\n    with temporary.open("w", encoding="utf-8") as stream:\n        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)\n        stream.write("\\n"); stream.flush(); os.fsync(stream.fileno())\n    os.replace(temporary, path)\n\n\ndef runtime_fingerprint(cfg: TrainConfig, contract: dict) -> str:\n    """Paths and time/space limits may change; learning and data choices may not."""\n    choices = asdict(cfg)\n    for key in ("data_dir", "run_dir", "resume_dir", "initial_checkpoint", "max_run_seconds", "min_free_bytes",\n                "max_checkpoint_bytes", "generation_examples_per_source", "generation_token_limit"):\n        choices.pop(key)\n    sources = {path.name: sha256_file(path) for path in sorted(Path(__file__).parent.glob("*.py"))}\n    payload = {"version": RUNTIME_VERSION, "config": choices,\n               "data": contract["fingerprint"], "sources": sources,\n               "torch": str(torch.__version__)}\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\n@contextmanager\ndef exclusive_run_lock(run_dir: Path):\n    """Linux advisory lock is released by the OS even after a killed kernel."""\n    import fcntl  # Kaggle/Linux only; production does not silently fall back.\n    run_dir.mkdir(parents=True, exist_ok=True)\n    with (run_dir / ".training.lock").open("a+") as stream:\n        try:\n            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as exc:\n            raise RuntimeError("Another Model A training process owns this run directory.") from exc\n        try:\n            yield\n        finally:\n            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)\n\n\ndef seed_everything(seed: int, deterministic: bool = False) -> None:\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = deterministic\n    torch.use_deterministic_algorithms(deterministic)\n\n\ndef capture_rng(device: torch.device) -> dict:\n    numpy_state = np.random.get_state()\n    return {"python": random.getstate(), "torch_cpu": torch.get_rng_state(),\n            "numpy": {"kind": numpy_state[0], "keys": torch.tensor(numpy_state[1].astype(np.int64)),\n                      "position": numpy_state[2], "has_gauss": numpy_state[3],\n                      "cached_gaussian": numpy_state[4]},\n            "torch_cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}\n\n\ndef restore_rng(state: dict, device: torch.device) -> None:\n    random.setstate(state["python"]); torch.set_rng_state(state["torch_cpu"])\n    ns = state["numpy"]\n    np.random.set_state((ns["kind"], ns["keys"].numpy().astype(np.uint32), ns["position"],\n                         ns["has_gauss"], ns["cached_gaussian"]))\n    if device.type == "cuda":\n        torch.cuda.set_rng_state(state["torch_cuda"], device)\n\n\ndef cpu_tree(value):\n    """Own a CPU snapshot, retaining only tensors and weights-only-safe primitives."""\n    if isinstance(value, torch.Tensor):\n        return value.detach().cpu()\n    if isinstance(value, dict):\n        return {key: cpu_tree(item) for key, item in value.items()}\n    if isinstance(value, list):\n        return [cpu_tree(item) for item in value]\n    if isinstance(value, tuple):\n        return tuple(cpu_tree(item) for item in value)\n    return value\n\n\nclass CheckpointManager:\n    """Two full recovery slots plus one full best checkpoint, each with SHA-256.\n\n    The inactive slot is replaced first; its receipt commits last. A crash\n    between those writes leaves the previously committed slot recoverable.\n    Every full checkpoint includes optimizer, scaler, scheduler, and cursor.\n    """\n\n    def __init__(self, cfg: TrainConfig, fingerprint: str) -> None:\n        self.cfg, self.fingerprint, self.generation = cfg, fingerprint, 0\n        self.directory = Path(cfg.run_dir) / "checkpoints"\n        self.directory.mkdir(parents=True, exist_ok=True)\n\n    def latest(self) -> tuple[Path, dict] | None:\n        roots = [self.directory]\n        if self.cfg.resume_dir:\n            roots.append(Path(self.cfg.resume_dir))\n        candidates, seen, problems = [], False, []\n        for root in roots:\n            for name in ("slot_0.pt", "slot_1.pt", "best.pt"):\n                path, receipt_path = root / name, root / (name + ".json")\n                if not path.exists() and not receipt_path.exists():\n                    continue\n                seen = True\n                try:\n                    receipt = read_json(receipt_path)\n                    assert isinstance(receipt, dict)\n                    assert all(type(receipt[key]) is int for key in ("bytes", "step", "generation", "epoch"))\n                    assert receipt["bytes"] > 0 and receipt["step"] >= 0\n                    assert receipt["generation"] > 0 and receipt["epoch"] >= 0\n                    assert path.stat().st_size == receipt["bytes"]\n                    assert sha256_file(path) == receipt["sha256"]\n                    candidates.append((path, receipt))\n                except (OSError, ValueError, TypeError, KeyError, AssertionError) as exc:\n                    problems.append(f"{name}: {type(exc).__name__}")\n        if seen and not candidates:\n            raise RuntimeError(f"No intact checkpoint; preserve files and inspect receipts. {problems}")\n        if not candidates:\n            if self.cfg.resume_dir:\n                raise FileNotFoundError("resume_dir contains no committed checkpoints.")\n            return None\n        chosen = max(candidates, key=lambda item: (item[1]["step"], item[1]["generation"]))\n        if chosen[1]["fingerprint"] != self.fingerprint:\n            raise RuntimeError("Checkpoint data/configuration/runtime differs. Use a fresh run directory.")\n        self.generation = chosen[1]["generation"]\n        if problems and (not distributed() or dist.get_rank() == 0):\n            print(f"Ignoring incomplete checkpoints: {problems}; recovered step {chosen[1][\'step\']}", flush=True)\n        return chosen\n\n    def ensure_space(self, estimated_bytes: int) -> None:\n        owned = sum(path.stat().st_size for path in self.directory.iterdir() if path.is_file())\n        free = shutil.disk_usage(self.directory).free\n        if free < estimated_bytes + self.cfg.min_free_bytes:\n            raise OSError("Insufficient disk space for an atomic checkpoint and free-space reserve.")\n        if owned + estimated_bytes > self.cfg.max_checkpoint_bytes:\n            raise OSError("Checkpoint storage cap reached. Inspect this run\'s checkpoint files.")\n\n    def restore(self, model, optimizer, scaler, scheduler, device: torch.device) -> dict | None:\n        chosen = self.latest()\n        if chosen is None:\n            return None\n        path, receipt = chosen\n        payload = torch.load(path, map_location="cpu", weights_only=True)\n        assert payload["fingerprint"] == self.fingerprint\n        assert payload["step"] == receipt["step"] and payload["epoch"] == receipt["epoch"]\n        assert payload["world_size"] == self.cfg.world_size\n        model.load_state_dict(payload["model"], strict=True)\n        optimizer.load_state_dict(payload["optimizer"])\n        scaler.load_state_dict(payload["scaler"])\n        scheduler.load_state_dict(payload["scheduler"])\n        assert scheduler.steps == payload["cursor"]["optimizer_steps"]\n        return payload\n\n\n    def save(self, model, optimizer, scaler, scheduler, cursor: dict,\n             device: torch.device, rank: int, best: bool = False) -> None:\n        local_rng = capture_rng(device)\n        rng_states = [None] * dist.get_world_size() if distributed() else [local_rng]\n        if distributed():\n            dist.all_gather_object(rng_states, local_rng)\n        outcome = [None]\n        if rank == 0:\n            try:\n                self.generation += 1\n                path = self.directory / f"slot_{self.generation % 2}.pt"\n                temporary = path.with_name(path.name + ".partial")\n                # FP32 weights + Adam\'s two FP32 moments + serialization headroom.\n                estimated = sum(p.numel() for p in model.parameters()) * 12 + (32 << 20)\n                self.ensure_space(estimated)\n                payload = {"version": RUNTIME_VERSION, "fingerprint": self.fingerprint,\n                           "world_size": self.cfg.world_size, "config": asdict(self.cfg),\n                           "epoch": int(cursor["epoch"]), "step": int(cursor["global_step"]),\n                           "model": cpu_tree(model.state_dict()),\n                           "optimizer": cpu_tree(optimizer.state_dict()), "scaler": scaler.state_dict(),\n                           "scheduler": scheduler.state_dict(), "cursor": dict(cursor),\n                           "rng_by_rank": rng_states}\n                with temporary.open("wb") as stream:\n                    torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())\n                receipt = {"bytes": temporary.stat().st_size, "sha256": sha256_file(temporary),\n                           "fingerprint": self.fingerprint, "generation": self.generation,\n                           "epoch": payload["epoch"], "step": payload["step"],\n                           "validation_nll": (cursor["best_validation_nll"]\n                                              if math.isfinite(cursor["best_validation_nll"]) else None)}\n                os.replace(temporary, path)\n                atomic_json(path.with_name(path.name + ".json"), receipt)\n                if best:\n                    self.publish_best(path, receipt)\n                outcome[0] = {"ok": True, "generation": self.generation}\n            except Exception as exc:\n                outcome[0] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}\n        if distributed():\n            dist.broadcast_object_list(outcome, src=0)\n        if not outcome[0]["ok"]:\n            raise RuntimeError(f"Checkpoint failed: {outcome[0][\'error\']}")\n        self.generation = outcome[0]["generation"]\n\n    def publish_best(self, source: Path, receipt: dict) -> None:\n        """Hard-link the complete immutable snapshot; copy if links are unavailable."""\n        path = self.directory / "best.pt"\n        temporary = self.directory / "best.pt.partial"\n        if temporary.exists():\n            temporary.unlink()\n        try:\n            os.link(source, temporary)\n        except OSError:\n            self.ensure_space(source.stat().st_size)\n            shutil.copyfile(source, temporary)\n            with temporary.open("r+b") as stream:\n                os.fsync(stream.fileno())\n        os.replace(temporary, path)\n        atomic_json(path.with_name(path.name + ".json"), receipt)\n\n\nclass WarmupCosineScheduler:\n    """LR advances only after a successful optimizer update, including resume."""\n\n    def __init__(self, optimizer, total_steps: int, warmup_steps: int,\n                 peak_lr: float, minimum_ratio: float) -> None:\n        self.optimizer, self.total_steps = optimizer, total_steps\n        self.warmup_steps = min(warmup_steps, max(0, total_steps - 1))\n        self.peak_lr, self.minimum_ratio, self.steps = peak_lr, minimum_ratio, 0\n        self.apply_lr()\n\n    def apply_lr(self) -> None:\n        if self.warmup_steps and self.steps < self.warmup_steps:\n            ratio = (self.steps + 1) / self.warmup_steps\n        else:\n            denominator = max(1, self.total_steps - self.warmup_steps - 1)\n            progress = min(1.0, max(0.0, (self.steps - self.warmup_steps) / denominator))\n            ratio = self.minimum_ratio + (1 - self.minimum_ratio) * (1 + math.cos(math.pi * progress)) / 2\n        for group in self.optimizer.param_groups:\n            group["lr"] = self.peak_lr * ratio\n\n    def step(self) -> None:\n        self.steps += 1\n        self.apply_lr()\n\n    def state_dict(self) -> dict:\n        return {key: getattr(self, key) for key in (\n            "total_steps", "warmup_steps", "peak_lr", "minimum_ratio", "steps")}\n\n    def load_state_dict(self, state: dict) -> None:\n        for key in ("total_steps", "warmup_steps", "peak_lr", "minimum_ratio"):\n            assert state[key] == getattr(self, key), f"Scheduler mismatch: {key}"\n        self.steps = state["steps"]\n        self.apply_lr()\ndef verify_initial_checkpoint(cfg: TrainConfig, contract: dict) -> None:\n    """Check the original data identity and selected parent SHA, without changing it."""\n    if not cfg.initial_checkpoint:\n        return\n    if cfg.parent_data_fingerprint != contract["fingerprint"]:\n        raise ValueError("Initial checkpoint\'s tokenizer/data fingerprint does not match this stage")\n    path = Path(cfg.initial_checkpoint)\n    receipt = read_json(path.with_name(path.name + ".json"))\n    if receipt.get("sha256") != cfg.initial_checkpoint_sha256:\n        raise ValueError("Initial checkpoint receipt differs from the recorded parent")\n    if path.stat().st_size != receipt["bytes"] or sha256_file(path) != cfg.initial_checkpoint_sha256:\n        raise ValueError("Initial checkpoint size/SHA verification failed")\n\n\ndef load_initial_weights(model, cfg: TrainConfig, contract: dict) -> dict | None:\n    """Strictly load only parent weights; fresh optimizer/scaler/scheduler are retained.\n\n    Memory-map the parent full checkpoint so Adam moments never need a RAM copy.\n    The actual loaded FP32 weights must be finite and have the exact architecture.\n    """\n    if not cfg.initial_checkpoint:\n        return None\n    verify_initial_checkpoint(cfg, contract)\n    path = Path(cfg.initial_checkpoint)\n    before = path.stat()\n    receipt = read_json(path.with_name(path.name + ".json"))\n    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)\n    if payload["fingerprint"] != receipt["fingerprint"] or payload["epoch"] != receipt["epoch"] or payload["step"] != receipt["step"]:\n        raise ValueError("Parent payload and checkpoint receipt disagree")\n    if payload["cursor"].get("optimizer_steps", 0) <= 0:\n        raise ValueError("Parent checkpoint contains no successful optimizer updates")\n    for name, value in payload["model"].items():\n        if value.is_floating_point():\n            if value.dtype != torch.float32:\n                raise ValueError(f"Expected FP32 parent master weights: {name}")\n            flat = value.reshape(-1)\n            for start in range(0, flat.numel(), 1_000_000):\n                if not bool(torch.isfinite(flat[start:start + 1_000_000]).all()):\n                    raise ValueError(f"Non-finite parent weights: {name}")\n    model.load_state_dict(payload["model"], strict=True)\n    after = path.stat()\n    if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):\n        raise ValueError("Parent checkpoint changed while loading")\n    result = {"parent_epoch": payload["epoch"], "parent_step": payload["step"],\n              "parent_sha256": cfg.initial_checkpoint_sha256, "strict_parent_weights": "passed"}\n    del payload\n    return result\n', 'train.py': 'import argparse\nfrom contextlib import nullcontext\nfrom datetime import timedelta\nfrom functools import partial\nfrom pathlib import Path\nimport json\nimport math\nimport os\nimport signal\nimport socket\nimport time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nimport torch.distributed as dist\nimport torch.multiprocessing as mp\nfrom torch.utils.data import DataLoader, Subset\nfrom tqdm.auto import tqdm\nfrom .config import TrainConfig, assert_parameter_boundary, resolve_training_config\nfrom .data import (verify_data_contract, prepare_indices, ContinuationDataset,\n                   ValidationDataset, EpochBatchSampler, collate_continuations)\nfrom .model import LegalCausalDecoder\nfrom .state import (CheckpointManager, WarmupCosineScheduler, atomic_json, distributed,\n                    exclusive_run_lock, runtime_fingerprint, seed_everything, restore_rng, load_initial_weights, verify_initial_checkpoint)\n\n\ndef build_optimizer(model: nn.Module, cfg: TrainConfig):\n    """Decay matrix weights once; exclude bias and normalization vectors."""\n    decay, no_decay = [], []\n    for parameter in model.parameters():\n        if parameter.requires_grad:\n            (decay if parameter.ndim >= 2 else no_decay).append(parameter)\n    return torch.optim.AdamW(\n        [{"params": decay, "weight_decay": cfg.weight_decay},\n         {"params": no_decay, "weight_decay": 0.0}],\n        lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2), eps=cfg.adam_epsilon,\n        foreach=False, fused=cfg.fused_adamw and next(model.parameters()).is_cuda,\n    )\n\n\ndef reduce_sum(tensor: torch.Tensor) -> torch.Tensor:\n    if distributed():\n        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)\n    return tensor\n\n\ndef any_rank(flag: bool, device: torch.device) -> bool:\n    value = torch.tensor(int(flag), device=device, dtype=torch.int32)\n    if distributed():\n        dist.all_reduce(value, op=dist.ReduceOp.MAX)\n    return bool(value.item())\n\n\ndef move_batch(batch: dict, device: torch.device) -> dict:\n    """Select implicit masks on CPU and transfer pinned tensors asynchronously."""\n    result = {}\n    for key, value in batch.items():\n        if key in {"attention_mask", "source_mask", "decoder_mask"} and bool(value.all()):\n            result[key] = None\n        else:\n            result[key] = value.to(device, non_blocking=device.type == "cuda")\n    return result\n\n\ndef autocast_context(device: torch.device):\n    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()\n\n\ndef summed_loss(model, batch: dict) -> torch.Tensor:\n    return model(batch["input_ids"], batch["attention_mask"], labels=batch["labels"],\n                 label_indices=batch.get("label_indices"))\n\n\ndef train_one_update(model, optimizer, scaler, scheduler, batches: list[dict],\n                     cfg: TrainConfig, device: torch.device) -> dict:\n    """One token-weighted update with one distributed finite check per group.\n\n    Accumulate device flags through all microbatches; all ranks finish the same\n    DDP collective sequence before checking. No optimizer mutation occurs when\n    loss is invalid; synchronized AMP overflow skips the whole update together.\n    """\n    world_size = dist.get_world_size() if distributed() else 1\n    count = sum(int((batch["labels"] != -100).sum()) for batch in batches)\n    global_tokens = int(reduce_sum(torch.tensor(count, dtype=torch.int64, device=device)).item())\n    assert global_tokens > 0\n    optimizer.zero_grad(set_to_none=True)\n    local_loss = torch.zeros((), dtype=torch.float64, device=device)\n    invalid_loss = torch.zeros((), dtype=torch.bool, device=device)\n    used_lr = optimizer.param_groups[0]["lr"]\n    for index, cpu_batch in enumerate(batches):\n        batch = move_batch(cpu_batch, device)\n        synchronize = index == len(batches) - 1\n        context = model.no_sync() if isinstance(model, DDP) and not synchronize else nullcontext()\n        with context:\n            with autocast_context(device):\n                loss_sum = summed_loss(model, batch)\n            invalid_loss.logical_or_(~torch.isfinite(loss_sum.detach()))\n            local_loss += loss_sum.detach().double()\n            scaler.scale(loss_sum * (world_size / global_tokens)).backward()\n    scaler.unscale_(optimizer)\n    gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm,\n                                            error_if_nonfinite=False, foreach=device.type == "cuda")\n    flags = torch.stack((invalid_loss, ~torch.isfinite(gradient_norm))).to(torch.int32)\n    if distributed():\n        dist.all_reduce(flags, op=dist.ReduceOp.MAX)\n    bad_loss, bad_gradient = flags.tolist()\n    if bad_loss:\n        optimizer.zero_grad(set_to_none=True)\n        raise FloatingPointError("Non-finite loss; optimizer unchanged. Recover from the committed checkpoint.")\n    overflow = bool(bad_gradient)\n    if overflow:\n        if not scaler.is_enabled():\n            raise FloatingPointError("Non-finite FP32 gradients.")\n        scaler.update(new_scale=scaler.get_scale() * scaler.get_backoff_factor())\n    else:\n        scaler.step(optimizer)\n        scaler.update()\n        scheduler.step()\n    optimizer.zero_grad(set_to_none=True)\n    loss_value = float(reduce_sum(local_loss).item())\n    return {"loss_sum": loss_value, "tokens": global_tokens, "loss": loss_value / global_tokens,\n            "learning_rate": used_lr, "overflow": overflow,\n            "gradient_norm": float(gradient_norm.item())}\n\n\n@torch.inference_mode()\ndef evaluate(model: LegalCausalDecoder, cfg: TrainConfig, device: torch.device,\n             rank: int, world_size: int) -> dict:\n    """Visit every held-out pair exactly once globally, without sampler padding."""\n    model.eval()\n    dataset = ValidationDataset(cfg)\n    subset = Subset(dataset, range(rank, len(dataset), world_size))\n    loader = DataLoader(subset, batch_size=cfg.micro_batch_size, shuffle=False,\n                        collate_fn=collate_continuations, num_workers=cfg.num_workers,\n                        generator=torch.Generator().manual_seed(cfg.seed + 10_000),\n                        pin_memory=device.type == "cuda",\n                        **({"multiprocessing_context": "spawn", "prefetch_factor": cfg.loader_prefetch_factor,\n                            "worker_init_fn": loader_worker_init} if cfg.num_workers else {}))\n    totals = torch.zeros(2, dtype=torch.float64, device=device)\n    for cpu_batch in tqdm(loader, desc="Held-out continuation NLL", disable=rank != 0, mininterval=1):\n        batch = move_batch(cpu_batch, device)\n        with autocast_context(device):\n            loss = summed_loss(model, batch)\n        totals[0] += loss.double()\n        totals[1] += (batch["labels"] != -100).sum()\n    reduce_sum(totals)\n    assert totals[1] > 0 and torch.isfinite(totals).all(), "Invalid validation loss"\n    nll = float((totals[0] / totals[1]).item())\n    model.train()\n    return {"conditional_nll": nll, "conditional_perplexity": math.exp(min(nll, 80)),\n            "perplexity_capped": nll > 80, "target_tokens": int(totals[1].item()),\n            "examples": len(dataset)}\n\n\ndef new_cursor() -> dict:\n    return {"epoch": 0, "next_batch": 0, "global_step": 0, "optimizer_steps": 0,\n            "epoch_loss_sum": 0.0, "epoch_tokens": 0, "consecutive_overflows": 0,\n            "epoch_overflows": 0, "best_validation_nll": float("inf"), "history": []}\n\n\ndef training_loader(dataset, cfg: TrainConfig, epoch: int, start_batch: int, rank: int):\n    sampler = EpochBatchSampler(len(dataset), cfg.micro_batch_size, rank, cfg.world_size,\n                                cfg.seed, epoch, start_batch)\n    loader = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_continuations,\n                        num_workers=cfg.num_workers, pin_memory=True, persistent_workers=False,\n                        generator=torch.Generator().manual_seed(cfg.seed + epoch * 31 + rank),\n                        **({"multiprocessing_context": "spawn", "prefetch_factor": cfg.loader_prefetch_factor, "worker_init_fn": loader_worker_init}\n                           if cfg.num_workers else {}))\n    return loader, sampler\n\n\ndef repair_step_log(path: Path, saved_step: int) -> None:\n    """Remove interrupted/replayed log tails newer than the committed cursor."""\n    if not path.exists():\n        return\n    temporary = path.with_name(path.name + ".partial")\n    with path.open(encoding="utf-8") as source, temporary.open("w", encoding="utf-8") as target:\n        for line in source:\n            try:\n                row = json.loads(line)\n            except ValueError:\n                break\n            if row["step"] <= saved_step:\n                target.write(line)\n        target.flush(); os.fsync(target.fileno())\n    os.replace(temporary, path)\n\n\ndef worker(rank: int, cfg: TrainConfig, contract: dict, fingerprint: str,\n           port: int, stop_event) -> None:\n    """One spawned process per T4; all training collectives follow the same order."""\n    os.environ["MASTER_ADDR"], os.environ["MASTER_PORT"] = "127.0.0.1", str(port)\n    torch.cuda.set_device(rank)\n    device = torch.device("cuda", rank)\n    signal.signal(signal.SIGTERM, lambda *_: stop_event.set())\n    signal.signal(signal.SIGINT, lambda *_: stop_event.set())\n    dist.init_process_group(backend="nccl", rank=rank, world_size=cfg.world_size,\n                            timeout=timedelta(minutes=15))\n    started = time.monotonic()\n    try:\n        torch.set_num_threads(1)\n        seed_everything(cfg.seed + rank, cfg.deterministic)\n        model = LegalCausalDecoder(cfg).to(device)\n        assert_parameter_boundary(model)\n        ddp = DDP(model, device_ids=[rank], output_device=rank,\n                  broadcast_buffers=False, find_unused_parameters=False,\n                  gradient_as_bucket_view=True)\n        optimizer = build_optimizer(model, cfg)\n        dataset = ContinuationDataset(cfg)\n        batches_per_epoch = math.ceil(len(dataset) / (cfg.world_size * cfg.micro_batch_size))\n        updates_per_epoch = math.ceil(batches_per_epoch / cfg.accumulation_steps)\n        scheduler = WarmupCosineScheduler(optimizer, updates_per_epoch * cfg.epochs,\n                                         cfg.warmup_updates, cfg.learning_rate, cfg.min_lr_ratio)\n        scaler = torch.amp.GradScaler("cuda", enabled=True)\n        manager = CheckpointManager(cfg, fingerprint)\n        payload = manager.restore(model, optimizer, scaler, scheduler, device)\n        if payload is None:\n            parent = load_initial_weights(model, cfg, contract)\n            if parent and rank == 0:\n                print("Loaded parent weights:", json.dumps(parent), flush=True)\n        cursor = new_cursor() if payload is None else payload["cursor"]\n        assert 0 <= cursor["epoch"] <= cfg.epochs and 0 <= cursor["next_batch"] <= batches_per_epoch\n        assert cursor["next_batch"] % cfg.accumulation_steps == 0 or cursor["next_batch"] == batches_per_epoch\n        if payload is not None:\n            assert len(payload["rng_by_rank"]) == cfg.world_size\n            restore_rng(payload["rng_by_rank"][rank], device)\n        log_path = Path(cfg.run_dir) / "training_steps.jsonl"\n        if rank == 0:\n            repair_step_log(log_path, cursor["global_step"])\n            print(f"Model A: {model.parameter_count:,} unique trainable parameters", flush=True)\n            print(f"Resume: epoch={cursor[\'epoch\']} next_batch={cursor[\'next_batch\']} "\n                  f"step={cursor[\'global_step\']} optimizer_step={cursor[\'optimizer_steps\']}", flush=True)\n        run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,\n                            cursor, cfg, device, rank, stop_event, started, log_path)\n    finally:\n        dist.destroy_process_group()\n\n\ndef run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,\n                        cursor: dict, cfg: TrainConfig, device: torch.device, rank: int,\n                        stop_event, started: float, log_path: Path) -> None:\n    epoch_bar = tqdm(total=cfg.epochs, initial=cursor["epoch"], desc="Model A epochs", disable=rank != 0)\n    stopped = False\n    try:\n        while cursor["epoch"] < cfg.epochs:\n            loader, sampler = training_loader(dataset, cfg, cursor["epoch"], cursor["next_batch"], rank)\n            iterator = iter(loader)\n            total_updates = math.ceil(sampler.num_batches / cfg.accumulation_steps)\n            step_bar = tqdm(total=total_updates,\n                            initial=math.ceil(cursor["next_batch"] / cfg.accumulation_steps),\n                            desc=f"Epoch {cursor[\'epoch\'] + 1}/{cfg.epochs}", disable=rank != 0,\n                            mininterval=1, leave=False)\n            try:\n                while cursor["next_batch"] < sampler.num_batches:\n                    update_started = time.perf_counter()\n                    count = min(cfg.accumulation_steps, sampler.num_batches - cursor["next_batch"])\n                    batches = [next(iterator) for _ in range(count)]\n                    metrics = train_one_update(ddp, optimizer, scaler, scheduler, batches, cfg, device)\n                    update_seconds = time.perf_counter() - update_started\n                    target_tokens_per_second = metrics["tokens"] / max(update_seconds, 1e-9)\n                    cursor["next_batch"] += count\n                    cursor["global_step"] += 1\n                    cursor["optimizer_steps"] += int(not metrics["overflow"])\n                    cursor["epoch_loss_sum"] += metrics["loss_sum"]\n                    cursor["epoch_tokens"] += metrics["tokens"]\n                    cursor["epoch_overflows"] += int(metrics["overflow"])\n                    cursor["consecutive_overflows"] = cursor["consecutive_overflows"] + 1 if metrics["overflow"] else 0\n                    assert cursor["optimizer_steps"] == scheduler.steps\n                    if rank == 0:\n                        with log_path.open("a", encoding="utf-8") as stream:\n                            row = {"epoch": cursor["epoch"], "step": cursor["global_step"],\n                                   "optimizer_step": cursor["optimizer_steps"],\n                                   "loss": metrics["loss"], "learning_rate": metrics["learning_rate"],\n                                   "tokens": metrics["tokens"], "overflow": metrics["overflow"],\n                                   "update_seconds": update_seconds, "target_tokens_per_second": target_tokens_per_second}\n                            stream.write(json.dumps(row, allow_nan=False) + "\\n")\n                        step_bar.update(1)\n                        step_bar.set_postfix(step=cursor["global_step"], loss=f"{metrics[\'loss\']:.4f}",\n                                             lr=f"{metrics[\'learning_rate\']:.2e}", tok_s=f"{target_tokens_per_second:.0f}", refresh=False)\n                    stopping = any_rank(stop_event.is_set() or time.monotonic() - started >= cfg.max_run_seconds, device)\n                    repeated_overflow = cursor["consecutive_overflows"] >= cfg.max_consecutive_overflows\n                    if cursor["global_step"] % cfg.checkpoint_every_steps == 0 or stopping or repeated_overflow:\n                        manager.save(model, optimizer, scaler, scheduler, cursor, device, rank)\n                    if repeated_overflow:\n                        raise FloatingPointError("Repeated AMP overflows; checkpoint saved. Inspect the scale/LR before resuming.")\n                    if stopping:\n                        stopped = True\n                        break\n            finally:\n                step_bar.close()\n                del iterator, loader\n            if stopped:\n                break\n            validation = evaluate(model, cfg, device, rank, cfg.world_size)\n            improved = validation["conditional_nll"] < cursor["best_validation_nll"]\n            if improved:\n                cursor["best_validation_nll"] = validation["conditional_nll"]\n            record = {"epoch": cursor["epoch"] + 1,\n                      "train_nll": cursor["epoch_loss_sum"] / cursor["epoch_tokens"],\n                      "training_target_tokens": cursor["epoch_tokens"],\n                      "overflows": cursor["epoch_overflows"], **validation}\n            cursor["history"].append(record)\n            cursor.update(epoch=cursor["epoch"] + 1, next_batch=0, epoch_loss_sum=0.0,\n                          epoch_tokens=0, epoch_overflows=0)\n            manager.save(model, optimizer, scaler, scheduler, cursor, device, rank, best=improved)\n            if rank == 0:\n                epoch_bar.update(1); epoch_bar.set_postfix(val_nll=f"{validation[\'conditional_nll\']:.4f}")\n                print(json.dumps(record), flush=True)\n        if rank == 0:\n            atomic_json(Path(cfg.run_dir) / "training_summary.json", {\n                "status": "paused_at_saved_update" if stopped else "complete",\n                "epoch": cursor["epoch"], "next_batch": cursor["next_batch"],\n                "step": cursor["global_step"], "optimizer_steps": cursor["optimizer_steps"],\n                "history": cursor["history"], "fingerprint": manager.fingerprint,\n            })\n    finally:\n        epoch_bar.close()\n\n\ndef require_dual_t4() -> None:\n    version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])\n    assert version >= (2, 5), "Use Kaggle\'s current PyTorch CUDA runtime (>=2.5)."\n    assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select Kaggle accelerator T4 x2."\n    assert dist.is_nccl_available(), "The PyTorch build needs NCCL support."\n    for rank in range(2):\n        name = torch.cuda.get_device_name(rank)\n        assert "T4" in name, f"Expected T4 on device {rank}: {name}"\n        print(f"GPU {rank}: {name}, {torch.cuda.get_device_properties(rank).total_memory / 2**30:.1f} GiB")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n    cfg = resolve_training_config(TrainConfig.load(args.config)); cfg.validate()\n    require_dual_t4()\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n    os.environ.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    with exclusive_run_lock(Path(cfg.run_dir)):\n        contract = verify_data_contract(cfg)\n        report = prepare_indices(cfg, contract)\n        fingerprint = runtime_fingerprint(cfg, contract)\n        if CheckpointManager(cfg, fingerprint).latest() is None:\n            verify_initial_checkpoint(cfg, contract)\n        atomic_json(Path(cfg.run_dir) / "training_runtime_identity.json",\n                    {"torch": str(torch.__version__), "fingerprint": fingerprint,\n                     "experiment_kind": cfg.experiment_kind})\n        atomic_json(Path(cfg.run_dir) / "data_report.json", report)\n        atomic_json(Path(cfg.run_dir) / "performance_settings.json", {\n            "optimization_version": cfg.optimization_version, "fused_adamw": cfg.fused_adamw,\n            "loader_workers_per_rank": cfg.num_workers, "prefetch_factor": cfg.loader_prefetch_factor,\n            "loss_chunk_tokens": cfg.loss_chunk_tokens, "full_window_coverage": True,\n            "epochs": cfg.epochs, "effective_global_batch": cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps})\n        with torch.device("meta"):\n            count = assert_parameter_boundary(LegalCausalDecoder(cfg))\n        print(f"Verified Model A parameter count: {count:,}", flush=True)\n        # Uncommitted pipeline-owned temporaries cannot be restored.\n        checkpoint_dir = Path(cfg.run_dir) / "checkpoints"\n        if checkpoint_dir.exists():\n            for name in ("slot_0.pt.partial", "slot_1.pt.partial", "best.pt.partial",\n                         "slot_0.pt.json.partial", "slot_1.pt.json.partial", "best.pt.json.partial"):\n                (checkpoint_dir / name).unlink(missing_ok=True)\n        context = mp.get_context("spawn")\n        stop_event = context.Event()\n        signal.signal(signal.SIGTERM, lambda *_: stop_event.set())\n        signal.signal(signal.SIGINT, lambda *_: stop_event.set())\n        with socket.socket() as sock:\n            sock.bind(("127.0.0.1", 0)); port = sock.getsockname()[1]\n        processes = mp.spawn(worker, args=(cfg, contract, fingerprint, port, stop_event),\n                             nprocs=cfg.world_size, join=False)\n        while not processes.join(timeout=1):\n            pass\n\n\n# CELL: Bounded loader processes and recorded execution settings\ndef loader_worker_init(worker_id: int) -> None:\n    """One data worker per GPU; prevent nested CPU/tokenizer thread oversubscription."""\n    torch.set_num_threads(1)\n    os.environ["TOKENIZERS_PARALLELISM"] = "false"\n\nif __name__ == "__main__":\n    main()\n', 'infer.py': 'import argparse\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport shutil\nimport sqlite3\nimport zipfile\nimport torch\nfrom tokenizers import Tokenizer\nfrom tqdm.auto import tqdm\nfrom .config import TrainConfig, SOURCES, SPECIAL_TOKENS, resolve_training_config\nfrom .data import read_json, sha256_file, verify_data_contract\nfrom .model import LegalCausalDecoder, decode_one_token\nfrom .state import CheckpointManager, atomic_json, runtime_fingerprint, seed_everything, exclusive_run_lock\nfrom .train import autocast_context\n\n\n@torch.inference_mode()\ndef generate_continuation(model: LegalCausalDecoder, tokenizer: Tokenizer, prompt: str,\n                          device: torch.device, token_limit: int, target_words: int = 128) -> str:\n    """Generate until an extra word closes the requested final word; return 128.\n\n    Token count is not word count. EOS/control tokens are suppressed; exhausting\n    the context/token bound fails explicitly instead of inventing padding text.\n    """\n    model.eval()\n    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids\n    ids = [1] + prompt_ids[-(model.cfg.max_prompt_tokens - 1):]\n    token_ids = torch.tensor([ids], device=device)\n    generated = []\n    limit = min(token_limit, model.cfg.max_seq_tokens - len(ids) + 1)\n    forbidden = [tokenizer.token_to_id(special) for special in SPECIAL_TOKENS]\n    with autocast_context(device):\n        logits, cache = model.prefill(token_ids)\n    for step in range(limit):\n        if step:\n            with autocast_context(device):\n                logits = decode_one_token(model, token, cache)\n        logits = logits.float()\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError("Non-finite generation logits")\n        logits[:, forbidden] = float("-inf")\n        chosen = int(logits.argmax(dim=-1).item())\n        generated.append(chosen)\n        token = torch.tensor([[chosen]], device=device)\n        if (step + 1) % 8 == 0 or step + 1 == limit:\n            text = tokenizer.decode(generated, skip_special_tokens=True)\n            words = text.split()\n            if len(words) >= target_words + 1:\n                continuation = " ".join(words[:target_words])\n                assert len(continuation.split()) == target_words\n                return continuation\n    raise RuntimeError(f"Could not close {target_words} words within {limit} generated tokens. "\n                       "Inspect training/generation quality or use a larger causal context capacity in a new experiment.")\n\n\ndef load_inference_model(cfg: TrainConfig, fingerprint: str, device: torch.device):\n    manager = CheckpointManager(cfg, fingerprint)\n    latest = manager.latest()\n    if latest is None:\n        raise FileNotFoundError("No trained checkpoint exists. Run distributed training first.")\n    chosen = latest\n    roots = [manager.directory] + ([Path(cfg.resume_dir)] if cfg.resume_dir else [])\n    best_candidates = []\n    for root in roots:\n        path, receipt_path = root / "best.pt", root / "best.pt.json"\n        if path.exists() and receipt_path.exists():\n            try:\n                receipt = read_json(receipt_path)\n                assert receipt["fingerprint"] == fingerprint and path.stat().st_size == receipt["bytes"]\n                assert sha256_file(path) == receipt["sha256"]\n                score = receipt.get("validation_nll")\n                assert score is None or isinstance(score, (int, float))\n                best_candidates.append((path, receipt))\n            except (OSError, ValueError, TypeError, KeyError, AssertionError):\n                continue\n    if best_candidates:\n        chosen = min(best_candidates, key=lambda item: (\n            item[1].get("validation_nll") if item[1].get("validation_nll") is not None else float("inf"),\n            -item[1]["step"], -item[1]["generation"]))\n    payload = torch.load(chosen[0], map_location="cpu", weights_only=True)\n    assert payload["fingerprint"] == fingerprint\n    model = LegalCausalDecoder(cfg)\n    model.load_state_dict(payload["model"], strict=True)\n    model.to(device).eval()\n    receipt = chosen[1]\n    del payload\n    return model, receipt\n\n\ndef generation_records(cfg: TrainConfig, kind: str, input_json: str | None):\n    """Use exact official string IDs; development records come only from held-out data."""\n    if kind == "test":\n        if input_json is None:\n            raise ValueError("Provide --input-json pointing to the official Phase-1 test prompts.")\n        payload = read_json(Path(input_json))\n        assert isinstance(payload, dict) and payload\n        for identifier, row in payload.items():\n            assert isinstance(identifier, str) and isinstance(row, dict)\n            assert isinstance(row.get("text"), str) and len(row["text"].split()) == 512\n            yield {"id": identifier, "prompt": row["text"]}\n    else:\n        counts = dict.fromkeys(SOURCES, 0)\n        with (Path(cfg.data_dir) / "validation_pairs.jsonl").open(encoding="utf-8") as stream:\n            for line in stream:\n                row = json.loads(line)\n                source = row["source"]\n                if counts[source] < cfg.generation_examples_per_source:\n                    counts[source] += 1\n                    yield row\n        assert all(count == cfg.generation_examples_per_source for count in counts.values()), counts\n\n\ndef export_predictions(path: Path, predictions: dict) -> None:\n    assert predictions and all(len(row["text"].split()) == 128 for row in predictions.values())\n    atomic_json(path, predictions)\n\n\ndef run_generation(cfg: TrainConfig, kind: str, input_json: str | None = None,\n                   device: torch.device | None = None) -> None:\n    cfg.validate()\n    if device is None:\n        assert torch.cuda.is_available(), "Generation requires the Kaggle CUDA runtime."\n        device = torch.device("cuda", 0)\n    seed_everything(cfg.seed, cfg.deterministic)\n    contract = verify_data_contract(cfg)\n    fingerprint = runtime_fingerprint(cfg, contract)\n    model, receipt = load_inference_model(cfg, fingerprint, device)\n    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))\n    directory = Path(cfg.run_dir) / "generation" / kind\n    directory.mkdir(parents=True, exist_ok=True)\n    records = list(generation_records(cfg, kind, input_json))\n    assert len({row["id"] for row in records}) == len(records)\n    prompt_truncations = sum(len(tokenizer.encode(row["prompt"], add_special_tokens=False).ids)\n                             > cfg.max_prompt_tokens - 1 for row in records)\n    namespace = hashlib.sha256(json.dumps({"checkpoint": receipt["sha256"], "kind": kind,\n                "tokens": cfg.generation_token_limit, "words": 128}, sort_keys=True).encode()).hexdigest()\n    connection = sqlite3.connect(directory / "generation_cache.sqlite")\n    connection.execute("PRAGMA journal_mode=WAL")\n    connection.execute("PRAGMA synchronous=FULL")\n    connection.execute("CREATE TABLE IF NOT EXISTS predictions (namespace TEXT, id TEXT, "\n                       "prompt_sha TEXT, text TEXT, PRIMARY KEY(namespace,id))")\n    predictions, truth = {}, {}\n    try:\n        for row in tqdm(records, desc=f"Model A {kind}: 128-word continuations"):\n            prompt_sha = hashlib.sha256(row["prompt"].encode()).hexdigest()\n            cached = connection.execute("SELECT prompt_sha,text FROM predictions WHERE namespace=? AND id=?",\n                                        (namespace, row["id"])).fetchone()\n            if cached is not None and cached[0] == prompt_sha and len(cached[1].split()) == 128:\n                text = cached[1]\n            else:\n                if shutil.disk_usage(directory).free < cfg.min_free_bytes + (1 << 20):\n                    raise OSError("Insufficient disk space for durable generation outputs.")\n                text = generate_continuation(model, tokenizer, row["prompt"], device, cfg.generation_token_limit)\n                connection.execute("INSERT OR REPLACE INTO predictions VALUES (?,?,?,?)",\n                                   (namespace, row["id"], prompt_sha, text))\n                connection.commit()  # Each completed example survives a later failure.\n            predictions[row["id"]] = {"text": text}\n            if "reference" in row:\n                truth[row["id"]] = {"text": row["reference"]}\n        prediction_path = directory / "prediction.json"\n        export_predictions(prediction_path, predictions)\n        if truth:\n            assert truth.keys() == predictions.keys()\n            atomic_json(directory / "truth.json", truth)\n        if kind == "test":\n            temporary = directory / "submission_model_a.zip.partial"\n            with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:\n                archive.write(prediction_path, arcname="prediction.json")\n            with temporary.open("r+b") as stream:\n                os.fsync(stream.fileno())\n            os.replace(temporary, directory / "submission_model_a.zip")\n        atomic_json(directory / "generation_report.json", {\n            "checkpoint_sha256": receipt["sha256"], "checkpoint_epoch": receipt["epoch"],\n            "checkpoint_step": receipt["step"], "examples": len(predictions),\n            "words_per_continuation": 128, "decoding": "greedy with cached rotary self-attention K/V",\n            "prompt_tail_limit": cfg.max_prompt_tokens - 1,\n            "prompt_truncations": prompt_truncations,\n            "token_limit": cfg.generation_token_limit, "data_fingerprint": contract["fingerprint"],\n        })\n        print(f"Saved {len(predictions)} continuations to {prediction_path}", flush=True)\n    finally:\n        connection.close()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--kind", choices=("validation", "test"), default="validation")\n    parser.add_argument("--input-json")\n    args = parser.parse_args()\n    cfg = resolve_training_config(TrainConfig.load(args.config))\n    with exclusive_run_lock(Path(cfg.run_dir)):\n        run_generation(cfg, args.kind, args.input_json)\n\n\nif __name__ == "__main__":\n    main()\n', 'sanity.py': 'import gc\nimport torch\nfrom torch.nn import functional as F\nfrom .config import TrainConfig, assert_parameter_boundary\nfrom .model import LegalCausalDecoder, RotaryEmbedding, decode_one_token\nfrom .data import collate_continuations\n\n\ndef architecture_sanity(cfg: TrainConfig, device: torch.device | None = None) -> dict:\n    """Exercise the full production architecture on short sequences before DDP."""\n    device = torch.device("cpu") if device is None else device\n    threads = torch.get_num_threads()\n    torch.set_num_threads(min(4, threads))\n    try:\n        with torch.device("meta"):\n            count = assert_parameter_boundary(LegalCausalDecoder(cfg))\n        expected = cfg.vocab_size * cfg.d_model + cfg.n_layers * (\n            4 * cfg.d_model ** 2 + 3 * cfg.d_model * cfg.d_ff + 2 * cfg.d_model) + cfg.d_model\n        assert count == expected\n        with torch.device(device):\n            model = LegalCausalDecoder(cfg).eval()\n        ids = torch.randint(6, cfg.vocab_size, (2, 12), device=device)\n        mask = torch.ones_like(ids, dtype=torch.bool)\n        mask[1, -3:] = False\n        with torch.inference_mode():\n            logits = model(ids, mask)\n            assert logits.shape == (2, 12, cfg.vocab_size) and torch.isfinite(logits).all()\n            changed = ids.clone(); changed[:, 6:] = (changed[:, 6:] + 11) % cfg.vocab_size\n            torch.testing.assert_close(logits[:, :6], model(changed, mask)[:, :6], rtol=1e-4, atol=3e-6)\n            padded = ids.clone(); padded[1, -3:] = 9\n            torch.testing.assert_close(logits[1, :9], model(padded, mask)[1, :9], rtol=1e-4, atol=3e-6)\n            unpadded = ids[:1]\n            full = model(unpadded)\n            first, cache = model.prefill(unpadded[:, :4])\n            cached = [first]\n            for position in range(4, unpadded.shape[1]):\n                cached.append(decode_one_token(model, unpadded[:, position:position + 1], cache))\n            incremental = torch.stack(cached, dim=1)\n            torch.testing.assert_close(full[:, 3:], incremental, rtol=1e-4, atol=1e-5)\n            cache_error = float((full[:, 3:] - incremental).abs().max())\n            labels = torch.randint(6, cfg.vocab_size, ids.shape, device=device)\n            labels[1, -3:] = -100\n            labels[:, :2] = -100\n            dense_loss = F.cross_entropy(logits.flatten(0, 1).float(), labels.flatten(),\n                                         ignore_index=-100, reduction="sum")\n            chunk_loss = model(ids, mask, labels)\n            torch.testing.assert_close(dense_loss, chunk_loss, rtol=1e-5, atol=1e-4)\n            rotary = RotaryEmbedding(8, cfg.rope_base)\n            q, k = torch.randn(1, 2, 4, 8, device=device), torch.randn(1, 2, 4, 8, device=device)\n            positions = torch.arange(4, device=device)\n            q1, k1 = rotary(q, k, positions)\n            q2, k2 = rotary(q, k, positions + 17)\n            torch.testing.assert_close(q1 @ k1.transpose(-1, -2), q2 @ k2.transpose(-1, -2),\n                                       rtol=1e-4, atol=4e-6)\n        batch = collate_continuations([\n            {"input_ids": [1, 6, 7], "labels": [6, 7, 2]},\n            {"input_ids": [1, 8], "labels": [-100, 9]},\n        ])\n        assert batch["labels"][1].tolist() == [-100, 9, -100]\n        assert batch["attention_mask"][1].tolist() == [True, True, False]\n        model.train()\n        batch = {name: value.to(device) for name, value in batch.items()}\n        loss = model(batch["input_ids"], batch["attention_mask"], batch["labels"])\n        loss.backward()\n        for parameter in (model.token_embedding.weight, model.layers[0].attention.qkv.weight,\n                          model.layers[-1].ffn.gate.weight):\n            assert parameter.grad is not None and torch.isfinite(parameter.grad).all()\n            assert parameter.grad.abs().sum() > 0\n        result = {"parameters": count, "logit_shape": list(logits.shape), "device": str(device),\n                  "causal_isolation": "passed", "right_padding": "passed", "rotary_relative_positions": "passed",\n                  "cached_logits_max_abs_error": cache_error, "dense_vs_chunk_loss": "passed",\n                  "activation_checkpointed_gradient_flow": "passed"}\n        del model, loss, logits, full, incremental, cache, chunk_loss, dense_loss\n        gc.collect()\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n        return result\n    finally:\n        torch.set_num_threads(threads)\n', '__init__.py': '"""From-scratch causal legal LM and resumable DDP runtime."""\n'}}


In [34]:
# [Code Cell]
from continuation_support import prepare_extended_continuation
cfg, continuation_session = prepare_extended_continuation(
    TrainConfig, resolve_preprocessed_data_dir, original_runtime_sources,
    run_dir=RUN_DIR, parent_run=PARENT_RUN_DIR, data_dir=DATA_DIR,
    additional_epochs=ADDITIONAL_EPOCHS, target_total_epochs=TARGET_TOTAL_EPOCHS,
    continuation_lr=CONTINUATION_LR, max_run_seconds=MAX_RUN_SECONDS,
    allow_torch_upgrade=ALLOW_TORCH_UPGRADE,
    verified_runtime_sources=VERIFIED_RUNTIME_SOURCES,
)
code_dir = Path(cfg.run_dir) / "code"
package_dir = code_dir / "model_a"
config_path = Path(cfg.run_dir) / "train_config.json"
(code_dir / "resume_session.py").write_text(RESUME_SESSION_SOURCE, encoding="utf-8", newline="\n")
# Keep setup helpers in the saved output for inspection, outside the fingerprinted package.
(code_dir / "continuation_support.py").write_text(CONTINUATION_SUPPORT_SOURCE, encoding="utf-8", newline="\n")
sys.path.insert(0, str(code_dir))
for module_name in list(sys.modules):
    if module_name == "model_a" or module_name.startswith("model_a."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print("Completed epochs at latest checkpoint:", continuation_session["latest_saved_epoch"],
      "; train through epoch:", continuation_session["target_epoch"])
print("Learning-rate policy:", continuation_session["learning_rate_policy"])
print("Configuration:", config_path)


Verified saved runtime: epoch12_fast; retaining its code and training settings
Using latest committed checkpoint: /kaggle/input/notebooks/omprakashnetam69/modela-retrain-epoch12/model_a_fast_run/checkpoints/slot_1.pt (step 29942)
Writing migrated slot_1.pt; saved step 29942 is retained
Writing migrated best.pt; saved step 29942 is retained
{
  "verified_runtime": "epoch12_fast",
  "completed_epochs": 2,
  "saved_next_batch": 0,
  "saved_step": 29942,
  "train_through_epoch": 4,
  "remaining_epoch_boundaries": 2,
  "active_plan_epochs": 4,
  "scheduler_global_steps_retained": 29928,
  "learning_rate_policy": "saved_schedule_then_final_rate",
  "final_learning_rate": 2.9999999999999997e-05,
  "output": "/kaggle/working/model_a_extended_run"
}
Completed epochs at latest checkpoint: 2 ; train through epoch: 4
Learning-rate policy: saved_schedule_then_final_rate
Configuration: /kaggle/working/model_a_extended_run/train_config.json


In [35]:
# [Code Cell]
from model_a.sanity import architecture_sanity
sanity_device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
sanity_report = architecture_sanity(cfg, sanity_device)
print(json.dumps(sanity_report, indent=2))
assert 50_000_000 < sanity_report["parameters"] < 300_000_000


{
  "parameters": 110119680,
  "logit_shape": [
    2,
    12,
    32768
  ],
  "device": "cuda:0",
  "causal_isolation": "passed",
  "right_padding": "passed",
  "rotary_relative_positions": "passed",
  "cached_logits_max_abs_error": 2.6226043701171875e-06,
  "dense_vs_chunk_loss": "passed",
  "activation_checkpointed_gradient_flow": "passed"
}


[Markdown Cell]
## 12. Continue training from the saved batch
The original worker restores weights, Adam optimizer, scaler, scheduler and
rank RNGs. It resumes at `next_batch`, validates at epoch end, writes a full
checkpoint, and stops at the session target. A time limit or interruption saves
at an update boundary. If that happens, retain the output and attach it next time.


In [36]:
# [Code Cell]
def run_streamed(command: list[str], working_directory: Path) -> None:
    """Forward subprocess output, including tqdm carriage returns, in Jupyter."""
    import select
    import codecs
    process = subprocess.Popen(command, cwd=str(working_directory), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=0, env=os.environ.copy())
    interrupted = False
    decoder = codecs.getincrementaldecoder("utf-8")("replace")
    assert process.stdout is not None
    try:
        while True:
            try:
                ready, _, _ = select.select([process.stdout], [], [], 1.0)
                if ready:
                    chunk = os.read(process.stdout.fileno(), 8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                interrupted = True
                print("\nRequesting a checkpoint at the next update boundary; keep this cell running.")
                process.send_signal(signal.SIGINT)
        return_code = process.wait()
        sys.stdout.write(decoder.decode(b"", final=True)); sys.stdout.flush()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
        if interrupted:
            print("Trainer exited after the checkpoint request.")
    finally:
        process.stdout.close()

run_streamed([sys.executable, "-u", str(code_dir / "resume_session.py"), "--config", str(config_path)], code_dir)


Preprocessed data: /kaggle/input/datasets/omprakashnetam69/dataset/kaggle/working/downloaded_output
GPU 0: Tesla T4, 14.6 GiB
GPU 1: Tesla T4, 14.6 GiB
Verify preprocessing receipts: 100%|██████████| 6/6 [00:10<00:00,  1.76s/it]
Verified Model A parameter count: 110,119,680
Model A: 110,119,680 unique trainable parameters
Resume: epoch=2 next_batch=0 step=29942 optimizer_step=29928
Epoch 3/4: 100%|██████████| 14971/14971 [5:08:41<00:00,  1.14s/it, loss=1.9676, lr=3.00e-05, step=44912, tok_s=11183]
                                                                                                                     
Model A epochs:  75%|███████▌  | 3/4 [5:10:24<5:10:24, 18624.95s/it, val_nll=2.2751]{"epoch": 3, "train_nll": 1.9845679275969204, "training_target_tokens": 199840457, "overflows": 7, "conditional_nll": 2.275090144959123, "conditional_perplexity": 9.728795974947898, "perplexity_capped": false, "target_tokens": 991323, "examples": 6279}

Epoch 4/4: 100%|█████████▉| 14970/14971 [

In [37]:
# [Code Cell]
for report_name in ("training_summary.json", "data_report.json"):
    report_path = Path(cfg.run_dir) / report_name
    if report_path.exists():
        print(report_name)
        print(report_path.read_text(encoding="utf-8"))


training_summary.json
{
  "status": "complete",
  "epoch": 4,
  "next_batch": 0,
  "step": 59884,
  "optimizer_steps": 59856,
  "history": [
    {
      "epoch": 1,
      "train_nll": 2.663728790524513,
      "training_target_tokens": 199840457,
      "overflows": 7,
      "conditional_nll": 2.5057167371989353,
      "conditional_perplexity": 12.25233752554353,
      "perplexity_capped": false,
      "target_tokens": 991323,
      "examples": 6279
    },
    {
      "epoch": 2,
      "train_nll": 2.091094251076069,
      "training_target_tokens": 199840457,
      "overflows": 7,
      "conditional_nll": 2.308989565633948,
      "conditional_perplexity": 10.06425025127321,
      "perplexity_capped": false,
      "target_tokens": 991323,
      "examples": 6279
    },
    {
      "epoch": 3,
      "train_nll": 1.9845679275969204,
      "training_target_tokens": 199840457,
      "overflows": 7,
      "conditional_nll": 2.275090144959123,
      "conditional_perplexity": 9.728795974947898,
 

[Markdown Cell]
## 13. Optional development continuations
Generation is disabled by default so Run All focuses on training and saving
checkpoints. Set `RUN_DEVELOPMENT_GENERATION=True` to generate held-out examples.


In [38]:
# [Code Cell]
if RUN_DEVELOPMENT_GENERATION:
    run_streamed([sys.executable, "-u", "-m", "model_a.infer", "--config", str(config_path),
                  "--kind", "validation"], code_dir)
    validation_dir = Path(cfg.run_dir) / "generation/validation"
    print("Predictions:", validation_dir / "prediction.json")
    print("References:", validation_dir / "truth.json")
else:
    print("Development generation skipped; continuation checkpoints are saved.")


Development generation skipped; continuation checkpoints are saved.


[Markdown Cell]
## 14. Optional official inference
Set `OFFICIAL_TEST_JSON` in the configuration cell to the attached official prompt file. IDs are preserved exactly. Inspect the generated JSON and ZIP before submitting them yourself.


In [39]:
# [Code Cell]
if OFFICIAL_TEST_JSON is not None:
    test_path = Path(OFFICIAL_TEST_JSON)
    assert test_path.is_file(), test_path
    run_streamed([sys.executable, "-u", "-m", "model_a.infer", "--config", str(config_path),
                  "--kind", "test", "--input-json", str(test_path)], code_dir)
    print("Submission ZIP:", Path(cfg.run_dir) / "generation/test/submission_model_a.zip")
else:
    print("Set OFFICIAL_TEST_JSON to enable official inference.")


Set OFFICIAL_TEST_JSON to enable official inference.


[Markdown Cell]
## 15. Save this output for the next continuation
Save the complete `model_a_extended_run/` output, including `checkpoints/` and
their `.pt.json` receipts, `code/`, `train_config.json`, `data_report.json`,
`continuation_session.json` and `continuation_runtime_identity.json`.
On a fresh Kaggle session, attach this **latest saved notebook output** and the
same tokenized dataset. Set `ADDITIONAL_EPOCHS` to the number of extra epochs.
Re-running in the current session keeps the same target; it will not add another
epoch accidentally. There is no fixed total-epoch cap. `TARGET_TOTAL_EPOCHS`
overrides the relative count and can explicitly raise the target in the same
working directory; clear it to `None` when using automatic additional epochs.
Leave `CONTINUATION_LR=None` to preserve the saved schedule and its final rate.
An optional positive number chooses a constant rate while retaining Adam/RNGs.

The checkpoint's saved cursor is authoritative. This notebook does not reset
epoch/step counters, discard the Adam/scaler state, edit original Input receipts,
restart warmup, or replay completed batches. The supported framework upgrade
creates new working checkpoint/receipt pairs and records the migration; every
training-state value is retained. Abrupt termination can lose progress
since the last committed checkpoint; periodic saves remain every 500 updates.
The time limit requests a save at the next update boundary. A new GPU session
still has to fit the remaining work within Kaggle's available runtime.

The original executed notebook is preserved locally under
`Phase1_FINAL_outputs/Historical_A_Paused/model-a-decoder.ipynb`.
